In [ ]:
import os
import sys
%cd ~/erc-src/cuneiform-ocr-main/
os.getcwd()

module_path = os.path.abspath('./mmdetection')
sys.path.insert(0, module_path)

%load_ext autoreload
%autoreload 2

import mmdet
import mmengine
print(f"MMDetection version: {mmdet.__version__}")
print(f"MMEngine version: {mmengine.__version__}")

from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules
from mmdet.registry import VISUALIZERS
import mmcv
from pymongo import MongoClient
from dotenv import load_dotenv


In [ ]:
config_file = r"cuneiform-ocr-main\checkpoint_config\detr_more_transformations.py"
checkpoint_file = r"cuneiform-ocr-main\checkpoint_config\detr-173-classes-10-2025\epoch_1000.pth"


In [ ]:
import os

ROOT_DIR = r"cuneiform-ocr-main"

CONFIG_FILE = os.path.join(
    ROOT_DIR,
    "checkpoint_config",
    "detr_more_transformations.py"
)

CHECKPOINT_FILE = os.path.join(
    ROOT_DIR,
    "checkpoint_config",
    "detr-173-classes-10-2025",
    "epoch_1000.pth"
)

print(CONFIG_FILE)
print(os.path.exists(CONFIG_FILE))

print(CHECKPOINT_FILE)
print(os.path.exists(CHECKPOINT_FILE))


In [ ]:
from mmengine.config import Config
from mmdet.apis import init_detector
from mmdet.utils import register_all_modules

register_all_modules()

cfg = Config.fromfile(CONFIG_FILE)

print("Loaded config from:", CONFIG_FILE)
print("Before override:", cfg.model.bbox_head.num_classes)

cfg.model.bbox_head.num_classes = 173

print("After override:", cfg.model.bbox_head.num_classes)

model = init_detector(
    cfg,
    CHECKPOINT_FILE,
    device="cpu"
)

print("Model loaded.")
print("Model bbox classes:", model.bbox_head.num_classes)


In [ ]:
from mmengine.config import Config
from mmdet.apis import init_detector
from mmdet.utils import register_all_modules

register_all_modules()

config_file = r"cuneiform-ocr-main\checkpoint_config\detr_more_transformations.py"
checkpoint_file = r"cuneiform-ocr-main\checkpoint_config\detr-173-classes-10-2025\epoch_1000.pth"

cfg = Config.fromfile(config_file)
cfg.model.bbox_head.num_classes = 173

model = init_detector(
    cfg,
    checkpoint_file,
    device="cpu"
)

print(model.bbox_head.num_classes)

In [ ]:
# ============================================================
# eBL DETR INFERENCE ON ONE UNSEEN TABLET IMAGE
# WITH OPTIONAL TABLET/FRAGMENT DIVISION + CROP SAVING
# ============================================================

import os
import cv2
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules

from data_processing.divide_photos import divide_tablet_photo

# ============================================================
# SETTINGS
# ============================================================

ROOT_DIR = os.path.join(os.path.expanduser("~"), "Desktop", "cuneiform-ocr-main")

CONFIG_FILE = os.path.join(
    ROOT_DIR,
    "checkpoint_config/",
    "detr_more_transformations.py"
)

CHECKPOINT_FILE = os.path.join(
    ROOT_DIR,
    "checkpoint_config",
    "detr-173-classes-10-2025",
    "epoch_1000.pth"
)

UNSEEN_TABLET_DIR = os.path.join(ROOT_DIR, "unseen_tablets")

OUTPUT_DIR = os.path.join(ROOT_DIR, "unseen_tablet_outputs")
FRAGMENT_DIR = os.path.join(OUTPUT_DIR, "fragments")
CROP_DIR = os.path.join(OUTPUT_DIR, "sign_crops")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FRAGMENT_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)

SCORE_THRESHOLD = 0.60

# use "cuda:0" if CUDA-enabled mmcv works
# use "cpu" if you still get ms_deform_attn CUDA error
#DEVICE = "cuda:0"
DEVICE = "cpu"

USE_TABLET_DIVISION = True

# ============================================================
# CHECK FILES
# ============================================================

print("Config exists:", os.path.exists(CONFIG_FILE), CONFIG_FILE)
print("Checkpoint exists:", os.path.exists(CHECKPOINT_FILE), CHECKPOINT_FILE)
print("Unseen folder exists:", os.path.exists(UNSEEN_TABLET_DIR), UNSEEN_TABLET_DIR)

images = [
    f for f in os.listdir(UNSEEN_TABLET_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))
]

print("Images found:", images)

if len(images) == 0:
    raise FileNotFoundError("No image found in unseen_tablets folder.")

# Use first image
IMAGE_NAME = images[0]
IMAGE_PATH = os.path.join(UNSEEN_TABLET_DIR, IMAGE_NAME)

print("Using image:", IMAGE_PATH)

# ============================================================
# LOAD MODEL
# ============================================================

from mmengine.config import Config

register_all_modules()

cfg = Config.fromfile(CONFIG_FILE)

print("Before override:", cfg.model.bbox_head.num_classes)

# Your checkpoint has 173 classes
cfg.model.bbox_head.num_classes = 173

print("After override:", cfg.model.bbox_head.num_classes)

model = init_detector(
    cfg,                 # important: use cfg, not CONFIG_FILE
    CHECKPOINT_FILE,
    device=DEVICE
)

class_names = model.dataset_meta["classes"]

print("Model loaded.")
print("Model bbox classes:", model.bbox_head.num_classes)
print("Number of dataset_meta classes:", len(class_names))

# ============================================================
# LOAD / DIVIDE TABLET IMAGE
# ============================================================

if USE_TABLET_DIVISION:
    print("Dividing tablet photo into fragments...")

    fragments = divide_tablet_photo(
        IMAGE_PATH,
        visualize=False,
        output_path=os.path.join(OUTPUT_DIR, "division_visualization.jpg"),
    )

    # Convert BGR â†’ RGB for correct display and cropping
    fragments = [
        cv2.cvtColor(frag, cv2.COLOR_BGR2RGB)
        for frag in fragments
    ]

    print("Fragments found:", len(fragments))

else:
    print("Skipping tablet division. Using full image directly.")

    img_bgr = cv2.imread(IMAGE_PATH)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    fragments = [img_rgb]

# Save fragments for inspection
fragment_paths = []

for frag_id, frag_img in enumerate(fragments):
    frag_path = os.path.join(FRAGMENT_DIR, f"fragment_{frag_id:03d}.jpg")

    # fragment is RGB numpy array, cv2 saves BGR
    frag_bgr = cv2.cvtColor(frag_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(frag_path, frag_bgr)

    fragment_paths.append(frag_path)

print("Saved fragments to:", FRAGMENT_DIR)

# Optional quick visualization
for frag_id, frag_img in enumerate(fragments):
    plt.figure(figsize=(8, 8))
    plt.imshow(frag_img)
    plt.title(f"Fragment {frag_id}")
    plt.axis("off")
    plt.show()

# ============================================================
# RUN DETR INFERENCE ON EACH FRAGMENT
# ============================================================

rows = []

for frag_id, frag_img in enumerate(fragments):

    print(f"Running DETR on fragment {frag_id}...")

    result = inference_detector(model, frag_img)

    pred = result.pred_instances

    boxes = pred.bboxes.cpu().numpy()
    scores = pred.scores.cpu().numpy()
    labels = pred.labels.cpu().numpy()

    print(f"Raw detections on fragment {frag_id}:", len(boxes))

    for det_id, (box, score, label) in enumerate(zip(boxes, scores, labels)):

        if score < SCORE_THRESHOLD:
            continue

        x1, y1, x2, y2 = box

        x1 = max(0, int(round(x1)))
        y1 = max(0, int(round(y1)))
        x2 = int(round(x2))
        y2 = int(round(y2))

        if x2 <= x1 or y2 <= y1:
            continue

        sign_name = class_names[int(label)]

        rows.append({
            "source_image": IMAGE_NAME,
            "fragment_id": frag_id,
            "fragment_path": fragment_paths[frag_id],
            "det_id": det_id,
            "signName_detr": sign_name,
            "score_detr": float(score),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "width": x2 - x1,
            "height": y2 - y1,
        })

# ============================================================
# SAVE DETECTIONS CSV
# ============================================================

df = pd.DataFrame(rows)

csv_path = os.path.join(OUTPUT_DIR, "unseen_tablet_detr_detections.csv")
df.to_csv(csv_path, index=False)

print("Filtered detections:", len(df))
print("Saved CSV:", csv_path)

display(df.head())

# ============================================================
# SAVE SIGN CROPS
# ============================================================

crop_paths = []

for idx, row in df.iterrows():

    frag_img = fragments[int(row["fragment_id"])]

    x1 = int(row["x1"])
    y1 = int(row["y1"])
    x2 = int(row["x2"])
    y2 = int(row["y2"])

    crop = frag_img[y1:y2, x1:x2]

    if crop.size == 0:
        crop_paths.append("")
        continue

    crop_name = (
        f"fragment_{int(row['fragment_id']):03d}_"
        f"det_{int(row['det_id']):04d}_"
        f"{row['signName_detr']}_"
        f"{row['score_detr']:.2f}.png"
    )

    crop_path = os.path.join(CROP_DIR, crop_name)

    crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
    cv2.imwrite(crop_path, crop_bgr)

    crop_paths.append(crop_path)

df["crop_path"] = crop_paths

csv_with_crops_path = os.path.join(
    OUTPUT_DIR,
    "unseen_tablet_detr_detections_with_crops.csv"
)

df.to_csv(csv_with_crops_path, index=False)

print("Saved crops to:", CROP_DIR)
print("Saved CSV with crop paths:", csv_with_crops_path)

display(df.head())


In [ ]:
# ============================================================
# APPLY RESNET101 TO DETR CROPS
# COMPARE OCR SIGN WITH RESNET SIGN
# PERIOD VOTING FROM AGREED SIGNS
# ============================================================

import os
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
from torchvision import models, transforms
from tqdm.auto import tqdm

# ============================================================
# PATHS
# ============================================================

ROOT_DIR = r"cuneiform-ocr-main"

DETR_CSV = os.path.join(
    ROOT_DIR,
    "unseen_tablet_outputs",
    "unseen_tablet_detr_detections_with_crops.csv"
)

TRAIN_CSV = (
    r"ebl_tablets_and_sign_crops"
    r"\sign_classification_extension\ResNet"
    r"\tablet_holdout_train.csv"
)

RESNET_MODEL_PATH = (
    r"\ebl_tablets_and_sign_crops"
    r"\sign_classification_extension\ResNet"
    r"\best_resnet101_tablet_holdout_sign_classifier.pth"
)

OUTPUT_CSV = os.path.join(
    ROOT_DIR,
    "unseen_tablet_outputs",
    "resnet101_ocr_comparison_predictions.csv"
)

PERIOD_SUMMARY_CSV = os.path.join(
    ROOT_DIR,
    "unseen_tablet_outputs",
    "resnet101_period_vote_summary.csv"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

RESNET_CONF_THRESHOLD = 0.50

# ============================================================
# LOAD TRAIN CSV AND REBUILD LABEL MAPPING
# Must match training:
# label_to_idx = enumerate(sorted(train_labels))
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)

train_labels = set(train_df["label"].unique())

label_to_idx = {
    label: idx
    for idx, label in enumerate(sorted(train_labels))
}

idx_to_label = {
    idx: label
    for label, idx in label_to_idx.items()
}

NUM_CLASSES = len(label_to_idx)

print("NUM_CLASSES:", NUM_CLASSES)
print("Example labels:", list(idx_to_label.items())[:10])

# Build label -> signName + period lookup
label_info = (
    train_df[["label", "signName", "period"]]
    .drop_duplicates()
    .set_index("label")
    .to_dict("index")
)

# ============================================================
# TRANSFORM
# ============================================================

test_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# ============================================================
# LOAD RESNET101
# ============================================================

model = models.resnet101(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    NUM_CLASSES
)

state_dict = torch.load(
    RESNET_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

print("Loaded ResNet101 model.")

# ============================================================
# LOAD DETR CSV
# ============================================================

df = pd.read_csv(DETR_CSV)

print("DETR detections:", len(df))
display(df.head())

# ============================================================
# PREDICT ON CROPS
# ============================================================

rows = []

with torch.no_grad():

    for _, row in tqdm(df.iterrows(), total=len(df)):

        crop_path = row["crop_path"]

        if not isinstance(crop_path, str) or not os.path.exists(crop_path):
            continue

        img = Image.open(crop_path).convert("RGB")
        x = test_transform(img).unsqueeze(0).to(DEVICE)

        outputs = model(x)
        probs = torch.softmax(outputs, dim=1)

        top_probs, top_idxs = torch.topk(probs, k=3, dim=1)

        top_probs = top_probs.cpu().numpy()[0]
        top_idxs = top_idxs.cpu().numpy()[0]

        pred_idx = int(top_idxs[0])
        pred_label = idx_to_label[pred_idx]

        pred_sign = label_info[pred_label]["signName"]
        pred_period = label_info[pred_label]["period"]
        pred_prob = float(top_probs[0])

        ocr_sign = str(row["signName_detr"])

        same_sign = (
            ocr_sign == pred_sign
            and pred_prob >= RESNET_CONF_THRESHOLD
        )

        out = row.to_dict()

        out.update({
            "resnet_pred_label": pred_label,
            "resnet_pred_sign": pred_sign,
            "resnet_pred_period": pred_period,
            "resnet_pred_prob": pred_prob,

            "resnet_top2_label": idx_to_label[int(top_idxs[1])],
            "resnet_top2_prob": float(top_probs[1]),

            "resnet_top3_label": idx_to_label[int(top_idxs[2])],
            "resnet_top3_prob": float(top_probs[2]),

            "ocr_resnet_same_sign": same_sign
        })

        rows.append(out)

pred_df = pd.DataFrame(rows)
pred_df.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)
display(pred_df.head())

# ============================================================
# AGREEMENT TABLE
# ============================================================

agreement_table = (
    pred_df
    .groupby(["signName_detr", "resnet_pred_sign", "ocr_resnet_same_sign"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(agreement_table.head(50))

print("Total crops:", len(pred_df))
print("Same OCR/ResNet sign:", pred_df["ocr_resnet_same_sign"].sum())
print("Different:", len(pred_df) - pred_df["ocr_resnet_same_sign"].sum())

# ============================================================
# PERIOD VOTING ONLY FROM AGREED SIGNS
# ============================================================

agreed_df = pred_df[pred_df["ocr_resnet_same_sign"] == True].copy()

if len(agreed_df) == 0:
    print("No agreed OCR/ResNet signs. Cannot predict period.")
else:
    period_votes = (
        agreed_df
        .groupby("resnet_pred_period")
        .agg(
            num_votes=("resnet_pred_period", "count"),
            mean_confidence=("resnet_pred_prob", "mean"),
            weighted_score=("resnet_pred_prob", "sum")
        )
        .reset_index()
        .sort_values("weighted_score", ascending=False)
    )

    total_score = period_votes["weighted_score"].sum()
    period_votes["vote_probability"] = period_votes["weighted_score"] / total_score

    period_votes.to_csv(PERIOD_SUMMARY_CSV, index=False)

    print("Saved:", PERIOD_SUMMARY_CSV)
    display(period_votes)

    print("Predicted tablet period:", period_votes.iloc[0]["resnet_pred_period"])


In [ ]:
import os
import sys

ROOT_DIR = r"cuneiform-ocr-main"

os.chdir(ROOT_DIR)

module_path = os.path.join(ROOT_DIR, "mmdetection")
sys.path.insert(0, module_path)

import mmdet
import mmengine


In [ ]:
# ============================================================
# BLOCK 1 â€” RUN IN ebl-ocr
# DETR ON ALL TABLET-HOLDOUT TEST TABLETS
# OUTPUT: DETR CROPS + CSV
# ============================================================

import os
import sys
import cv2
import pandas as pd
from tqdm.auto import tqdm

from mmengine.config import Config
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules

from data_processing.divide_photos import divide_tablet_photo

# ============================================================
# PATHS
# ============================================================

OCR_ROOT = r"cuneiform-ocr-main"

# Important if mmdet is inside local cuneiform-ocr-main/mmdetection
os.chdir(OCR_ROOT)

MMDET_PATH = os.path.join(OCR_ROOT, "mmdetection")
if MMDET_PATH not in sys.path:
    sys.path.insert(0, MMDET_PATH)

SIGN_ROOT = (
    r"ebl_tablets_and_sign_crops"
    r"\sign_classification_extension\ResNet"
)

CONFIG_FILE = os.path.join(
    OCR_ROOT,
    "checkpoint_config",
    "detr_more_transformations.py"
)

DETR_CHECKPOINT_FILE = os.path.join(
    OCR_ROOT,
    "checkpoint_config",
    "detr-173-classes-10-2025",
    "epoch_1000.pth"
)

TEST_CSV = os.path.join(
    SIGN_ROOT,
    "tablet_holdout_test.csv"
)

OUTPUT_DIR = os.path.join(
    OCR_ROOT,
    "all_unseen_tablet_outputs_all_resnets"
)

CROP_DIR = os.path.join(OUTPUT_DIR, "sign_crops")
FRAGMENT_DIR = os.path.join(OUTPUT_DIR, "fragments")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)
os.makedirs(FRAGMENT_DIR, exist_ok=True)

DETR_CROPS_CSV = os.path.join(
    OUTPUT_DIR,
    "all_unseen_detr_detections_with_crops.csv"
)

# ============================================================
# SETTINGS
# ============================================================

DETR_SCORE_THRESHOLD = 0.60
DEVICE_DETR = DEVICE_DETR = "cpu"
USE_TABLET_DIVISION = True

IMAGE_PATH_COLUMN = "imagePath"
TABLET_ID_COLUMN = "fragmentNumber"
PERIOD_COLUMN = "period"

# ============================================================
# CHECK FILES
# ============================================================

print("OCR_ROOT exists:", os.path.exists(OCR_ROOT), OCR_ROOT)
print("Config exists:", os.path.exists(CONFIG_FILE), CONFIG_FILE)
print("DETR checkpoint exists:", os.path.exists(DETR_CHECKPOINT_FILE), DETR_CHECKPOINT_FILE)
print("Test CSV exists:", os.path.exists(TEST_CSV), TEST_CSV)

# ============================================================
# LOAD HOLDOUT TEST TABLETS
# ============================================================

test_df = pd.read_csv(TEST_CSV, dtype=str)

required_cols = [
    TABLET_ID_COLUMN,
    IMAGE_PATH_COLUMN,
    PERIOD_COLUMN
]

for col in required_cols:
    if col not in test_df.columns:
        raise ValueError(
            f"Column '{col}' not found in TEST_CSV. "
            f"Available columns: {test_df.columns.tolist()}"
        )

tablet_df = (
    test_df[[TABLET_ID_COLUMN, IMAGE_PATH_COLUMN, PERIOD_COLUMN]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

tablet_df = tablet_df[
    tablet_df[IMAGE_PATH_COLUMN].apply(os.path.exists)
].reset_index(drop=True)

print("Unique unseen tablets:", len(tablet_df))
display(tablet_df.head())

if len(tablet_df) == 0:
    raise FileNotFoundError("No valid unseen tablet image paths found.")

# ============================================================
# LOAD DETR MODEL
# ============================================================

register_all_modules()

cfg = Config.fromfile(CONFIG_FILE)

print("DETR classes before override:", cfg.model.bbox_head.num_classes)

cfg.model.bbox_head.num_classes = 173

print("DETR classes after override:", cfg.model.bbox_head.num_classes)

detr_model = init_detector(
    cfg,
    DETR_CHECKPOINT_FILE,
    device=DEVICE_DETR
)

class_names = detr_model.dataset_meta["classes"]

print("Loaded DETR model.")
print("Number of DETR classes:", len(class_names))

# ============================================================
# RUN DETR ON ALL UNSEEN TABLETS
# ============================================================

detr_rows = []

for _, tablet_row in tqdm(
    tablet_df.iterrows(),
    total=len(tablet_df),
    desc="Processing unseen tablets"
):

    tablet_id = str(tablet_row[TABLET_ID_COLUMN])
    image_path = tablet_row[IMAGE_PATH_COLUMN]
    true_period = tablet_row[PERIOD_COLUMN]
    image_name = os.path.basename(image_path)

    print("\n======================================")
    print("Tablet:", tablet_id)
    print("True period:", true_period)
    print("Image:", image_path)

    safe_tablet_id = (
        tablet_id
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(",", "_")
    )

    tablet_crop_dir = os.path.join(CROP_DIR, safe_tablet_id)
    tablet_fragment_dir = os.path.join(FRAGMENT_DIR, safe_tablet_id)

    os.makedirs(tablet_crop_dir, exist_ok=True)
    os.makedirs(tablet_fragment_dir, exist_ok=True)

    # --------------------------------------------------------
    # LOAD / DIVIDE TABLET
    # --------------------------------------------------------

    try:
        if USE_TABLET_DIVISION:

            fragments = divide_tablet_photo(
                image_path,
                visualize=False,
                output_path=os.path.join(
                    tablet_fragment_dir,
                    "division_visualization.jpg"
                ),
            )

            fragments = [
                cv2.cvtColor(frag, cv2.COLOR_BGR2RGB)
                for frag in fragments
            ]

        else:

            img_bgr = cv2.imread(image_path)

            if img_bgr is None:
                print("Could not read image. Skipping:", image_path)
                continue

            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            fragments = [img_rgb]

    except Exception as e:

        print("Tablet division failed. Using full image instead.")
        print("Error:", e)

        img_bgr = cv2.imread(image_path)

        if img_bgr is None:
            print("Could not read image. Skipping:", image_path)
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        fragments = [img_rgb]

    print("Fragments:", len(fragments))

    # --------------------------------------------------------
    # SAVE FRAGMENTS
    # --------------------------------------------------------

    fragment_paths = []

    for frag_id, frag_img in enumerate(fragments):

        frag_path = os.path.join(
            tablet_fragment_dir,
            f"{safe_tablet_id}_fragment_{frag_id:03d}.jpg"
        )

        frag_bgr = cv2.cvtColor(frag_img, cv2.COLOR_RGB2BGR)
        cv2.imwrite(frag_path, frag_bgr)

        fragment_paths.append(frag_path)

    # --------------------------------------------------------
    # DETR INFERENCE ON EACH FRAGMENT
    # --------------------------------------------------------

    for frag_id, frag_img in enumerate(fragments):

        print(f"Running DETR on {tablet_id}, fragment {frag_id}")

        try:
            result = inference_detector(detr_model, frag_img)

        except Exception as e:
            print("DETR failed on fragment.")
            print("Error:", e)
            continue

        pred = result.pred_instances

        boxes = pred.bboxes.cpu().numpy()
        scores = pred.scores.cpu().numpy()
        labels = pred.labels.cpu().numpy()

        print("Raw detections:", len(boxes))

        for det_id, (box, score, label) in enumerate(
            zip(boxes, scores, labels)
        ):

            if score < DETR_SCORE_THRESHOLD:
                continue

            x1, y1, x2, y2 = box

            x1 = max(0, int(round(x1)))
            y1 = max(0, int(round(y1)))
            x2 = int(round(x2))
            y2 = int(round(y2))

            if x2 <= x1 or y2 <= y1:
                continue

            sign_name_detr = class_names[int(label)]

            crop = frag_img[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            crop_name = (
                f"{safe_tablet_id}_fragment_{frag_id:03d}_"
                f"det_{det_id:04d}_"
                f"{sign_name_detr}_"
                f"{float(score):.2f}.png"
            )

            crop_path = os.path.join(
                tablet_crop_dir,
                crop_name
            )

            crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
            cv2.imwrite(crop_path, crop_bgr)

            detr_rows.append({
                "tablet_id": tablet_id,
                "true_period": true_period,
                "source_image": image_name,
                "source_image_path": image_path,

                "fragment_id": frag_id,
                "fragment_path": fragment_paths[frag_id],

                "det_id": det_id,
                "crop_path": crop_path,

                "signName_detr": sign_name_detr,
                "score_detr": float(score),

                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "width": x2 - x1,
                "height": y2 - y1
            })

# ============================================================
# SAVE DETR DETECTIONS + CROPS CSV
# ============================================================

detr_df = pd.DataFrame(detr_rows)

detr_df.to_csv(
    DETR_CROPS_CSV,
    index=False
)

print("\nSaved DETR crops CSV:")
print(DETR_CROPS_CSV)

print("Total filtered DETR crops:", len(detr_df))

display(detr_df.head())

In [ ]:
import pandas as pd

csv_path = r"cuneiform-ocr-main\all_unseen_tablet_outputs_all_resnets\all_unseen_detr_detections_with_crops.csv"

df = pd.read_csv(csv_path)

num_tablets = df["tablet_id"].nunique()

print("Unique tablets:", num_tablets)

In [ ]:
# ============================================================
# BLOCK 2 â€” RUN RESNETS ON SAVED DETR CROPS
# INPUT : all_unseen_detr_detections_with_crops.csv
# OUTPUT: crop predictions + tablet-level period voting
# ============================================================

import os
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from torchvision import models, transforms

# ============================================================
# PATHS
# ============================================================

SIGN_ROOT = (
    r"ebl_tablets_and_sign_crops"
    r"\sign_classification_extension\ResNet"
)

OCR_ROOT = r"Desktop\cuneiform-ocr-main"

OUTPUT_DIR = os.path.join(
    OCR_ROOT,
    "all_unseen_tablet_outputs_all_resnets"
)

DETR_CROPS_CSV = os.path.join(
    OUTPUT_DIR,
    "all_unseen_detr_detections_with_crops.csv"
)

TRAIN_CSV = os.path.join(
    SIGN_ROOT,
    "tablet_holdout_train.csv"
)

ALL_PREDICTIONS_CSV = os.path.join(
    OUTPUT_DIR,
    "all_unseen_detr_resnet_crop_predictions.csv"
)

PERIOD_SUMMARY_CSV = os.path.join(
    OUTPUT_DIR,
    "all_unseen_tablet_period_summary_by_resnet.csv"
)

MODEL_COMPARISON_CSV = os.path.join(
    OUTPUT_DIR,
    "all_unseen_period_model_comparison.csv"
)

# ============================================================
# SETTINGS
# ============================================================

DEVICE_RESNET = "cuda" if torch.cuda.is_available() else "cpu"

RESNET_CONF_THRESHOLD = 0.50

print("DEVICE_RESNET:", DEVICE_RESNET)

print("DETR CSV exists:", os.path.exists(DETR_CROPS_CSV))
print("TRAIN CSV exists:", os.path.exists(TRAIN_CSV))

# ============================================================
# LOAD DETR CROP CSV
# ============================================================

detr_df = pd.read_csv(DETR_CROPS_CSV, dtype=str)

# convert numeric columns
detr_df["score_detr"] = detr_df["score_detr"].astype(float)

print("Total DETR crop rows:", len(detr_df))
print("Unique tablets in DETR CSV:", detr_df["tablet_id"].nunique())

# keep only crops that exist
detr_df = detr_df[
    detr_df["crop_path"].apply(lambda p: isinstance(p, str) and os.path.exists(p))
].reset_index(drop=True)

print("Existing crop files:", len(detr_df))
print("Unique tablets with existing crops:", detr_df["tablet_id"].nunique())

display(detr_df.head())

# ============================================================
# LOAD TRAIN CSV AND LABEL MAPPING
# ============================================================

train_df = pd.read_csv(TRAIN_CSV, dtype=str)

train_labels = set(train_df["label"].unique())

label_to_idx = {
    label: idx
    for idx, label in enumerate(sorted(train_labels))
}

idx_to_label = {
    idx: label
    for label, idx in label_to_idx.items()
}

NUM_CLASSES = len(label_to_idx)

label_info = (
    train_df[["label", "signName", "period"]]
    .drop_duplicates()
    .set_index("label")
    .to_dict("index")
)

print("ResNet NUM_CLASSES:", NUM_CLASSES)

# ============================================================
# LOAD RESNET18 / RESNET50 / RESNET101
# ============================================================

MODEL_DICT = {
    "resnet18": (
        models.resnet18,
        os.path.join(SIGN_ROOT, "best_resnet18_tablet_holdout_sign_classifier.pth")
    ),
    "resnet50": (
        models.resnet50,
        os.path.join(SIGN_ROOT, "best_resnet50_tablet_holdout_sign_classifier.pth")
    ),
    "resnet101": (
        models.resnet101,
        os.path.join(SIGN_ROOT, "best_resnet101_tablet_holdout_sign_classifier.pth")
    ),
}

resnet_models = {}

for model_name, (model_fn, model_path) in MODEL_DICT.items():

    print("\nLoading:", model_name)
    print("Path:", model_path)
    print("Exists:", os.path.exists(model_path))

    if not os.path.exists(model_path):
        raise FileNotFoundError(model_path)

    model = model_fn(weights=None)

    model.fc = nn.Linear(
        model.fc.in_features,
        NUM_CLASSES
    )

    state_dict = torch.load(
        model_path,
        map_location=DEVICE_RESNET
    )

    model.load_state_dict(state_dict)
    model = model.to(DEVICE_RESNET)
    model.eval()

    resnet_models[model_name] = model

print("\nLoaded ResNet models:", list(resnet_models.keys()))

# ============================================================
# TRANSFORM
# ============================================================

test_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# ============================================================
# APPLY RESNETS TO ALL SAVED DETR CROPS
# ============================================================

all_rows = []

for _, row in tqdm(
    detr_df.iterrows(),
    total=len(detr_df),
    desc="Applying ResNets to DETR crops"
):

    crop_path = row["crop_path"]

    try:
        img = Image.open(crop_path).convert("RGB")
        x = test_transform(img).unsqueeze(0).to(DEVICE_RESNET)

    except Exception as e:
        print("Could not load crop:", crop_path)
        print("Error:", e)
        continue

    sign_name_detr = row["signName_detr"]

    for model_name, resnet_model in resnet_models.items():

        try:
            with torch.no_grad():
                outputs = resnet_model(x)
                probs = torch.softmax(outputs, dim=1)
                top_probs, top_idxs = torch.topk(probs, k=3, dim=1)

            top_probs = top_probs.cpu().numpy()[0]
            top_idxs = top_idxs.cpu().numpy()[0]

            pred_idx = int(top_idxs[0])
            pred_label = idx_to_label[pred_idx]

            pred_sign = label_info[pred_label]["signName"]
            pred_period = label_info[pred_label]["period"]
            pred_prob = float(top_probs[0])

            same_sign = (
                str(sign_name_detr) == str(pred_sign)
                and pred_prob >= RESNET_CONF_THRESHOLD
            )

            all_rows.append({
                "tablet_id": row["tablet_id"],
                "true_period": row["true_period"],
                "model_name": model_name,

                "source_image": row["source_image"],
                "source_image_path": row["source_image_path"],

                "fragment_id": row["fragment_id"],
                "fragment_path": row["fragment_path"],

                "det_id": row["det_id"],
                "crop_path": crop_path,

                "signName_detr": sign_name_detr,
                "score_detr": float(row["score_detr"]),

                "x1": row["x1"],
                "y1": row["y1"],
                "x2": row["x2"],
                "y2": row["y2"],
                "width": row["width"],
                "height": row["height"],

                "resnet_pred_label": pred_label,
                "resnet_pred_sign": pred_sign,
                "resnet_pred_period": pred_period,
                "resnet_pred_prob": pred_prob,

                "resnet_top2_label": idx_to_label[int(top_idxs[1])],
                "resnet_top2_prob": float(top_probs[1]),

                "resnet_top3_label": idx_to_label[int(top_idxs[2])],
                "resnet_top3_prob": float(top_probs[2]),

                "ocr_resnet_same_sign": same_sign
            })

        except Exception as e:
            print("ResNet failed.")
            print("Model:", model_name)
            print("Crop:", crop_path)
            print("Error:", e)

# ============================================================
# SAVE ALL CROP-LEVEL PREDICTIONS
# ============================================================

pred_df = pd.DataFrame(all_rows)

pred_df.to_csv(
    ALL_PREDICTIONS_CSV,
    index=False
)

print("\nSaved all crop-level predictions:")
print(ALL_PREDICTIONS_CSV)

print("Total prediction rows:", len(pred_df))
print("Unique tablets:", pred_df["tablet_id"].nunique())

display(pred_df.head())

# ============================================================
# TABLET-LEVEL PERIOD VOTING FOR EACH MODEL
# ============================================================

summary_rows = []

for (tablet_id, model_name), g in pred_df.groupby(["tablet_id", "model_name"]):

    true_period = g["true_period"].iloc[0]

    agreed_df = g[g["ocr_resnet_same_sign"] == True].copy()

    if len(agreed_df) == 0:

        summary_rows.append({
            "tablet_id": tablet_id,
            "model_name": model_name,
            "true_period": true_period,
            "source_image": g["source_image"].iloc[0],
            "source_image_path": g["source_image_path"].iloc[0],

            "num_detections": len(g),
            "num_agreed_signs": 0,

            "predicted_period": None,
            "prediction_confidence": 0.0,
            "period_correct": False,

            "top_period_votes": 0,
            "top_period_weighted_score": 0.0
        })

        continue

    period_votes = (
        agreed_df
        .groupby("resnet_pred_period")
        .agg(
            num_votes=("resnet_pred_period", "count"),
            mean_confidence=("resnet_pred_prob", "mean"),
            weighted_score=("resnet_pred_prob", "sum")
        )
        .reset_index()
        .sort_values("weighted_score", ascending=False)
    )

    total_score = period_votes["weighted_score"].sum()
    period_votes["vote_probability"] = (
        period_votes["weighted_score"] / total_score
    )

    best = period_votes.iloc[0]

    predicted_period = best["resnet_pred_period"]

    summary_rows.append({
        "tablet_id": tablet_id,
        "model_name": model_name,
        "true_period": true_period,
        "source_image": g["source_image"].iloc[0],
        "source_image_path": g["source_image_path"].iloc[0],

        "num_detections": len(g),
        "num_agreed_signs": len(agreed_df),

        "predicted_period": predicted_period,
        "prediction_confidence": float(best["vote_probability"]),
        "period_correct": predicted_period == true_period,

        "top_period_votes": int(best["num_votes"]),
        "top_period_weighted_score": float(best["weighted_score"])
    })

period_summary_df = pd.DataFrame(summary_rows)

period_summary_df.to_csv(
    PERIOD_SUMMARY_CSV,
    index=False
)

print("\nSaved tablet-level period summary:")
print(PERIOD_SUMMARY_CSV)

print("Tablet summary rows:", len(period_summary_df))
print("Unique tablets in summary:", period_summary_df["tablet_id"].nunique())

display(period_summary_df.head())

# ============================================================
# MODEL-LEVEL PERIOD PREDICTION COMPARISON
# ============================================================

model_comparison_df = (
    period_summary_df
    .groupby("model_name")
    .agg(
        num_tablets=("tablet_id", "count"),
        correct_tablets=("period_correct", "sum"),
        mean_confidence=("prediction_confidence", "mean"),
        mean_detections=("num_detections", "mean"),
        mean_agreed_signs=("num_agreed_signs", "mean")
    )
    .reset_index()
)

model_comparison_df["period_accuracy"] = (
    model_comparison_df["correct_tablets"] /
    model_comparison_df["num_tablets"]
)

model_comparison_df = model_comparison_df.sort_values(
    "period_accuracy",
    ascending=False
)

model_comparison_df.to_csv(
    MODEL_COMPARISON_CSV,
    index=False
)

print("\nSaved model comparison:")
print(MODEL_COMPARISON_CSV)

display(model_comparison_df)

# ============================================================
# EXTRA: COUNT UNIQUE TABLETS
# ============================================================

print("\nFinal counts:")
print("Unique tablets in DETR crops CSV:", detr_df["tablet_id"].nunique())
print("Unique tablets in ResNet predictions:", pred_df["tablet_id"].nunique())
print("Unique tablets in period summary:", period_summary_df["tablet_id"].nunique())

In [ ]:
# ============================================================
# BLOCK 2 â€” RUN ConvNeXt ON SAVED DETR CROPS
# INPUT : all_unseen_detr_detections_with_crops.csv
# OUTPUT: crop predictions + tablet-level period voting
# ============================================================

import os
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from torchvision import models, transforms

# ============================================================
# PATHS
# ============================================================

SIGN_ROOT = (
    r"Desktop\ebl_tablets_and_sign_crops"
    r"\sign_classification_extension\ConvNeXt"
)

OCR_ROOT = r"Desktop\cuneiform-ocr-main"

OUTPUT_DIR = os.path.join(
    OCR_ROOT,
    "all_unseen_tablet_outputs_all_convnext"
)

DETR_CROPS_CSV = os.path.join(
    OUTPUT_DIR,
    "all_unseen_detr_detections_with_crops.csv"
)

TRAIN_CSV = os.path.join(
    SIGN_ROOT,
    "tablet_holdout_train.csv"
)

ALL_PREDICTIONS_CSV = os.path.join(
    OUTPUT_DIR,
    "all_unseen_detr_convnext_crop_predictions.csv"
)

PERIOD_SUMMARY_CSV = os.path.join(
    OUTPUT_DIR,
    "all_unseen_tablet_period_summary_by_convnext.csv"
)

MODEL_COMPARISON_CSV = os.path.join(
    OUTPUT_DIR,
    "all_unseen_period_model_comparison_convnext.csv"
)

# ============================================================
# SETTINGS
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ConvNext_CONF_THRESHOLD = 0.50

print("DEVICE:", DEVICE)

print("DETR CSV exists:", os.path.exists(DETR_CROPS_CSV))
print("TRAIN CSV exists:", os.path.exists(TRAIN_CSV))

# ============================================================
# LOAD DETR CROP CSV
# ============================================================

detr_df = pd.read_csv(DETR_CROPS_CSV, dtype=str)

# convert numeric columns
detr_df["score_detr"] = detr_df["score_detr"].astype(float)

print("Total DETR crop rows:", len(detr_df))
print("Unique tablets in DETR CSV:", detr_df["tablet_id"].nunique())

# keep only crops that exist
detr_df = detr_df[
    detr_df["crop_path"].apply(lambda p: isinstance(p, str) and os.path.exists(p))
].reset_index(drop=True)

print("Existing crop files:", len(detr_df))
print("Unique tablets with existing crops:", detr_df["tablet_id"].nunique())

display(detr_df.head())

# ============================================================
# LOAD TRAIN CSV AND LABEL MAPPING
# ============================================================

train_df = pd.read_csv(TRAIN_CSV, dtype=str)

train_labels = set(train_df["label"].unique())

label_to_idx = {
    label: idx
    for idx, label in enumerate(sorted(train_labels))
}

idx_to_label = {
    idx: label
    for label, idx in label_to_idx.items()
}

NUM_CLASSES = len(label_to_idx)

label_info = (
    train_df[["label", "signName", "period"]]
    .drop_duplicates()
    .set_index("label")
    .to_dict("index")
)

print("ConvNeXt NUM_CLASSES:", NUM_CLASSES)

# ============================================================
# LOAD ConvNeXt
# ============================================================

MODEL_DICT = {
    "convnext_base": (
        models.convnext_base,
        os.path.join(
            SIGN_ROOT,
            "best_convnext_base_tablet_holdout_sign_classifier.pth"
        )
    ),
}

convnext_models = {}

for model_name, (
    model_fn,
    model_path
) in MODEL_DICT.items():

    print("\nLoading:", model_name)
    print("Weights:", model_path)

    if not os.path.exists(model_path):
        raise FileNotFoundError(model_path)

    model = model_fn(
        weights=None
    )

    model.classifier[2] = nn.Linear(
        model.classifier[2].in_features,
        NUM_CLASSES
    )

    state_dict = torch.load(
        model_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        state_dict
    )

    model = model.to(
        DEVICE
    )

    model.eval()

    convnext_models[
        model_name
    ] = model

print("\nLoaded models:")
print(list(convnext_models.keys()))

# ============================================================
# TRANSFORM
# ============================================================

test_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# ============================================================
# APPLY RESNETS TO ALL SAVED DETR CROPS
# ============================================================

all_rows = []

for _, row in tqdm(
    detr_df.iterrows(),
    total=len(detr_df),
    desc="Applying ConvNeXt"
):

    crop_path = row["crop_path"]

    # --------------------------------------------------------
    # LOAD IMAGE
    # --------------------------------------------------------

    try:

        img = Image.open(crop_path).convert("RGB")

        x = test_transform(img).unsqueeze(0).to(DEVICE)

    except Exception as e:

        print("Could not load crop:")
        print(crop_path)
        print(e)
        continue

    sign_name_detr = row["signName_detr"]

    # --------------------------------------------------------
    # RUN CONVNEXT
    # --------------------------------------------------------

    for model_name, model in convnext_models.items():

        try:

            with torch.no_grad():

                outputs = model(x)

                probs = torch.softmax(
                    outputs,
                    dim=1
                )

                top_probs, top_idxs = torch.topk(
                    probs,
                    k=3,
                    dim=1
                )

            top_probs = top_probs.cpu().numpy()[0]
            top_idxs = top_idxs.cpu().numpy()[0]

            pred_idx = int(top_idxs[0])

            pred_label = idx_to_label[pred_idx]

            pred_sign = label_info[pred_label]["signName"]

            pred_period = label_info[pred_label]["period"]

            pred_prob = float(top_probs[0])

            same_sign = (
                str(sign_name_detr) == str(pred_sign)
                and pred_prob >= ConvNext_CONF_THRESHOLD
            )

            all_rows.append({

                # --------------------------------------------
                # Tablet information
                # --------------------------------------------

                "tablet_id":
                    row["tablet_id"],

                "true_period":
                    row["true_period"],

                "model_name":
                    model_name,

                # --------------------------------------------
                # Source image
                # --------------------------------------------

                "source_image":
                    row["source_image"],

                "source_image_path":
                    row["source_image_path"],

                # --------------------------------------------
                # Fragment
                # --------------------------------------------

                "fragment_id":
                    row["fragment_id"],

                "fragment_path":
                    row["fragment_path"],

                # --------------------------------------------
                # Detection
                # --------------------------------------------

                "det_id":
                    row["det_id"],

                "crop_path":
                    crop_path,

                "signName_detr":
                    sign_name_detr,

                "score_detr":
                    float(row["score_detr"]),

                # --------------------------------------------
                # Bounding box
                # --------------------------------------------

                "x1":
                    int(row["x1"]),

                "y1":
                    int(row["y1"]),

                "x2":
                    int(row["x2"]),

                "y2":
                    int(row["y2"]),

                "width":
                    int(row["width"]),

                "height":
                    int(row["height"]),

                # --------------------------------------------
                # ConvNeXt predictions
                # --------------------------------------------

                "convnext_pred_label":
                    pred_label,

                "convnext_pred_sign":
                    pred_sign,

                "convnext_pred_period":
                    pred_period,

                "convnext_pred_prob":
                    pred_prob,

                # --------------------------------------------
                # Top-2
                # --------------------------------------------

                "convnext_top2_label":
                    idx_to_label[
                        int(top_idxs[1])
                    ],

                "convnext_top2_prob":
                    float(top_probs[1]),

                # --------------------------------------------
                # Top-3
                # --------------------------------------------

                "convnext_top3_label":
                    idx_to_label[
                        int(top_idxs[2])
                    ],

                "convnext_top3_prob":
                    float(top_probs[2]),

                # --------------------------------------------
                # Agreement
                # --------------------------------------------

                "ocr_convnext_same_sign":
                    same_sign
            })

        except Exception as e:

            print("\nConvNeXt failed")

            print("Model:",
                  model_name)

            print("Crop:",
                  crop_path)

            print(e)

# ============================================================
# SAVE ALL CROP-LEVEL PREDICTIONS
# ============================================================

pred_df = pd.DataFrame(all_rows)

pred_df.to_csv(
    ALL_PREDICTIONS_CSV,
    index=False
)

print("\nSaved all crop-level predictions:")
print(ALL_PREDICTIONS_CSV)

print("Total prediction rows:", len(pred_df))
print("Unique tablets:", pred_df["tablet_id"].nunique())

display(pred_df.head())

# ============================================================
# TABLET-LEVEL PERIOD VOTING FOR EACH MODEL
# ============================================================

summary_rows = []

for (tablet_id, model_name), g in pred_df.groupby(["tablet_id", "model_name"]):

    true_period = g["true_period"].iloc[0]

    agreed_df = g[g["ocr_convnext_same_sign"] == True].copy()

    if len(agreed_df) == 0:

        summary_rows.append({
            "tablet_id": tablet_id,
            "model_name": model_name,
            "true_period": true_period,
            "source_image": g["source_image"].iloc[0],
            "source_image_path": g["source_image_path"].iloc[0],

            "num_detections": len(g),
            "num_agreed_signs": 0,

            "predicted_period": None,
            "prediction_confidence": 0.0,
            "period_correct": False,

            "top_period_votes": 0,
            "top_period_weighted_score": 0.0
        })

        continue

    period_votes = (
        agreed_df
        .groupby("convnext_pred_period")
        .agg(
            num_votes=("convnext_pred_period", "count"),
            mean_confidence=("convnext_pred_prob", "mean"),
            weighted_score=("convnext_pred_prob", "sum")
        )
        .reset_index()
        .sort_values("weighted_score", ascending=False)
    )

    total_score = period_votes["weighted_score"].sum()
    period_votes["vote_probability"] = (
        period_votes["weighted_score"] / total_score
    )

    best = period_votes.iloc[0]

    predicted_period = best["convnext_pred_period"]

    summary_rows.append({
        "tablet_id": tablet_id,
        "model_name": model_name,
        "true_period": true_period,
        "source_image": g["source_image"].iloc[0],
        "source_image_path": g["source_image_path"].iloc[0],

        "num_detections": len(g),
        "num_agreed_signs": len(agreed_df),

        "predicted_period": predicted_period,
        "prediction_confidence": float(best["vote_probability"]),
        "period_correct": predicted_period == true_period,

        "top_period_votes": int(best["num_votes"]),
        "top_period_weighted_score": float(best["weighted_score"])
    })

period_summary_df = pd.DataFrame(summary_rows)

period_summary_df.to_csv(
    PERIOD_SUMMARY_CSV,
    index=False
)

print("\nSaved tablet-level period summary:")
print(PERIOD_SUMMARY_CSV)

print("Tablet summary rows:", len(period_summary_df))
print("Unique tablets in summary:", period_summary_df["tablet_id"].nunique())

display(period_summary_df.head())

# ============================================================
# MODEL-LEVEL PERIOD PREDICTION COMPARISON
# ============================================================

model_comparison_df = (
    period_summary_df
    .groupby("model_name")
    .agg(
        num_tablets=("tablet_id", "count"),
        correct_tablets=("period_correct", "sum"),
        mean_confidence=("prediction_confidence", "mean"),
        mean_detections=("num_detections", "mean"),
        mean_agreed_signs=("num_agreed_signs", "mean")
    )
    .reset_index()
)

model_comparison_df["period_accuracy"] = (
    model_comparison_df["correct_tablets"] /
    model_comparison_df["num_tablets"]
)

model_comparison_df = model_comparison_df.sort_values(
    "period_accuracy",
    ascending=False
)

model_comparison_df.to_csv(
    MODEL_COMPARISON_CSV,
    index=False
)

print("\nSaved model comparison:")
print(MODEL_COMPARISON_CSV)

display(model_comparison_df)

# ============================================================
# EXTRA: COUNT UNIQUE TABLETS
# ============================================================

print("\nFinal counts:")
print("Unique tablets in DETR crops CSV:", detr_df["tablet_id"].nunique())
print("Unique tablets in ConvNeXt predictions:", pred_df["tablet_id"].nunique())
print("Unique tablets in period summary:", period_summary_df["tablet_id"].nunique())

In [ ]:
# ============================================================
# BLOCK 1 — MongoDB UNANNOTATED TABLETS → DETR OCR CROPS
# ENV: ebl-ocr
# OUTPUT:
#   csv_parts/detr_crops_part_000001.csv
#   csv_parts/detr_crops_part_000002.csv
#   saved DETR crops
#   checkpoint.json
# ============================================================

import os
import cv2
import json
import gridfs
import pandas as pd
import numpy as np

from io import BytesIO
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from pymongo import MongoClient

from mmengine.config import Config
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules


# ============================================================
# SETTINGS
# ============================================================

Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

MONGO_URI = #your mongo uri here

DB_NAME = "ebl"

OCR_ROOT = r"cuneiform-ocr-main"

DETR_CONFIG = (
    r"cuneiform-ocr-main"
    r"\checkpoint_config\detr_more_transformations.py"
)

DETR_CHECKPOINT = (
    r"cuneiform-ocr-main"
    r"\checkpoint_config\detr-173-classes-10-2025\epoch_1000.pth"
)

OUTPUT_DIR = os.path.join(
    OCR_ROOT,
    "mongo_unannotated_tablet_outputs"
)

CROP_DIR = os.path.join(
    OUTPUT_DIR,
    "detr_crops"
)

CSV_PARTS_DIR = os.path.join(
    OUTPUT_DIR,
    "csv_parts"
)

CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR,
    "checkpoint.json"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)
os.makedirs(CSV_PARTS_DIR, exist_ok=True)

DEVICE = "cpu"

SCORE_THR = 0.60
MAX_DETECTIONS_PER_IMAGE = 80

# One metadata CSV part after this many crop rows
CSV_PART_SIZE = 100_000

# Save checkpoint after this many processed unannotated fragments
SAVE_EVERY_TABLETS = 100

# For testing use e.g. 500. For full run use None.
MAX_TABLETS = None


# ============================================================
# CHECKPOINT FUNCTIONS
# ============================================================

def default_checkpoint():
    return {
        "seen_fragments": 0,
        "processed_unannotated": 0,
        "skipped_with_annotations": 0,
        "skipped_no_photo": 0,
        "failed_images": 0,
        "failed_ocr": 0,
        "total_crops": 0,
        "csv_part_index": 1,
        "rows_in_current_part": 0,
        "last_fragment_id": None,
    }


def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return default_checkpoint()


def save_checkpoint(stats):
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)


def get_csv_part_path(part_index):
    return os.path.join(
        CSV_PARTS_DIR,
        f"detr_crops_part_{part_index:06d}.csv"
    )


def append_rows_to_csv_part(rows, stats):
    if len(rows) == 0:
        return stats

    start = 0

    while start < len(rows):

        remaining_space = CSV_PART_SIZE - stats["rows_in_current_part"]
        chunk = rows[start:start + remaining_space]

        csv_path = get_csv_part_path(stats["csv_part_index"])
        file_exists = os.path.exists(csv_path)

        pd.DataFrame(chunk).to_csv(
            csv_path,
            mode="a",
            index=False,
            header=not file_exists,
            encoding="utf-8"
        )

        stats["rows_in_current_part"] += len(chunk)
        start += len(chunk)

        if stats["rows_in_current_part"] >= CSV_PART_SIZE:
            stats["csv_part_index"] += 1
            stats["rows_in_current_part"] = 0

    return stats


# ============================================================
# CONNECT TO MONGODB
# ============================================================

print("Connecting to MongoDB...")

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
fs = gridfs.GridFS(db, collection="photos")

print("Connected to MongoDB:", DB_NAME)

total_fragments = db.fragments.count_documents({})
total_with_period = db.fragments.count_documents({
    "script.period": {"$exists": True, "$ne": None}
})

print("Total fragments:", total_fragments)
print("Fragments with script.period:", total_with_period)


# ============================================================
# LOAD ANNOTATED FRAGMENT IDS ONCE
# ============================================================

print("Loading annotated fragment IDs...")

annotated_fragment_ids = set()

ann_projection = {
    "fragmentId": 1,
    "fragmentNumber": 1,
    "museumNumber": 1,
    "fragment": 1,
    "fragments": 1,
}

ann_cursor = db.annotations.find(
    {},
    ann_projection,
    no_cursor_timeout=True
)

for ann in tqdm(ann_cursor, desc="Loading annotation references"):

    for key in ["fragmentId", "fragmentNumber", "museumNumber", "fragment"]:
        value = ann.get(key)

        if value is not None:
            annotated_fragment_ids.add(str(value))

    fragments = ann.get("fragments")

    if isinstance(fragments, list):
        for value in fragments:
            if value is not None:
                annotated_fragment_ids.add(str(value))

ann_cursor.close()

print("Annotated fragment references:", len(annotated_fragment_ids))


# ============================================================
# LOAD DETR OCR MODEL
# ============================================================

register_all_modules()

cfg = Config.fromfile(DETR_CONFIG)
cfg.model.bbox_head.num_classes = 173

detr_model = init_detector(
    cfg,
    DETR_CHECKPOINT,
    device=DEVICE
)

class_names = detr_model.dataset_meta["classes"]

print("DETR OCR model loaded.")
print("Number of DETR classes:", len(class_names))
print("First 10 DETR classes:", class_names[:10])


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "_")
        .replace("?", "_")
        .replace("*", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
        .replace(",", "_")
    )


def get_fragment_number(fragment):
    return (
        fragment.get("_id")
        or fragment.get("museumNumber")
        or fragment.get("number")
    )


def has_manual_annotation(fragment):
    fragment_id = str(fragment.get("_id"))
    fragment_number = str(get_fragment_number(fragment))

    if fragment.get("annotations"):
        return True

    if fragment.get("signAnnotations"):
        return True

    if fragment_id in annotated_fragment_ids:
        return True

    if fragment_number in annotated_fragment_ids:
        return True

    return False


def get_photo_file_id(fragment):
    fragment_id = str(fragment.get("_id"))

    possible_filenames = [
        f"{fragment_id}.jpg",
        f"{fragment_id}.jpeg",
        f"{fragment_id}.png",
        fragment_id,
    ]

    photo_file = db["photos.files"].find_one({
        "filename": {"$in": possible_filenames}
    })

    if photo_file is None:
        return None

    return photo_file["_id"]


def load_image_from_gridfs(file_id):
    grid_file = fs.get(file_id)
    image_bytes = grid_file.read()

    image = Image.open(BytesIO(image_bytes)).convert("RGB")
    image.load()

    return image


def run_detr_on_image(pil_image, score_thr=0.60):
    image_np = np.array(pil_image)
    image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    result = inference_detector(detr_model, image_bgr)

    pred_instances = result.pred_instances

    bboxes = pred_instances.bboxes.cpu().numpy()
    scores = pred_instances.scores.cpu().numpy()
    labels = pred_instances.labels.cpu().numpy()

    detections = []

    for bbox, score, label in zip(bboxes, scores, labels):

        score = float(score)

        if score < score_thr:
            continue

        label_id = int(label)

        if 0 <= label_id < len(class_names):
            sign_name_detr = class_names[label_id]
        else:
            sign_name_detr = "UNKNOWN"

In [ ]:
# ============================================================
# BLOCK 1 MongoDB UNANNOTATED TABLETS/FRAGMENTS DETR OCR CROPS
# ENV: ebl-ocr
#
# OUTPUT:
#   mongo_unannotated_tablet_outputs/
#       detr_crops/
#       csv_parts/
#           detr_crops_part_000001.csv
#           detr_crops_part_000002.csv
#           ...
#       checkpoint.json
# ============================================================

import os
import cv2
import json
import time
import gridfs
import pandas as pd
import numpy as np

from io import BytesIO
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from pymongo import MongoClient

from mmengine.config import Config
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules


# ============================================================
# SETTINGS
# ============================================================

Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

    "START_TIME = time.time()\n",
    "\n",
    "MONGO_URI = # your mongo uri here\n",
    "\n",
    "DB_NAME = \"ebl\"\n",
    "\n",
    "OCR_ROOT = r\"C:\\Users\\re29faj\\Desktop\\cuneiform-ocr-main\"\n",
    "\n",
    "DETR_CONFIG = (\n",
    "    r\"C:\\Users\\re29faj\\Desktop\\cuneiform-ocr-main\"\n",
    "    r\"\\checkpoint_config\\detr_more_transformations.py\"\n",
)

DETR_CHECKPOINT = (
    r"cuneiform-ocr-main"
    r"\checkpoint_config\detr-173-classes-10-2025\epoch_1000.pth"
)

OUTPUT_DIR = os.path.join(
    OCR_ROOT,
    "mongo_unannotated_tablet_outputs"
)

CROP_DIR = os.path.join(
    OUTPUT_DIR,
    "detr_crops"
)

CSV_PARTS_DIR = os.path.join(
    OUTPUT_DIR,
    "csv_parts"
)

CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR,
    "checkpoint.json"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)
os.makedirs(CSV_PARTS_DIR, exist_ok=True)

DEVICE = "cpu"

SCORE_THR = 0.60
MAX_DETECTIONS_PER_IMAGE = 80

CSV_PART_SIZE = 100_000
SAVE_EVERY_TABLETS = 100
PROGRESS_EVERY_TABLETS = 100

# For testing:
# MAX_TABLETS = 500
# For full run:
MAX_TABLETS = None


# ============================================================
# CHECKPOINT FUNCTIONS
# ============================================================

def default_checkpoint():
    return {
        "seen_fragments": 0,
        "processed_unannotated": 0,
        "skipped_with_annotations": 0,
        "skipped_no_photo": 0,
        "failed_images": 0,
        "failed_ocr": 0,
        "total_crops": 0,
        "csv_part_index": 1,
        "rows_in_current_part": 0,
        "last_fragment_id": None,
    }


def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return default_checkpoint()


def save_checkpoint(stats):
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)


def get_csv_part_path(part_index):
    return os.path.join(
        CSV_PARTS_DIR,
        f"detr_crops_part_{part_index:06d}.csv"
    )


def append_rows_to_csv_part(rows, stats):
    if len(rows) == 0:
        return stats

    start = 0

    while start < len(rows):

        remaining_space = CSV_PART_SIZE - stats["rows_in_current_part"]
        chunk = rows[start:start + remaining_space]

        csv_path = get_csv_part_path(stats["csv_part_index"])
        file_exists = os.path.exists(csv_path)

        pd.DataFrame(chunk).to_csv(
            csv_path,
            mode="a",
            index=False,
            header=not file_exists,
            encoding="utf-8"
        )

        stats["rows_in_current_part"] += len(chunk)
        start += len(chunk)

        if stats["rows_in_current_part"] >= CSV_PART_SIZE:
            stats["csv_part_index"] += 1
            stats["rows_in_current_part"] = 0

    return stats


def print_progress(stats, total_to_scan):
    elapsed = time.time() - START_TIME

    seen_rate = stats["seen_fragments"] / elapsed if elapsed > 0 else 0
    processed_rate = stats["processed_unannotated"] / elapsed if elapsed > 0 else 0
    crop_rate = stats["total_crops"] / elapsed if elapsed > 0 else 0

    remaining_seen = max(total_to_scan - stats["seen_fragments"], 0)
    eta_seconds = remaining_seen / seen_rate if seen_rate > 0 else 0

    print("=" * 90)
    print(f"Seen fragments              : {stats['seen_fragments']:,} / {total_to_scan:,}")
    print(f"Processed unannotated        : {stats['processed_unannotated']:,}")
    print(f"Skipped with annotations     : {stats['skipped_with_annotations']:,}")
    print(f"Skipped no photo             : {stats['skipped_no_photo']:,}")
    print(f"Failed images                : {stats['failed_images']:,}")
    print(f"Failed OCR                   : {stats['failed_ocr']:,}")
    print(f"Total crops                  : {stats['total_crops']:,}")
    print(f"Current CSV part             : {stats['csv_part_index']}")
    print(f"Rows in current CSV part     : {stats['rows_in_current_part']:,}")
    print(f"Seen fragments/sec           : {seen_rate:.2f}")
    print(f"Processed fragments/sec      : {processed_rate:.2f}")
    print(f"Crops/sec                    : {crop_rate:.2f}")
    print(f"Elapsed time                 : {elapsed / 3600:.2f} hours")
    print(f"Estimated remaining time     : {eta_seconds / 3600:.2f} hours")
    print("=" * 90)


# ============================================================
# CONNECT TO MONGODB
# ============================================================

print("Connecting to MongoDB...")

client = MongoClient(mongo_uri)
db = client[DB_NAME]
fs = gridfs.GridFS(db, collection="photos")

print("Connected to MongoDB:", DB_NAME)

total_fragments = db.fragments.count_documents({})
total_with_period = db.fragments.count_documents({
    "script.period": {"$exists": True, "$ne": None}
})

print("Total fragments:", total_fragments)
print("Fragments with script.period:", total_with_period)


# ============================================================
# LOAD ANNOTATED FRAGMENT IDS ONCE
# ============================================================

print("Loading annotated fragment IDs...")

annotated_fragment_ids = set()

ann_projection = {
    "fragmentId": 1,
    "fragmentNumber": 1,
    "museumNumber": 1,
    "fragment": 1,
    "fragments": 1,
}

ann_cursor = db.annotations.find(
    {},
    ann_projection,
    no_cursor_timeout=True
)

for ann in tqdm(ann_cursor, desc="Loading annotation references"):

    for key in ["fragmentId", "fragmentNumber", "museumNumber", "fragment"]:
        value = ann.get(key)

        if value is not None:
            annotated_fragment_ids.add(str(value))

    fragments = ann.get("fragments")

    if isinstance(fragments, list):
        for value in fragments:
            if value is not None:
                annotated_fragment_ids.add(str(value))

ann_cursor.close()

print("Annotated fragment references:", len(annotated_fragment_ids))
print("Approx. unannotated fragments:", total_fragments - len(annotated_fragment_ids))


# ============================================================
# LOAD DETR OCR MODEL
# ============================================================

register_all_modules()

cfg = Config.fromfile(DETR_CONFIG)
cfg.model.bbox_head.num_classes = 173

detr_model = init_detector(
    cfg,
    DETR_CHECKPOINT,
    device=DEVICE
)

class_names = detr_model.dataset_meta["classes"]

print("DETR OCR model loaded.")
print("Number of DETR classes:", len(class_names))
print("First 10 DETR classes:", class_names[:10])


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "_")
        .replace("?", "_")
        .replace("*", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
        .replace(",", "_")
    )


def get_fragment_number(fragment):
    return (
        fragment.get("_id")
        or fragment.get("museumNumber")
        or fragment.get("number")
    )


def has_manual_annotation(fragment):
    fragment_id = str(fragment.get("_id"))
    fragment_number = str(get_fragment_number(fragment))

    if fragment.get("annotations"):
        return True

    if fragment.get("signAnnotations"):
        return True

    if fragment_id in annotated_fragment_ids:
        return True

    if fragment_number in annotated_fragment_ids:
        return True

    return False


def get_photo_file_id(fragment):
    fragment_id = str(fragment.get("_id"))

    possible_filenames = [
        f"{fragment_id}.jpg",
        f"{fragment_id}.jpeg",
        f"{fragment_id}.png",
        fragment_id,
    ]

    photo_file = db["photos.files"].find_one({
        "filename": {"$in": possible_filenames}
    })

    if photo_file is None:
        return None

    return photo_file["_id"]


def load_image_from_gridfs(file_id):
    grid_file = fs.get(file_id)
    image_bytes = grid_file.read()

    image = Image.open(BytesIO(image_bytes)).convert("RGB")
    image.load()

    return image


def run_detr_on_image(pil_image, score_thr=0.60):
    image_np = np.array(pil_image)
    image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    result = inference_detector(detr_model, image_bgr)

    pred_instances = result.pred_instances

    bboxes = pred_instances.bboxes.cpu().numpy()
    scores = pred_instances.scores.cpu().numpy()
    labels = pred_instances.labels.cpu().numpy()

    detections = []

    for bbox, score, label in zip(bboxes, scores, labels):

        score = float(score)

        if score < score_thr:
            continue

        label_id = int(label)

        if 0 <= label_id < len(class_names):
            sign_name_detr = class_names[label_id]
        else:
            sign_name_detr = "UNKNOWN"


In [ ]:
# ============================================================
# RESNET TABLET-HOLDOUT SIGN CLASSIFICATION + PERIOD VOTING
#
# Runs:
#   1. ResNet18 tablet-holdout
#   2. ResNet50 tablet-holdout
#   3. ResNet101 tablet-holdout
#

# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import gc
import time
import textwrap
import importlib
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from torchvision import models
from torch.utils.data import DataLoader


# ============================================================
# PATH SETTINGS
# ============================================================

OCR_ROOT = Path(
    r"cuneiform-ocr-main"
)

OUTPUT_DIR = (
    OCR_ROOT
    / "mongo_unannotated_tablet_outputs"
)

CSV_PARTS_DIR = (
    OUTPUT_DIR
    / "csv_parts"
)

BASE_ROOT = Path(
    r"ebl_tablets_and_sign_crops"
)

SIGN_ROOT = (
    BASE_ROOT
    / "sign_classification_extension"
    / "ResNet"
)

LABEL_CSV = (
    SIGN_ROOT
    / "tablet_holdout_train.csv"
)

MODEL_OUTPUT_ROOT = (
    OUTPUT_DIR
    / "resnet_tablet_holdout_predictions"
)

MODEL_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

ALL_MODELS_SUMMARY_CSV = (
    MODEL_OUTPUT_ROOT
    / "all_resnet_tablet_holdout_period_vote_summary.csv"
)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_CONFIGS = [

    {
        "run_name": "resnet18_tablet_holdout",
        "display_name": "ResNet18 Tablet Holdout",
        "arch": "resnet18",
        "checkpoint_candidates": [
            SIGN_ROOT / "best_resnet18_tablet_holdout_sign_classifier.pth",
            SIGN_ROOT / "best_resnet18_tablet_holdout_classifier.pth",
            SIGN_ROOT / "best_resnet18_holdout_sign_classifier.pth",
            SIGN_ROOT / "best_resnet18_holdout.pth",
        ],
    },

    {
        "run_name": "resnet50_tablet_holdout",
        "display_name": "ResNet50 Tablet Holdout",
        "arch": "resnet50",
        "checkpoint_candidates": [
            SIGN_ROOT / "best_resnet50_tablet_holdout_sign_classifier.pth",
            SIGN_ROOT / "best_resnet50_tablet_holdout_classifier.pth",
            SIGN_ROOT / "best_resnet50_holdout_sign_classifier.pth",
            SIGN_ROOT / "best_resnet50_holdout.pth",
        ],
    },

    {
        "run_name": "resnet101_tablet_holdout",
        "display_name": "ResNet101 Tablet Holdout",
        "arch": "resnet101",
        "checkpoint_candidates": [
            SIGN_ROOT / "best_resnet101_tablet_holdout_sign_classifier.pth",
            SIGN_ROOT / "best_resnet101_tablet_holdout_classifier.pth",
            SIGN_ROOT / "best_resnet101_holdout_sign_classifier.pth",
            SIGN_ROOT / "best_resnet101_holdout.pth",
        ],
    },
]


# ============================================================
# SETTINGS
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 32

NUM_WORKERS = 8

PREFETCH_FACTOR = 4

WRITE_BUFFER_SIZE = 10_000

COMBINE_CHUNK_SIZE = 50_000

SIGN_CONFIDENCE_THRESHOLD = 0.50

OVERWRITE_EXISTING_PARTS = False

FALLBACK_TO_SINGLE_WORKER = True


# ============================================================
# DEVICE INFO
# ============================================================

print("=" * 90)
print("DEVICE AND SETTINGS")
print("=" * 90)
print("Device              :", DEVICE)
print("Batch size          :", BATCH_SIZE)
print("Workers             :", NUM_WORKERS)
print("Confidence threshold:", SIGN_CONFIDENCE_THRESHOLD)
print("Output dir          :", MODEL_OUTPUT_ROOT)

if DEVICE.type == "cuda":
    print("GPU                 :", torch.cuda.get_device_name(0))
    print("CUDA                :", torch.version.cuda)

print("=" * 90)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


# ============================================================
# PATH CHECKS
# ============================================================

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Missing output dir:\n{OUTPUT_DIR}")

if not CSV_PARTS_DIR.exists():
    raise FileNotFoundError(f"Missing csv_parts dir:\n{CSV_PARTS_DIR}")

if not SIGN_ROOT.exists():
    raise FileNotFoundError(f"Missing ResNet dir:\n{SIGN_ROOT}")

if not LABEL_CSV.exists():
    raise FileNotFoundError(f"Missing label CSV:\n{LABEL_CSV}")


# ============================================================
# FIND DETR CSV PARTS
# ============================================================

csv_part_files = sorted(
    CSV_PARTS_DIR.glob("detr_crops_part_*.csv")
)

if not csv_part_files:

    csv_part_files = sorted(
        OUTPUT_DIR.glob("detr_crops_part_*.csv")
    )

if not csv_part_files:

    raise FileNotFoundError(
        "No detr_crops_part_*.csv files found."
    )

print()
print("=" * 90)
print("DETR CROP CSV FILES")
print("=" * 90)
print("Number of parts:", len(csv_part_files))

for file in csv_part_files:
    print(" -", file.name)


# ============================================================
# CREATE WORKER MODULE
# ============================================================

WORKER_MODULE_NAME = "_cuneiform_crop_loader_holdout_only"

WORKER_MODULE_PATH = (
    OUTPUT_DIR
    / f"{WORKER_MODULE_NAME}.py"
)

worker_module_code = r'''
from pathlib import Path

import torch

from PIL import Image, ImageFile
from torchvision import transforms
from torch.utils.data import Dataset


Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True


class CropDataset(Dataset):

    def __init__(self, crop_paths, output_dir):

        self.crop_paths = [str(path) for path in crop_paths]

        self.output_dir = Path(output_dir)

        self.detr_crops_dir = self.output_dir / "detr_crops"

        self.transform = transforms.Compose([
            transforms.Resize(232),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])


    def __len__(self):

        return len(self.crop_paths)


    def resolve_crop_path(self, crop_path_value):

        crop_path_text = str(crop_path_value).strip()

        if not crop_path_text:
            return None

        crop_path = Path(crop_path_text)

        candidates = [
            crop_path,
            self.output_dir / crop_path,
            self.detr_crops_dir / crop_path,
            self.detr_crops_dir / crop_path.name
        ]

        for candidate in candidates:

            try:

                if candidate.exists():
                    return candidate

            except OSError:
                continue

        return None


    def __getitem__(self, index):

        original_crop_path = self.crop_paths[index]

        resolved_crop_path = self.resolve_crop_path(original_crop_path)

        if resolved_crop_path is None:

            return {
                "index": index,
                "image": None,
                "error": "Crop image file not found",
                "resolved_path": ""
            }

        try:

            with Image.open(resolved_crop_path) as image:

                image = image.convert("RGB")

                image_tensor = self.transform(image)

            return {
                "index": index,
                "image": image_tensor,
                "error": "",
                "resolved_path": str(resolved_crop_path)
            }

        except Exception as error:

            return {
                "index": index,
                "image": None,
                "error": str(error),
                "resolved_path": str(resolved_crop_path)
            }


def crop_collate_fn(batch):

    valid_images = []
    valid_indices = []
    valid_paths = []
    failed_items = []

    for item in batch:

        if item["image"] is None:

            failed_items.append({
                "index": int(item["index"]),
                "error": item["error"],
                "resolved_path": item["resolved_path"]
            })

        else:

            valid_images.append(item["image"])
            valid_indices.append(int(item["index"]))
            valid_paths.append(item["resolved_path"])

    if valid_images:
        image_batch = torch.stack(valid_images, dim=0)
    else:
        image_batch = None

    return {
        "images": image_batch,
        "indices": valid_indices,
        "resolved_paths": valid_paths,
        "failed": failed_items,
        "batch_size": len(batch)
    }


def dataloader_worker_init(worker_id):

    torch.set_num_threads(1)
'''

WORKER_MODULE_PATH.write_text(
    textwrap.dedent(worker_module_code),
    encoding="utf-8"
)

if str(OUTPUT_DIR) not in sys.path:

    sys.path.insert(0, str(OUTPUT_DIR))

importlib.invalidate_caches()

if WORKER_MODULE_NAME in sys.modules:

    del sys.modules[WORKER_MODULE_NAME]

worker_module = importlib.import_module(WORKER_MODULE_NAME)

CropDataset = worker_module.CropDataset
crop_collate_fn = worker_module.crop_collate_fn
dataloader_worker_init = worker_module.dataloader_worker_init

print()
print("Worker module created:")
print(WORKER_MODULE_PATH)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def first_existing_path(candidates):

    for path in candidates:

        path = Path(path)

        if path.exists():
            return path

    return None


def clean_checkpoint_state_dict(checkpoint):

    if isinstance(checkpoint, dict):

        if "model_state_dict" in checkpoint:
            checkpoint = checkpoint["model_state_dict"]

        elif "state_dict" in checkpoint:
            checkpoint = checkpoint["state_dict"]

    if not isinstance(checkpoint, dict):

        raise ValueError("Checkpoint is not a valid state_dict.")

    prefixes_to_strip = [
        "module.",
        "_orig_mod.",
        "model."
    ]

    cleaned = {}

    for key, value in checkpoint.items():

        new_key = key

        changed = True

        while changed:

            changed = False

            for prefix in prefixes_to_strip:

                if new_key.startswith(prefix):

                    new_key = new_key[len(prefix):]

                    changed = True

        cleaned[new_key] = value

    return cleaned


def load_checkpoint_state_dict(checkpoint_path):

    try:

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

    except TypeError:

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu"
        )

    return clean_checkpoint_state_dict(checkpoint)


def get_checkpoint_num_classes(state_dict):

    if "fc.weight" not in state_dict:

        raise ValueError("Could not find fc.weight in checkpoint.")

    return int(state_dict["fc.weight"].shape[0])


def build_resnet_model(arch, num_classes):

    if arch == "resnet18":

        model = models.resnet18(weights=None)

    elif arch == "resnet50":

        model = models.resnet50(weights=None)

    elif arch == "resnet101":

        model = models.resnet101(weights=None)

    else:

        raise ValueError(f"Unsupported architecture: {arch}")

    model.fc = nn.Linear(
        model.fc.in_features,
        num_classes
    )

    return model


def load_label_mapping(label_csv):

    df = pd.read_csv(
        label_csv,
        dtype=str,
        keep_default_na=False
    )

    required_columns = {
        "label",
        "signName",
        "period"
    }

    missing = required_columns - set(df.columns)

    if missing:

        raise ValueError(
            f"Missing columns in label CSV:\n{sorted(missing)}\n\n"
            f"File:\n{label_csv}\n\n"
            f"Available columns:\n{list(df.columns)}"
        )

    df["label"] = df["label"].astype(str).str.strip()

    df = df[df["label"] != ""].copy()

    labels = sorted(df["label"].unique())

    label_to_idx = {
        label: idx
        for idx, label in enumerate(labels)
    }

    idx_to_label = {
        idx: label
        for label, idx in label_to_idx.items()
    }

    label_info = (
        df[["label", "signName", "period"]]
        .drop_duplicates(subset=["label"])
        .set_index("label")
        .to_dict("index")
    )

    return {
        "df": df,
        "labels": labels,
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "label_info": label_info,
        "num_classes": len(labels)
    }


def load_model_for_run(model_cfg, checkpoint_path, num_classes):

    state_dict = load_checkpoint_state_dict(checkpoint_path)

    checkpoint_num_classes = get_checkpoint_num_classes(state_dict)

    print("Classes from label CSV :", num_classes)
    print("Classes from checkpoint:", checkpoint_num_classes)

    if checkpoint_num_classes != num_classes:

        raise RuntimeError(
            "Class-count mismatch.\n\n"
            f"Model: {model_cfg['run_name']}\n"
            f"Checkpoint: {checkpoint_path}\n"
            f"Label CSV classes: {num_classes}\n"
            f"Checkpoint classes: {checkpoint_num_classes}\n\n"
            "This model cannot use tablet_holdout_train.csv."
        )

    model = build_resnet_model(
        model_cfg["arch"],
        num_classes
    )

    model.load_state_dict(
        state_dict,
        strict=True
    )

    model = model.to(DEVICE)

    if DEVICE.type == "cuda":

        model = model.to(memory_format=torch.channels_last)

    model.eval()

    return model


def warmup_model(model):

    if DEVICE.type != "cuda":
        return

    print()
    print("Warming up GPU...")

    warmup_batch = torch.zeros(
        (BATCH_SIZE, 3, 224, 224),
        device=DEVICE
    )

    warmup_batch = warmup_batch.contiguous(
        memory_format=torch.channels_last
    )

    with torch.inference_mode():

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            _ = model(warmup_batch)

    torch.cuda.synchronize()

    del warmup_batch

    torch.cuda.empty_cache()

    print("GPU warm-up completed.")


def create_dataloader(dataset, num_workers):

    loader_args = {
        "dataset": dataset,
        "batch_size": BATCH_SIZE,
        "shuffle": False,
        "num_workers": num_workers,
        "pin_memory": DEVICE.type == "cuda",
        "drop_last": False,
        "collate_fn": crop_collate_fn
    }

    if num_workers > 0:

        loader_args.update({
            "prefetch_factor": PREFETCH_FACTOR,
            "persistent_workers": False,
            "worker_init_fn": dataloader_worker_init
        })

    return DataLoader(**loader_args)


def append_rows_to_csv(rows, output_path, header_written):

    if not rows:
        return header_written

    pd.DataFrame(rows).to_csv(
        output_path,
        mode="a",
        index=False,
        header=not header_written
    )

    return True


# ============================================================
# CLASSIFICATION FUNCTIONS
# ============================================================

def classify_tensor_batch(
    image_batch,
    model,
    idx_to_label,
    label_info
):

    if image_batch is None:
        return []

    image_batch = image_batch.to(
        DEVICE,
        non_blocking=True
    )

    if DEVICE.type == "cuda":

        image_batch = image_batch.contiguous(
            memory_format=torch.channels_last
        )

    with torch.inference_mode():

        if DEVICE.type == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(image_batch)

        else:

            logits = model(image_batch)

        probabilities = torch.softmax(logits, dim=1)

        top_probabilities, top_indices = probabilities.max(dim=1)

    top_probabilities = (
        top_probabilities
        .detach()
        .float()
        .cpu()
        .tolist()
    )

    top_indices = (
        top_indices
        .detach()
        .cpu()
        .tolist()
    )

    results = []

    for pred_idx, pred_conf in zip(
        top_indices,
        top_probabilities
    ):

        pred_idx = int(pred_idx)

        pred_label = idx_to_label[pred_idx]

        metadata = label_info.get(pred_label, {})

        results.append({
            "pred_label": pred_label,
            "pred_signName": metadata.get("signName", ""),
            "pred_period": metadata.get("period", ""),
            "pred_confidence": float(pred_conf)
        })

    return results


def process_prediction_part(
    input_csv,
    num_workers,
    model,
    idx_to_label,
    label_info,
    prediction_parts_dir,
    failed_crops_dir
):

    part_suffix = input_csv.stem.replace(
        "detr_crops_part_",
        ""
    )

    output_part_csv = (
        prediction_parts_dir
        / f"crop_predictions_part_{part_suffix}.csv"
    )

    temporary_output_csv = Path(str(output_part_csv) + ".tmp")

    failed_part_csv = (
        failed_crops_dir
        / f"failed_crops_part_{part_suffix}.csv"
    )

    if output_part_csv.exists() and not OVERWRITE_EXISTING_PARTS:

        print("Prediction part already exists. Skipping:")
        print(output_part_csv)

        return {
            "status": "skipped",
            "output_path": output_part_csv,
            "input_rows": 0,
            "predictions": 0,
            "failures": 0,
            "elapsed_seconds": 0.0,
            "gpu_batches": 0
        }

    if output_part_csv.exists():
        output_part_csv.unlink()

    if temporary_output_csv.exists():
        temporary_output_csv.unlink()

    if failed_part_csv.exists():
        failed_part_csv.unlink()

    print()
    print("Reading:")
    print(input_csv)

    part_df = pd.read_csv(
        input_csv,
        dtype=str,
        keep_default_na=False
    )

    if "cropPath" not in part_df.columns:

        raise ValueError(
            f"'cropPath' column missing in:\n{input_csv}"
        )

    number_of_rows = len(part_df)

    source_records = part_df.to_dict(orient="records")

    crop_paths = part_df["cropPath"].astype(str).tolist()

    del part_df

    gc.collect()

    dataset = CropDataset(
        crop_paths=crop_paths,
        output_dir=str(OUTPUT_DIR)
    )

    loader = create_dataloader(
        dataset,
        num_workers
    )

    prediction_buffer = []
    failure_buffer = []

    prediction_header_written = False
    failure_header_written = False

    total_predictions = 0
    total_failures = 0
    total_gpu_batches = 0

    start_time = time.time()

    progress_bar = tqdm(
        total=number_of_rows,
        desc=input_csv.name,
        unit="crop"
    )

    try:

        for loader_batch in loader:

            current_batch_size = int(loader_batch["batch_size"])

            for failed_item in loader_batch["failed"]:

                failed_index = int(failed_item["index"])

                failed_row = dict(source_records[failed_index])

                failed_row.update({
                    "failure_reason": failed_item["error"],
                    "resolved_crop_path": failed_item["resolved_path"]
                })

                failure_buffer.append(failed_row)

                total_failures += 1

            image_batch = loader_batch["images"]

            valid_indices = loader_batch["indices"]

            resolved_paths = loader_batch["resolved_paths"]

            if image_batch is not None:

                batch_predictions = classify_tensor_batch(
                    image_batch=image_batch,
                    model=model,
                    idx_to_label=idx_to_label,
                    label_info=label_info
                )

                total_gpu_batches += 1

                for source_index, resolved_path, prediction in zip(
                    valid_indices,
                    resolved_paths,
                    batch_predictions
                ):

                    prediction_row = dict(
                        source_records[int(source_index)]
                    )

                    prediction_row.update(prediction)

                    prediction_row["resolved_crop_path"] = resolved_path

                    prediction_buffer.append(prediction_row)

                    total_predictions += 1

            if len(prediction_buffer) >= WRITE_BUFFER_SIZE:

                prediction_header_written = append_rows_to_csv(
                    prediction_buffer,
                    temporary_output_csv,
                    prediction_header_written
                )

                prediction_buffer = []

            if len(failure_buffer) >= WRITE_BUFFER_SIZE:

                failure_header_written = append_rows_to_csv(
                    failure_buffer,
                    failed_part_csv,
                    failure_header_written
                )

                failure_buffer = []

            progress_bar.update(current_batch_size)

            elapsed = time.time() - start_time

            crops_per_second = (
                progress_bar.n / elapsed
                if elapsed > 0
                else 0.0
            )

            progress_bar.set_postfix({
                "GPU_batches": total_gpu_batches,
                "predicted": total_predictions,
                "failed": total_failures,
                "crops/s": f"{crops_per_second:.1f}"
            })

    finally:

        progress_bar.close()

    prediction_header_written = append_rows_to_csv(
        prediction_buffer,
        temporary_output_csv,
        prediction_header_written
    )

    failure_header_written = append_rows_to_csv(
        failure_buffer,
        failed_part_csv,
        failure_header_written
    )

    if not temporary_output_csv.exists():

        raise RuntimeError(
            f"No predictions produced for:\n{input_csv}"
        )

    os.replace(
        temporary_output_csv,
        output_part_csv
    )

    elapsed = time.time() - start_time

    del loader
    del dataset
    del source_records
    del crop_paths

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "status": "completed",
        "output_path": output_part_csv,
        "input_rows": number_of_rows,
        "predictions": total_predictions,
        "failures": total_failures,
        "elapsed_seconds": elapsed,
        "gpu_batches": total_gpu_batches
    }


def combine_prediction_parts(
    prediction_parts_dir,
    crop_predictions_csv
):

    prediction_part_files = sorted(
        prediction_parts_dir.glob("crop_predictions_part_*.csv")
    )

    if not prediction_part_files:

        raise RuntimeError(
            f"No prediction parts found in:\n{prediction_parts_dir}"
        )

    tmp_csv = Path(str(crop_predictions_csv) + ".tmp")

    if tmp_csv.exists():
        tmp_csv.unlink()

    header_written = False

    total_rows = 0

    for part_file in tqdm(
        prediction_part_files,
        desc="Combining prediction files",
        unit="file"
    ):

        reader = pd.read_csv(
            part_file,
            dtype=str,
            keep_default_na=False,
            chunksize=COMBINE_CHUNK_SIZE
        )

        for chunk in reader:

            chunk.to_csv(
                tmp_csv,
                mode="a",
                index=False,
                header=not header_written
            )

            header_written = True

            total_rows += len(chunk)

    os.replace(tmp_csv, crop_predictions_csv)

    return prediction_part_files, total_rows


# ============================================================
# VOTING
# ============================================================

def run_tablet_level_period_voting(
    crop_predictions_csv,
    tablet_period_votes_csv,
    period_details_csv
):

    all_predictions_df = pd.read_csv(
        crop_predictions_csv,
        dtype=str,
        keep_default_na=False
    )

    required_cols = {
        "fragmentNumber",
        "period",
        "pred_period",
        "pred_confidence"
    }

    missing = required_cols - set(all_predictions_df.columns)

    if missing:

        raise ValueError(
            f"Missing columns in predictions:\n{sorted(missing)}"
        )

    all_predictions_df["pred_confidence"] = pd.to_numeric(
        all_predictions_df["pred_confidence"],
        errors="coerce"
    )

    all_predictions_df = (
        all_predictions_df
        .dropna(
            subset=[
                "fragmentNumber",
                "pred_period",
                "pred_confidence"
            ]
        )
        .copy()
    )

    all_predictions_df["fragmentNumber"] = (
        all_predictions_df["fragmentNumber"]
        .astype(str)
        .str.strip()
    )

    all_predictions_df["pred_period"] = (
        all_predictions_df["pred_period"]
        .astype(str)
        .str.strip()
    )

    all_predictions_df = (
        all_predictions_df[
            (all_predictions_df["fragmentNumber"] != "")
            &
            (all_predictions_df["pred_period"] != "")
        ]
        .copy()
    )

    fragment_counts_before_threshold = (
        all_predictions_df
        .groupby("fragmentNumber")
        .size()
        .to_dict()
    )

    total_fragments_before_threshold = (
        all_predictions_df["fragmentNumber"].nunique()
    )

    predictions_before_threshold = len(all_predictions_df)

    pred_df = (
        all_predictions_df[
            all_predictions_df["pred_confidence"]
            >= SIGN_CONFIDENCE_THRESHOLD
        ]
        .copy()
    )

    predictions_after_threshold = len(pred_df)

    evaluated_fragment_count = (
        pred_df["fragmentNumber"].nunique()
    )

    fragments_without_confident_crops = (
        total_fragments_before_threshold
        - evaluated_fragment_count
    )

    print()
    print("=" * 90)
    print("TABLET-LEVEL VOTING INPUT")
    print("=" * 90)
    print("Predictions before filtering:", f"{predictions_before_threshold:,}")
    print("Predictions after filtering :", f"{predictions_after_threshold:,}")
    print("Fragments before filtering  :", f"{total_fragments_before_threshold:,}")
    print("Fragments evaluated         :", f"{evaluated_fragment_count:,}")
    print("Fragments without confident :", f"{fragments_without_confident_crops:,}")

    if len(pred_df) == 0:

        raise RuntimeError(
            "No predictions remain after threshold filtering."
        )

    tablet_rows = []
    detail_rows = []

    grouped = pred_df.groupby(
        "fragmentNumber",
        sort=False
    )

    for fragment_number, group in tqdm(
        grouped,
        total=evaluated_fragment_count,
        desc="Tablet-level period voting",
        unit="fragment"
    ):

        period_values = (
            group["period"]
            .astype(str)
            .str.strip()
        )

        period_values = period_values[
            period_values != ""
        ]

        if len(period_values) > 0:

            true_period = period_values.mode().iloc[0]

        else:

            true_period = ""

        period_summary = (
            group
            .groupby("pred_period", dropna=False)
            .agg(
                num_votes=("pred_period", "count"),
                mean_confidence=("pred_confidence", "mean"),
                weighted_score=("pred_confidence", "sum")
            )
            .reset_index()
        )

        period_summary = (
            period_summary
            .sort_values(
                by=[
                    "weighted_score",
                    "num_votes",
                    "mean_confidence",
                    "pred_period"
                ],
                ascending=[
                    False,
                    False,
                    False,
                    True
                ]
            )
            .reset_index(drop=True)
        )

        total_weight = period_summary["weighted_score"].sum()

        if total_weight > 0:

            period_summary["vote_probability"] = (
                period_summary["weighted_score"]
                / total_weight
            )

        else:

            period_summary["vote_probability"] = 0.0

        best_row = period_summary.iloc[0]

        predicted_period = str(best_row["pred_period"])

        correct = true_period == predicted_period

        if len(period_summary) > 1:

            second_vote_probability = float(
                period_summary.iloc[1]["vote_probability"]
            )

        else:

            second_vote_probability = 0.0

        winning_vote_probability = float(
            best_row["vote_probability"]
        )

        vote_margin = (
            winning_vote_probability
            - second_vote_probability
        )

        tablet_rows.append({
            "fragmentNumber": fragment_number,
            "true_period": true_period,
            "predicted_period": predicted_period,
            "num_crops_before_threshold": int(
                fragment_counts_before_threshold.get(
                    fragment_number,
                    len(group)
                )
            ),
            "num_crops_used": int(len(group)),
            "winning_votes": int(best_row["num_votes"]),
            "winning_mean_confidence": float(best_row["mean_confidence"]),
            "winning_weighted_score": float(best_row["weighted_score"]),
            "winning_vote_probability": winning_vote_probability,
            "second_vote_probability": second_vote_probability,
            "vote_margin": float(vote_margin),
            "correct": bool(correct)
        })

        for rank, row in period_summary.iterrows():

            detail_rows.append({
                "fragmentNumber": fragment_number,
                "true_period": true_period,
                "rank": int(rank + 1),
                "candidate_period": str(row["pred_period"]),
                "num_votes": int(row["num_votes"]),
                "mean_confidence": float(row["mean_confidence"]),
                "weighted_score": float(row["weighted_score"]),
                "vote_probability": float(row["vote_probability"])
            })

    tablet_votes_df = pd.DataFrame(tablet_rows)

    period_details_df = pd.DataFrame(detail_rows)

    tablet_votes_df.to_csv(
        tablet_period_votes_csv,
        index=False
    )

    period_details_df.to_csv(
        period_details_csv,
        index=False
    )

    if len(tablet_votes_df) > 0:

        tablet_votes_df["correct"] = (
            tablet_votes_df["correct"].astype(bool)
        )

        accuracy = tablet_votes_df["correct"].mean()

        number_correct = int(tablet_votes_df["correct"].sum())

        number_evaluated = len(tablet_votes_df)

        number_incorrect = number_evaluated - number_correct

    else:

        accuracy = 0.0
        number_correct = 0
        number_evaluated = 0
        number_incorrect = 0

    return {
        "predictions_before_threshold": int(predictions_before_threshold),
        "predictions_after_threshold": int(predictions_after_threshold),
        "predictions_removed": int(predictions_before_threshold - predictions_after_threshold),
        "fragments_before_threshold": int(total_fragments_before_threshold),
        "fragments_evaluated": int(number_evaluated),
        "fragments_without_confident_crops": int(fragments_without_confident_crops),
        "correct_period_predictions": int(number_correct),
        "incorrect_period_predictions": int(number_incorrect),
        "tablet_level_accuracy": float(accuracy),
        "tablet_votes_df": tablet_votes_df
    }


# ============================================================
# RUN ONE MODEL
# ============================================================

def run_one_model(model_cfg):

    run_name = model_cfg["run_name"]

    display_name = model_cfg["display_name"]

    print()
    print("#" * 100)
    print("STARTING MODEL:", display_name)
    print("#" * 100)

    checkpoint_path = first_existing_path(
        model_cfg["checkpoint_candidates"]
    )

    if checkpoint_path is None:

        raise FileNotFoundError(
            f"Could not find checkpoint for {run_name}"
        )

    label_bundle = load_label_mapping(LABEL_CSV)

    idx_to_label = label_bundle["idx_to_label"]

    label_info = label_bundle["label_info"]

    num_classes = label_bundle["num_classes"]

    print("Label CSV             :", LABEL_CSV)
    print("Number of label classes:", num_classes)
    print("Checkpoint            :", checkpoint_path)

    model = load_model_for_run(
        model_cfg=model_cfg,
        checkpoint_path=checkpoint_path,
        num_classes=num_classes
    )

    warmup_model(model)

    model_output_dir = MODEL_OUTPUT_ROOT / run_name

    prediction_parts_dir = (
        model_output_dir
        / "prediction_parts"
    )

    failed_crops_dir = (
        model_output_dir
        / "failed_prediction_crops"
    )

    prediction_parts_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    failed_crops_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    crop_predictions_csv = (
        model_output_dir
        / "mongo_unannotated_crop_predictions.csv"
    )

    tablet_period_votes_csv = (
        model_output_dir
        / "mongo_unannotated_tablet_period_votes.csv"
    )

    period_details_csv = (
        model_output_dir
        / "mongo_unannotated_period_vote_details.csv"
    )

    for part_idx, input_csv in enumerate(
        csv_part_files,
        start=1
    ):

        print()
        print("=" * 90)
        print(f"PROCESSING PART {part_idx}/{len(csv_part_files)}")
        print("Model:", display_name)
        print("Input:", input_csv)
        print("=" * 90)

        try:

            stats = process_prediction_part(
                input_csv=input_csv,
                num_workers=NUM_WORKERS,
                model=model,
                idx_to_label=idx_to_label,
                label_info=label_info,
                prediction_parts_dir=prediction_parts_dir,
                failed_crops_dir=failed_crops_dir
            )

        except Exception as error:

            if NUM_WORKERS > 0 and FALLBACK_TO_SINGLE_WORKER:

                print()
                print("Multi-worker failed. Retrying with num_workers=0.")
                print("Original error:", repr(error))

                stats = process_prediction_part(
                    input_csv=input_csv,
                    num_workers=0,
                    model=model,
                    idx_to_label=idx_to_label,
                    label_info=label_info,
                    prediction_parts_dir=prediction_parts_dir,
                    failed_crops_dir=failed_crops_dir
                )

            else:

                raise

        print("Part status:", stats["status"])

        if stats["status"] == "completed":

            print("Predictions:", f'{stats["predictions"]:,}')
            print("Failures   :", f'{stats["failures"]:,}')
            print("Elapsed min:", f'{stats["elapsed_seconds"] / 60:.2f}')

    print()
    print("=" * 90)
    print("COMBINING PREDICTION PARTS")
    print("=" * 90)

    prediction_part_files, combined_rows = combine_prediction_parts(
        prediction_parts_dir,
        crop_predictions_csv
    )

    print("Combined rows:", f"{combined_rows:,}")

    voting_stats = run_tablet_level_period_voting(
        crop_predictions_csv,
        tablet_period_votes_csv,
        period_details_csv
    )

    print()
    print("=" * 90)
    print("FINAL RESULTS:", display_name)
    print("=" * 90)
    print("Total crop predictions           :", f"{combined_rows:,}")
    print("Crop predictions used            :", f'{voting_stats["predictions_after_threshold"]:,}')
    print("Fragments before threshold       :", f'{voting_stats["fragments_before_threshold"]:,}')
    print("Fragments evaluated              :", f'{voting_stats["fragments_evaluated"]:,}')
    print("Fragments without confident crops:", f'{voting_stats["fragments_without_confident_crops"]:,}')
    print("Correct period predictions       :", f'{voting_stats["correct_period_predictions"]:,}')
    print("Incorrect period predictions     :", f'{voting_stats["incorrect_period_predictions"]:,}')
    print("Tablet-level accuracy            :", f'{voting_stats["tablet_level_accuracy"]:.4f}')
    print("Tablet-level accuracy (%)        :", f'{voting_stats["tablet_level_accuracy"] * 100:.2f}%')

    summary_row = {
        "run_name": run_name,
        "display_name": display_name,
        "architecture": model_cfg["arch"],
        "label_csv": str(LABEL_CSV),
        "checkpoint": str(checkpoint_path),
        "output_dir": str(model_output_dir),
        "detr_csv_parts": int(len(csv_part_files)),
        "prediction_csv_parts": int(len(prediction_part_files)),
        "combined_prediction_rows": int(combined_rows),
        "predictions_before_threshold": voting_stats["predictions_before_threshold"],
        "predictions_after_threshold": voting_stats["predictions_after_threshold"],
        "predictions_removed": voting_stats["predictions_removed"],
        "fragments_before_threshold": voting_stats["fragments_before_threshold"],
        "fragments_evaluated": voting_stats["fragments_evaluated"],
        "fragments_without_confident_crops": voting_stats["fragments_without_confident_crops"],
        "correct_period_predictions": voting_stats["correct_period_predictions"],
        "incorrect_period_predictions": voting_stats["incorrect_period_predictions"],
        "tablet_level_accuracy": voting_stats["tablet_level_accuracy"],
        "tablet_level_accuracy_percent": voting_stats["tablet_level_accuracy"] * 100.0,
        "confidence_threshold": float(SIGN_CONFIDENCE_THRESHOLD),
    }

    del model

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return summary_row


# ============================================================
# RUN ALL HOLDOUT MODELS
# ============================================================

print()
print("#" * 100)
print("STARTING RESNET TABLET-HOLDOUT RUNS")
print("#" * 100)

summary_rows = []

for model_cfg in MODEL_CONFIGS:

    summary_row = run_one_model(model_cfg)

    summary_rows.append(summary_row)

    summary_df = pd.DataFrame(summary_rows)

    summary_df.to_csv(
        ALL_MODELS_SUMMARY_CSV,
        index=False
    )

    print()
    print("Intermediate summary saved:")
    print(ALL_MODELS_SUMMARY_CSV)


summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    ALL_MODELS_SUMMARY_CSV,
    index=False
)

print()
print("#" * 100)
print("ALL TABLET-HOLDOUT RUNS COMPLETED")
print("#" * 100)

print()
print("Summary saved:")
print(ALL_MODELS_SUMMARY_CSV)

try:

    display(summary_df)

except NameError:

    print(summary_df.to_string(index=False))

In [ ]:
# ============================================================
# PERIOD ACCURACY BY NUMBER OF CONFIDENT CROPS
# FOR ALL RESNET TABLET-HOLDOUT MODELS
#
# GROUPS:
#   1 confident crop
#   2-4 confident crops
#   5-9 confident crops
#   10 or more confident crops
#
# INPUT:
#   mongo_unannotated_tablet_outputs/
#       resnet_tablet_holdout_predictions/
#           resnet18_tablet_holdout/
#               mongo_unannotated_tablet_period_votes.csv
#           resnet50_tablet_holdout/
#               mongo_unannotated_tablet_period_votes.csv
#           resnet101_tablet_holdout/
#               mongo_unannotated_tablet_period_votes.csv
#
# OUTPUT:
#   Each model folder:
#       period_accuracy_by_crop_count.csv
#
#   Combined:
#       all_resnet_tablet_holdout_accuracy_by_crop_count.csv
# ============================================================

from pathlib import Path

import pandas as pd


# ============================================================
# PATHS
# ============================================================

OUTPUT_DIR = Path(
    r"cuneiform-ocr-main"
    r"\mongo_unannotated_tablet_outputs"
)

MODEL_OUTPUT_ROOT = (
    OUTPUT_DIR
    / "resnet_tablet_holdout_predictions"
)

COMBINED_ACCURACY_CSV = (
    MODEL_OUTPUT_ROOT
    / "all_resnet_tablet_holdout_accuracy_by_crop_count.csv"
)


# ============================================================
# MODELS TO PROCESS
# ============================================================

MODEL_CONFIGS = [

    {
        "run_name": "resnet18_tablet_holdout",
        "display_name": "ResNet18 Tablet Holdout",
    },

    {
        "run_name": "resnet50_tablet_holdout",
        "display_name": "ResNet50 Tablet Holdout",
    },

    {
        "run_name": "resnet101_tablet_holdout",
        "display_name": "ResNet101 Tablet Holdout",
    },
]


# ============================================================
# CROP-COUNT GROUP FUNCTION
# ============================================================

def assign_crop_count_group(num_crops):
    """
    Assign each fragment to one of four evidence groups.
    """

    if num_crops == 1:
        return "1 confident crop"

    if 2 <= num_crops <= 4:
        return "2â€“4 confident crops"

    if 5 <= num_crops <= 9:
        return "5â€“9 confident crops"

    if num_crops >= 10:
        return "10 or more confident crops"

    return "No confident crops"


GROUP_ORDER = [
    "1 confident crop",
    "2â€“4 confident crops",
    "5â€“9 confident crops",
    "10 or more confident crops"
]


# ============================================================
# BOOLEAN CLEANING
# ============================================================

def clean_correct_column(series):
    """
    Convert correct column safely to boolean.
    Handles:
        True / False
        true / false
        1 / 0
        yes / no
    """

    if series.dtype == bool:
        return series

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False
        })
    )


# ============================================================
# PROCESS ONE MODEL
# ============================================================

def process_one_model(model_cfg):
    """
    Calculate period accuracy by number of confident crops
    for one model.
    """

    run_name = model_cfg["run_name"]
    display_name = model_cfg["display_name"]

    model_dir = (
        MODEL_OUTPUT_ROOT
        / run_name
    )

    tablet_period_votes_csv = (
        model_dir
        / "mongo_unannotated_tablet_period_votes.csv"
    )

    accuracy_by_crop_count_csv = (
        model_dir
        / "period_accuracy_by_crop_count.csv"
    )

    if not tablet_period_votes_csv.exists():

        print()
        print("=" * 100)
        print("SKIPPING MODEL â€” FILE NOT FOUND")
        print("=" * 100)
        print("Model:", display_name)
        print("Missing file:")
        print(tablet_period_votes_csv)

        return None

    print()
    print("=" * 100)
    print("PROCESSING MODEL")
    print("=" * 100)
    print("Model:", display_name)
    print("Input:", tablet_period_votes_csv)

    # --------------------------------------------------------
    # LOAD TABLET-LEVEL RESULTS
    # --------------------------------------------------------

    tablet_votes_df = pd.read_csv(
        tablet_period_votes_csv,
        dtype={
            "fragmentNumber": str,
            "true_period": str,
            "predicted_period": str
        },
        keep_default_na=False
    )

    print(
        "Loaded tablet-level predictions:",
        f"{len(tablet_votes_df):,}"
    )

    # --------------------------------------------------------
    # CHECK REQUIRED COLUMNS
    # --------------------------------------------------------

    required_columns = {
        "fragmentNumber",
        "true_period",
        "predicted_period",
        "num_crops_used",
        "correct"
    }

    missing_columns = (
        required_columns
        - set(tablet_votes_df.columns)
    )

    if missing_columns:

        raise ValueError(
            f"Missing columns in:\n{tablet_period_votes_csv}\n\n"
            f"Missing columns:\n{sorted(missing_columns)}\n\n"
            f"Available columns:\n{list(tablet_votes_df.columns)}"
        )

    # --------------------------------------------------------
    # CLEAN REQUIRED COLUMNS
    # --------------------------------------------------------

    tablet_votes_df["num_crops_used"] = pd.to_numeric(
        tablet_votes_df["num_crops_used"],
        errors="coerce"
    )

    tablet_votes_df["correct"] = clean_correct_column(
        tablet_votes_df["correct"]
    )

    tablet_votes_df = tablet_votes_df.dropna(
        subset=[
            "num_crops_used",
            "correct"
        ]
    ).copy()

    tablet_votes_df["num_crops_used"] = (
        tablet_votes_df["num_crops_used"]
        .astype(int)
    )

    tablet_votes_df["correct"] = (
        tablet_votes_df["correct"]
        .astype(bool)
    )

    if len(tablet_votes_df) == 0:

        raise RuntimeError(
            f"No valid rows remain after cleaning:\n{tablet_period_votes_csv}"
        )

    # --------------------------------------------------------
    # ASSIGN CROP-COUNT GROUP
    # --------------------------------------------------------

    tablet_votes_df["crop_count_group"] = (
        tablet_votes_df["num_crops_used"]
        .apply(assign_crop_count_group)
    )

    tablet_votes_df["crop_count_group"] = pd.Categorical(
        tablet_votes_df["crop_count_group"],
        categories=GROUP_ORDER,
        ordered=True
    )

    # Remove anything outside the four main groups.
    tablet_votes_df = (
        tablet_votes_df[
            tablet_votes_df["crop_count_group"].notna()
        ]
        .copy()
    )

    # --------------------------------------------------------
    # CALCULATE ACCURACY FOR EACH GROUP
    # --------------------------------------------------------

    accuracy_by_crop_count = (
        tablet_votes_df
        .groupby(
            "crop_count_group",
            observed=False
        )
        .agg(
            num_fragments=(
                "fragmentNumber",
                "count"
            ),
            correct_predictions=(
                "correct",
                "sum"
            ),
            mean_crops_used=(
                "num_crops_used",
                "mean"
            ),
            median_crops_used=(
                "num_crops_used",
                "median"
            )
        )
        .reset_index()
    )

    accuracy_by_crop_count["incorrect_predictions"] = (
        accuracy_by_crop_count["num_fragments"]
        - accuracy_by_crop_count["correct_predictions"]
    )

    accuracy_by_crop_count["accuracy"] = (
        accuracy_by_crop_count["correct_predictions"]
        / accuracy_by_crop_count["num_fragments"]
    )

    accuracy_by_crop_count["accuracy_percent"] = (
        accuracy_by_crop_count["accuracy"]
        * 100
    )

    accuracy_by_crop_count["percentage_of_evaluated_fragments"] = (
        accuracy_by_crop_count["num_fragments"]
        / len(tablet_votes_df)
        * 100
    )

    # --------------------------------------------------------
    # ADD MODEL INFORMATION
    # --------------------------------------------------------

    accuracy_by_crop_count.insert(
        0,
        "run_name",
        run_name
    )

    accuracy_by_crop_count.insert(
        1,
        "model",
        display_name
    )

    accuracy_by_crop_count.insert(
        2,
        "total_evaluated_fragments",
        len(tablet_votes_df)
    )

    # --------------------------------------------------------
    # FORMAT NUMERIC COLUMNS
    # --------------------------------------------------------

    accuracy_by_crop_count["num_fragments"] = (
        accuracy_by_crop_count["num_fragments"]
        .fillna(0)
        .astype(int)
    )

    accuracy_by_crop_count["correct_predictions"] = (
        accuracy_by_crop_count["correct_predictions"]
        .fillna(0)
        .astype(int)
    )

    accuracy_by_crop_count["incorrect_predictions"] = (
        accuracy_by_crop_count["incorrect_predictions"]
        .fillna(0)
        .astype(int)
    )

    accuracy_by_crop_count["mean_crops_used"] = (
        accuracy_by_crop_count["mean_crops_used"]
        .round(2)
    )

    accuracy_by_crop_count["median_crops_used"] = (
        accuracy_by_crop_count["median_crops_used"]
        .round(2)
    )

    accuracy_by_crop_count["accuracy"] = (
        accuracy_by_crop_count["accuracy"]
        .round(4)
    )

    accuracy_by_crop_count["accuracy_percent"] = (
        accuracy_by_crop_count["accuracy_percent"]
        .round(2)
    )

    accuracy_by_crop_count[
        "percentage_of_evaluated_fragments"
    ] = (
        accuracy_by_crop_count[
            "percentage_of_evaluated_fragments"
        ]
        .round(2)
    )

    # --------------------------------------------------------
    # SELECT OUTPUT COLUMN ORDER
    # --------------------------------------------------------

    accuracy_by_crop_count = accuracy_by_crop_count[
        [
            "run_name",
            "model",
            "total_evaluated_fragments",
            "crop_count_group",
            "num_fragments",
            "correct_predictions",
            "incorrect_predictions",
            "accuracy",
            "accuracy_percent",
            "percentage_of_evaluated_fragments",
            "mean_crops_used",
            "median_crops_used"
        ]
    ]

    # --------------------------------------------------------
    # SAVE MODEL-SPECIFIC RESULTS
    # --------------------------------------------------------

    accuracy_by_crop_count.to_csv(
        accuracy_by_crop_count_csv,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("Saved model result:")
    print(accuracy_by_crop_count_csv)

    # --------------------------------------------------------
    # DISPLAY MODEL RESULT
    # --------------------------------------------------------

    print()
    print("=" * 100)
    print(f"PERIOD ACCURACY BY NUMBER OF CONFIDENT CROPS â€” {display_name}")
    print("=" * 100)

    try:
        display(accuracy_by_crop_count)

    except NameError:
        print(
            accuracy_by_crop_count.to_string(
                index=False
            )
        )

    # --------------------------------------------------------
    # PRINT SIMPLE SUMMARY
    # --------------------------------------------------------

    print()

    for _, row in accuracy_by_crop_count.iterrows():

        print(
            f"{row['model']} | "
            f"{row['crop_count_group']}: "
            f"{int(row['num_fragments']):,} fragments | "
            f"{int(row['correct_predictions']):,} correct | "
            f"{int(row['incorrect_predictions']):,} incorrect | "
            f"accuracy = {row['accuracy_percent']:.2f}%"
        )

    return accuracy_by_crop_count


# ============================================================
# RUN ALL MODELS
# ============================================================

all_accuracy_tables = []

for model_cfg in MODEL_CONFIGS:

    result_df = process_one_model(
        model_cfg
    )

    if result_df is not None:

        all_accuracy_tables.append(
            result_df
        )


# ============================================================
# COMBINE ALL MODEL RESULTS
# ============================================================

if all_accuracy_tables:

    combined_accuracy_df = pd.concat(
        all_accuracy_tables,
        ignore_index=True
    )

    combined_accuracy_df.to_csv(
        COMBINED_ACCURACY_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("=" * 100)
    print("COMBINED PERIOD ACCURACY BY NUMBER OF CONFIDENT CROPS")
    print("=" * 100)

    try:
        display(combined_accuracy_df)

    except NameError:
        print(
            combined_accuracy_df.to_string(
                index=False
            )
        )

    print()
    print("Saved combined result:")
    print(COMBINED_ACCURACY_CSV)

else:

    print()
    print("No model results were created.")

In [ ]:
# ============================================================
# CONVNEXT BASE SIGN CLASSIFICATION + TABLET-LEVEL PERIOD VOTING
#
# INPUT:
#   cuneiform-ocr-main
#       \mongo_unannotated_tablet_outputs
#           \csv_parts
#               detr_crops_part_*.csv
#           \detr_crops
#
# OUTPUT:
#   mongo_unannotated_tablet_outputs/
#       convnext_base_predictions/
#           prediction_parts/
#           failed_prediction_crops/
#           mongo_unannotated_crop_predictions.csv
#           mongo_unannotated_tablet_period_votes.csv
#           mongo_unannotated_period_vote_details.csv
#           convnext_base_period_vote_summary.csv
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import gc
import time
import textwrap
import importlib
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torchvision import models


# ============================================================
# PATH SETTINGS
# ============================================================

OCR_ROOT = Path(
    r"cuneiform-ocr-main"
)

OUTPUT_DIR = (
    OCR_ROOT
    / "mongo_unannotated_tablet_outputs"
)

CSV_PARTS_DIR = (
    OUTPUT_DIR
    / "csv_parts"
)

BASE_ROOT = Path(
    r"ebl_tablets_and_sign_crops"
)

EXTENSION_ROOT = (
    BASE_ROOT
    / "sign_classification_extension"
)

CONVNEXT_ROOT = (
    EXTENSION_ROOT
    / "ConvNeXt"
)

RESNET_ROOT = (
    EXTENSION_ROOT
    / "ResNet"
)

TABLET_HOLDOUT_TRAIN_CSV = (
    RESNET_ROOT
    / "tablet_holdout_train.csv"
)

TABLET_HOLDOUT_VAL_CSV = (
    RESNET_ROOT
    / "tablet_holdout_val.csv"
)

TABLET_HOLDOUT_TEST_CSV = (
    RESNET_ROOT
    / "tablet_holdout_test.csv"
)

IMAGE_SPLIT_LABEL_MAPPING_CSV = (
    RESNET_ROOT
    / "combined_train_val_test_label_mapping_482.csv"
)

MODEL_OUTPUT_ROOT = (
    OUTPUT_DIR
    / "convnext_base_predictions"
)

MODEL_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

PREDICTION_PARTS_DIR = (
    MODEL_OUTPUT_ROOT
    / "prediction_parts"
)

FAILED_CROPS_DIR = (
    MODEL_OUTPUT_ROOT
    / "failed_prediction_crops"
)

PREDICTION_PARTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FAILED_CROPS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CROP_PREDICTIONS_CSV = (
    MODEL_OUTPUT_ROOT
    / "mongo_unannotated_crop_predictions.csv"
)

TABLET_PERIOD_VOTES_CSV = (
    MODEL_OUTPUT_ROOT
    / "mongo_unannotated_tablet_period_votes.csv"
)

PERIOD_DETAILS_CSV = (
    MODEL_OUTPUT_ROOT
    / "mongo_unannotated_period_vote_details.csv"
)

SUMMARY_CSV = (
    MODEL_OUTPUT_ROOT
    / "convnext_base_period_vote_summary.csv"
)


# ============================================================
# CHECKPOINT CANDIDATES
# ============================================================

CHECKPOINT_CANDIDATES = [

    CONVNEXT_ROOT / "best_convnext_base_tablet_holdout_sign_classifier.pth",
    CONVNEXT_ROOT / "best_convnext_base_tablet_holdout_classifier.pth",
    CONVNEXT_ROOT / "best_convnext_base_holdout_sign_classifier.pth",
    CONVNEXT_ROOT / "best_convnext_base_holdout.pth",

    CONVNEXT_ROOT / "best_convnext_base_sign_classifier.pth",
    CONVNEXT_ROOT / "best_convnext_base_classifier.pth",
    CONVNEXT_ROOT / "best_convnext_base.pth",
    CONVNEXT_ROOT / "convnext_base_best.pth",
    CONVNEXT_ROOT / "convnext_base_sign_classifier.pth",

    EXTENSION_ROOT / "best_convnext_base_tablet_holdout_sign_classifier.pth",
    EXTENSION_ROOT / "best_convnext_base_tablet_holdout_classifier.pth",
    EXTENSION_ROOT / "best_convnext_base_sign_classifier.pth",
    EXTENSION_ROOT / "best_convnext_base_classifier.pth",
    EXTENSION_ROOT / "best_convnext_base.pth",
]


# ============================================================
# SETTINGS
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 32

NUM_WORKERS = 8

PREFETCH_FACTOR = 4

WRITE_BUFFER_SIZE = 10_000

COMBINE_CHUNK_SIZE = 50_000

SIGN_CONFIDENCE_THRESHOLD = 0.50

OVERWRITE_EXISTING_PARTS = False

FALLBACK_TO_SINGLE_WORKER = True

RUN_NAME = "convnext_base"

DISPLAY_NAME = "ConvNeXt Base"


# ============================================================
# DEVICE INFO
# ============================================================

print("=" * 90)
print("DEVICE AND SETTINGS")
print("=" * 90)
print("Device              :", DEVICE)
print("Batch size          :", BATCH_SIZE)
print("Workers             :", NUM_WORKERS)
print("Confidence threshold:", SIGN_CONFIDENCE_THRESHOLD)
print("Output dir          :", MODEL_OUTPUT_ROOT)
print("Extension root      :", EXTENSION_ROOT)
print("ConvNeXt root       :", CONVNEXT_ROOT)
print("ResNet root for CSV :", RESNET_ROOT)

if DEVICE.type == "cuda":
    print("GPU                 :", torch.cuda.get_device_name(0))
    print("CUDA                :", torch.version.cuda)

print("=" * 90)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


# ============================================================
# PATH CHECKS
# ============================================================

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"Missing output dir:\n{OUTPUT_DIR}"
    )

if not CSV_PARTS_DIR.exists():
    raise FileNotFoundError(
        f"Missing csv_parts dir:\n{CSV_PARTS_DIR}"
    )

if not EXTENSION_ROOT.exists():
    raise FileNotFoundError(
        f"Missing extension root:\n{EXTENSION_ROOT}"
    )

if not RESNET_ROOT.exists():
    raise FileNotFoundError(
        f"Missing ResNet folder containing tablet_holdout_train.csv:\n{RESNET_ROOT}"
    )

if not TABLET_HOLDOUT_TRAIN_CSV.exists():
    raise FileNotFoundError(
        f"Missing label CSV:\n{TABLET_HOLDOUT_TRAIN_CSV}"
    )


# ============================================================
# FIND DETR CSV PARTS
# ============================================================

csv_part_files = sorted(
    CSV_PARTS_DIR.glob(
        "detr_crops_part_*.csv"
    )
)

if not csv_part_files:

    csv_part_files = sorted(
        OUTPUT_DIR.glob(
            "detr_crops_part_*.csv"
        )
    )

if not csv_part_files:

    raise FileNotFoundError(
        "No detr_crops_part_*.csv files found."
    )

print()
print("=" * 90)
print("DETR CROP CSV FILES")
print("=" * 90)
print("Number of parts:", len(csv_part_files))

for file in csv_part_files:
    print(" -", file.name)


# ============================================================
# CREATE WORKER MODULE
# ============================================================

WORKER_MODULE_NAME = "_cuneiform_crop_loader_convnext_base_torchvision"

WORKER_MODULE_PATH = (
    OUTPUT_DIR
    / f"{WORKER_MODULE_NAME}.py"
)

worker_module_code = r'''
from pathlib import Path

import torch

from PIL import Image, ImageFile
from torchvision import transforms
from torch.utils.data import Dataset


Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True


class CropDataset(Dataset):

    def __init__(self, crop_paths, output_dir):

        self.crop_paths = [str(path) for path in crop_paths]

        self.output_dir = Path(output_dir)

        self.detr_crops_dir = self.output_dir / "detr_crops"

        self.transform = transforms.Compose([
            transforms.Resize(232),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])


    def __len__(self):

        return len(self.crop_paths)


    def resolve_crop_path(self, crop_path_value):

        crop_path_text = str(crop_path_value).strip()

        if not crop_path_text:
            return None

        crop_path = Path(crop_path_text)

        candidates = [
            crop_path,
            self.output_dir / crop_path,
            self.detr_crops_dir / crop_path,
            self.detr_crops_dir / crop_path.name
        ]

        for candidate in candidates:

            try:

                if candidate.exists():
                    return candidate

            except OSError:
                continue

        return None


    def __getitem__(self, index):

        original_crop_path = self.crop_paths[index]

        resolved_crop_path = self.resolve_crop_path(original_crop_path)

        if resolved_crop_path is None:

            return {
                "index": index,
                "image": None,
                "error": "Crop image file not found",
                "resolved_path": ""
            }

        try:

            with Image.open(resolved_crop_path) as image:

                image = image.convert("RGB")

                image_tensor = self.transform(image)

            return {
                "index": index,
                "image": image_tensor,
                "error": "",
                "resolved_path": str(resolved_crop_path)
            }

        except Exception as error:

            return {
                "index": index,
                "image": None,
                "error": str(error),
                "resolved_path": str(resolved_crop_path)
            }


def crop_collate_fn(batch):

    valid_images = []
    valid_indices = []
    valid_paths = []
    failed_items = []

    for item in batch:

        if item["image"] is None:

            failed_items.append({
                "index": int(item["index"]),
                "error": item["error"],
                "resolved_path": item["resolved_path"]
            })

        else:

            valid_images.append(item["image"])
            valid_indices.append(int(item["index"]))
            valid_paths.append(item["resolved_path"])

    if valid_images:
        image_batch = torch.stack(valid_images, dim=0)
    else:
        image_batch = None

    return {
        "images": image_batch,
        "indices": valid_indices,
        "resolved_paths": valid_paths,
        "failed": failed_items,
        "batch_size": len(batch)
    }


def dataloader_worker_init(worker_id):

    torch.set_num_threads(1)
'''

WORKER_MODULE_PATH.write_text(
    textwrap.dedent(worker_module_code),
    encoding="utf-8"
)

if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))

importlib.invalidate_caches()

if WORKER_MODULE_NAME in sys.modules:
    del sys.modules[WORKER_MODULE_NAME]

worker_module = importlib.import_module(
    WORKER_MODULE_NAME
)

CropDataset = worker_module.CropDataset
crop_collate_fn = worker_module.crop_collate_fn
dataloader_worker_init = worker_module.dataloader_worker_init

print()
print("Worker module created:")
print(WORKER_MODULE_PATH)


# ============================================================
# GENERAL HELPERS
# ============================================================

def first_existing_path(candidates):

    for path in candidates:

        path = Path(path)

        if path.exists():
            return path

    return None


def auto_find_convnext_checkpoint():

    all_pths = sorted(
        EXTENSION_ROOT.rglob("*.pth")
    )

    candidates = []

    for path in all_pths:

        name = path.name.lower()

        if "convnext" not in name:
            continue

        # Prefer base, but still allow if filename does not say base.
        if (
            "tiny" in name
            or "small" in name
            or "large" in name
            or "xlarge" in name
        ):
            continue

        candidates.append(path)

    if not candidates:
        return None

    def checkpoint_score(path):

        name = path.name.lower()

        score = 0

        if "base" in name:
            score += 30

        if "best" in name:
            score += 20

        if "tablet" in name:
            score += 10

        if "holdout" in name:
            score += 10

        if "sign" in name:
            score += 5

        if "classifier" in name:
            score += 5

        return -score, name

    candidates = sorted(
        candidates,
        key=checkpoint_score
    )

    return candidates[0]


def find_convnext_checkpoint():

    checkpoint_path = first_existing_path(
        CHECKPOINT_CANDIDATES
    )

    if checkpoint_path is not None:
        return checkpoint_path

    checkpoint_path = auto_find_convnext_checkpoint()

    if checkpoint_path is None:

        all_pths = sorted(
            EXTENSION_ROOT.rglob("*.pth")
        )

        pth_text = "\n".join(
            str(path)
            for path in all_pths
        )

        raise FileNotFoundError(
            "Could not find ConvNeXt Base checkpoint.\n\n"
            f"Searched under:\n{EXTENSION_ROOT}\n\n"
            f"Available .pth files:\n{pth_text}"
        )

    return checkpoint_path


def clean_checkpoint_state_dict(checkpoint):

    if isinstance(checkpoint, dict):

        if "model_state_dict" in checkpoint:
            checkpoint = checkpoint["model_state_dict"]

        elif "state_dict" in checkpoint:
            checkpoint = checkpoint["state_dict"]

        elif "model" in checkpoint:
            checkpoint = checkpoint["model"]

    if not isinstance(checkpoint, dict):
        raise ValueError("Checkpoint is not a valid state_dict.")

    prefixes_to_strip = [
        "module.",
        "_orig_mod.",
        "model."
    ]

    cleaned = {}

    for key, value in checkpoint.items():

        new_key = key

        changed = True

        while changed:

            changed = False

            for prefix in prefixes_to_strip:

                if new_key.startswith(prefix):

                    new_key = new_key[len(prefix):]

                    changed = True

        cleaned[new_key] = value

    return cleaned


def load_checkpoint_state_dict(checkpoint_path):

    try:

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

    except TypeError:

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu"
        )

    return clean_checkpoint_state_dict(
        checkpoint
    )


# ============================================================
# LABEL MAPPING HELPERS
# ============================================================

def create_combined_label_mapping_csv():

    required_columns = {
        "label",
        "signName",
        "period"
    }

    csv_files = [
        TABLET_HOLDOUT_TRAIN_CSV,
        TABLET_HOLDOUT_VAL_CSV,
        TABLET_HOLDOUT_TEST_CSV,
    ]

    dfs = []

    for csv_file in csv_files:

        if not csv_file.exists():
            continue

        df = pd.read_csv(
            csv_file,
            dtype=str,
            keep_default_na=False
        )

        missing = required_columns - set(df.columns)

        if missing:

            raise ValueError(
                f"Missing columns in:\n{csv_file}\n\n"
                f"Missing:\n{sorted(missing)}\n\n"
                f"Available columns:\n{list(df.columns)}"
            )

        dfs.append(df)

    if not dfs:

        raise RuntimeError(
            "Could not create combined label mapping because no CSV files were found."
        )

    combined_df = pd.concat(
        dfs,
        ignore_index=True
    )

    combined_df["label"] = (
        combined_df["label"]
        .astype(str)
        .str.strip()
    )

    combined_df = (
        combined_df[
            combined_df["label"] != ""
        ]
        .copy()
    )

    combined_label_df = (
        combined_df[
            [
                "label",
                "signName",
                "period"
            ]
        ]
        .drop_duplicates(
            subset=["label"]
        )
        .sort_values(
            by="label"
        )
        .reset_index(drop=True)
    )

    combined_label_df.to_csv(
        IMAGE_SPLIT_LABEL_MAPPING_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("=" * 90)
    print("COMBINED LABEL MAPPING CREATED")
    print("=" * 90)
    print("Saved:", IMAGE_SPLIT_LABEL_MAPPING_CSV)
    print("Number of labels:", combined_label_df["label"].nunique())
    print("=" * 90)

    return IMAGE_SPLIT_LABEL_MAPPING_CSV


def load_label_mapping(label_csv):

    df = pd.read_csv(
        label_csv,
        dtype=str,
        keep_default_na=False
    )

    required_columns = {
        "label",
        "signName",
        "period"
    }

    missing = required_columns - set(df.columns)

    if missing:

        raise ValueError(
            f"Missing columns in label CSV:\n{sorted(missing)}\n\n"
            f"File:\n{label_csv}\n\n"
            f"Available columns:\n{list(df.columns)}"
        )

    df["label"] = (
        df["label"]
        .astype(str)
        .str.strip()
    )

    df = (
        df[
            df["label"] != ""
        ]
        .copy()
    )

    labels = sorted(
        df["label"].unique()
    )

    label_to_idx = {
        label: idx
        for idx, label in enumerate(labels)
    }

    idx_to_label = {
        idx: label
        for label, idx in label_to_idx.items()
    }

    label_info = (
        df[
            [
                "label",
                "signName",
                "period"
            ]
        ]
        .drop_duplicates(subset=["label"])
        .set_index("label")
        .to_dict("index")
    )

    return {
        "df": df,
        "labels": labels,
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "label_info": label_info,
        "num_classes": len(labels)
    }


def choose_label_csv_for_checkpoint(checkpoint_num_classes):

    if checkpoint_num_classes == 471:

        return TABLET_HOLDOUT_TRAIN_CSV

    if checkpoint_num_classes == 482:

        label_csv = create_combined_label_mapping_csv()

        label_bundle = load_label_mapping(
            label_csv
        )

        if label_bundle["num_classes"] != 482:

            raise RuntimeError(
                "ConvNeXt checkpoint has 482 classes, but the available CSV files "
                "do not provide a 482-class label mapping.\n\n"
                f"Combined label CSV:\n{label_csv}\n"
                f"Classes found: {label_bundle['num_classes']}\n\n"
                "You need the original 482-class image-split training CSV used "
                "when training the ConvNeXt Base checkpoint."
            )

        return label_csv

    raise RuntimeError(
        f"Unsupported checkpoint class count: {checkpoint_num_classes}\n\n"
        "Expected either 471 for tablet-holdout or 482 for image-split."
    )


# ============================================================
# TORCHVISION CONVNEXT BASE HELPERS
# ============================================================

def get_convnext_num_classes(state_dict):
    """
    Infer output classes from torchvision ConvNeXt classifier.

    Your checkpoint uses:
        classifier.2.weight
    """

    possible_keys = [
        "classifier.2.weight",
        "classifier.weight",
        "head.fc.weight",
        "fc.weight"
    ]

    for key in possible_keys:

        if key in state_dict:

            weight = state_dict[key]

            if hasattr(weight, "shape") and len(weight.shape) == 2:

                return int(weight.shape[0]), key

    raise ValueError(
        "Could not infer ConvNeXt output classes.\n"
        "Expected classifier.2.weight for torchvision ConvNeXt."
    )


def build_convnext_base_model(num_classes):
    """
    Build torchvision ConvNeXt Base.

    This matches checkpoint keys:
        features.*
        classifier.2.weight
    """

    model = models.convnext_base(
        weights=None
    )

    in_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        in_features,
        num_classes
    )

    return model


def load_convnext_model(checkpoint_path, num_classes):

    state_dict = load_checkpoint_state_dict(
        checkpoint_path
    )

    checkpoint_num_classes, classifier_key = get_convnext_num_classes(
        state_dict
    )

    print("Classes from label CSV :", num_classes)
    print("Classes from checkpoint:", checkpoint_num_classes)
    print("Classifier key         :", classifier_key)

    if checkpoint_num_classes != num_classes:

        raise RuntimeError(
            "Class-count mismatch.\n\n"
            f"Checkpoint: {checkpoint_path}\n"
            f"Label CSV classes: {num_classes}\n"
            f"Checkpoint classes: {checkpoint_num_classes}\n\n"
            "The selected label mapping does not match this ConvNeXt checkpoint."
        )

    model = build_convnext_base_model(
        num_classes=num_classes
    )

    model.load_state_dict(
        state_dict,
        strict=True
    )

    model = model.to(
        DEVICE
    )

    if DEVICE.type == "cuda":

        model = model.to(
            memory_format=torch.channels_last
        )

    model.eval()

    return model, classifier_key, checkpoint_num_classes


def warmup_model(model):

    if DEVICE.type != "cuda":
        return

    print()
    print("Warming up GPU...")

    warmup_batch = torch.zeros(
        (BATCH_SIZE, 3, 224, 224),
        device=DEVICE
    )

    warmup_batch = warmup_batch.contiguous(
        memory_format=torch.channels_last
    )

    with torch.inference_mode():

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            _ = model(
                warmup_batch
            )

    torch.cuda.synchronize()

    del warmup_batch

    torch.cuda.empty_cache()

    print("GPU warm-up completed.")


# ============================================================
# DATALOADER + CSV HELPERS
# ============================================================

def create_dataloader(dataset, num_workers):

    loader_args = {
        "dataset": dataset,
        "batch_size": BATCH_SIZE,
        "shuffle": False,
        "num_workers": num_workers,
        "pin_memory": DEVICE.type == "cuda",
        "drop_last": False,
        "collate_fn": crop_collate_fn
    }

    if num_workers > 0:

        loader_args.update({
            "prefetch_factor": PREFETCH_FACTOR,
            "persistent_workers": False,
            "worker_init_fn": dataloader_worker_init
        })

    return DataLoader(
        **loader_args
    )


def append_rows_to_csv(rows, output_path, header_written):

    if not rows:
        return header_written

    pd.DataFrame(rows).to_csv(
        output_path,
        mode="a",
        index=False,
        header=not header_written
    )

    return True


# ============================================================
# CLASSIFICATION
# ============================================================

def classify_tensor_batch(
    image_batch,
    model,
    idx_to_label,
    label_info
):

    if image_batch is None:
        return []

    image_batch = image_batch.to(
        DEVICE,
        non_blocking=True
    )

    if DEVICE.type == "cuda":

        image_batch = image_batch.contiguous(
            memory_format=torch.channels_last
        )

    with torch.inference_mode():

        if DEVICE.type == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(
                    image_batch
                )

        else:

            logits = model(
                image_batch
            )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        top_probabilities, top_indices = probabilities.max(
            dim=1
        )

    top_probabilities = (
        top_probabilities
        .detach()
        .float()
        .cpu()
        .tolist()
    )

    top_indices = (
        top_indices
        .detach()
        .cpu()
        .tolist()
    )

    results = []

    for pred_idx, pred_conf in zip(
        top_indices,
        top_probabilities
    ):

        pred_idx = int(
            pred_idx
        )

        pred_label = idx_to_label[
            pred_idx
        ]

        metadata = label_info.get(
            pred_label,
            {}
        )

        results.append({
            "pred_label": pred_label,
            "pred_signName": metadata.get("signName", ""),
            "pred_period": metadata.get("period", ""),
            "pred_confidence": float(pred_conf)
        })

    return results


# ============================================================
# PROCESS ONE DETR CSV PART
# ============================================================

def process_prediction_part(
    input_csv,
    num_workers,
    model,
    idx_to_label,
    label_info
):

    part_suffix = input_csv.stem.replace(
        "detr_crops_part_",
        ""
    )

    output_part_csv = (
        PREDICTION_PARTS_DIR
        / f"crop_predictions_part_{part_suffix}.csv"
    )

    temporary_output_csv = Path(
        str(output_part_csv)
        + ".tmp"
    )

    failed_part_csv = (
        FAILED_CROPS_DIR
        / f"failed_crops_part_{part_suffix}.csv"
    )

    if output_part_csv.exists() and not OVERWRITE_EXISTING_PARTS:

        print("Prediction part already exists. Skipping:")
        print(output_part_csv)

        return {
            "status": "skipped",
            "output_path": output_part_csv,
            "input_rows": 0,
            "predictions": 0,
            "failures": 0,
            "elapsed_seconds": 0.0,
            "gpu_batches": 0
        }

    if output_part_csv.exists():
        output_part_csv.unlink()

    if temporary_output_csv.exists():
        temporary_output_csv.unlink()

    if failed_part_csv.exists():
        failed_part_csv.unlink()

    print()
    print("Reading:")
    print(input_csv)

    part_df = pd.read_csv(
        input_csv,
        dtype=str,
        keep_default_na=False
    )

    if "cropPath" not in part_df.columns:

        raise ValueError(
            f"'cropPath' column missing in:\n{input_csv}"
        )

    number_of_rows = len(part_df)

    source_records = part_df.to_dict(
        orient="records"
    )

    crop_paths = (
        part_df["cropPath"]
        .astype(str)
        .tolist()
    )

    del part_df

    gc.collect()

    dataset = CropDataset(
        crop_paths=crop_paths,
        output_dir=str(OUTPUT_DIR)
    )

    loader = create_dataloader(
        dataset,
        num_workers
    )

    prediction_buffer = []
    failure_buffer = []

    prediction_header_written = False
    failure_header_written = False

    total_predictions = 0
    total_failures = 0
    total_gpu_batches = 0

    start_time = time.time()

    progress_bar = tqdm(
        total=number_of_rows,
        desc=input_csv.name,
        unit="crop"
    )

    try:

        for loader_batch in loader:

            current_batch_size = int(
                loader_batch["batch_size"]
            )

            for failed_item in loader_batch["failed"]:

                failed_index = int(
                    failed_item["index"]
                )

                failed_row = dict(
                    source_records[failed_index]
                )

                failed_row.update({
                    "failure_reason": failed_item["error"],
                    "resolved_crop_path": failed_item["resolved_path"]
                })

                failure_buffer.append(
                    failed_row
                )

                total_failures += 1

            image_batch = loader_batch["images"]

            valid_indices = loader_batch["indices"]

            resolved_paths = loader_batch["resolved_paths"]

            if image_batch is not None:

                batch_predictions = classify_tensor_batch(
                    image_batch=image_batch,
                    model=model,
                    idx_to_label=idx_to_label,
                    label_info=label_info
                )

                total_gpu_batches += 1

                for source_index, resolved_path, prediction in zip(
                    valid_indices,
                    resolved_paths,
                    batch_predictions
                ):

                    prediction_row = dict(
                        source_records[int(source_index)]
                    )

                    prediction_row.update(
                        prediction
                    )

                    prediction_row["resolved_crop_path"] = resolved_path

                    prediction_buffer.append(
                        prediction_row
                    )

                    total_predictions += 1

            if len(prediction_buffer) >= WRITE_BUFFER_SIZE:

                prediction_header_written = append_rows_to_csv(
                    prediction_buffer,
                    temporary_output_csv,
                    prediction_header_written
                )

                prediction_buffer = []

            if len(failure_buffer) >= WRITE_BUFFER_SIZE:

                failure_header_written = append_rows_to_csv(
                    failure_buffer,
                    failed_part_csv,
                    failure_header_written
                )

                failure_buffer = []

            progress_bar.update(
                current_batch_size
            )

            elapsed = time.time() - start_time

            crops_per_second = (
                progress_bar.n / elapsed
                if elapsed > 0
                else 0.0
            )

            progress_bar.set_postfix({
                "GPU_batches": total_gpu_batches,
                "predicted": total_predictions,
                "failed": total_failures,
                "crops/s": f"{crops_per_second:.1f}"
            })

    finally:

        progress_bar.close()

    prediction_header_written = append_rows_to_csv(
        prediction_buffer,
        temporary_output_csv,
        prediction_header_written
    )

    failure_header_written = append_rows_to_csv(
        failure_buffer,
        failed_part_csv,
        failure_header_written
    )

    if not temporary_output_csv.exists():

        raise RuntimeError(
            f"No predictions produced for:\n{input_csv}"
        )

    os.replace(
        temporary_output_csv,
        output_part_csv
    )

    elapsed = time.time() - start_time

    del loader
    del dataset
    del source_records
    del crop_paths

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "status": "completed",
        "output_path": output_part_csv,
        "input_rows": number_of_rows,
        "predictions": total_predictions,
        "failures": total_failures,
        "elapsed_seconds": elapsed,
        "gpu_batches": total_gpu_batches
    }


# ============================================================
# COMBINE PREDICTION PARTS
# ============================================================

def combine_prediction_parts():

    prediction_part_files = sorted(
        PREDICTION_PARTS_DIR.glob(
            "crop_predictions_part_*.csv"
        )
    )

    if not prediction_part_files:

        raise RuntimeError(
            f"No prediction parts found in:\n{PREDICTION_PARTS_DIR}"
        )

    tmp_csv = Path(
        str(CROP_PREDICTIONS_CSV)
        + ".tmp"
    )

    if tmp_csv.exists():
        tmp_csv.unlink()

    header_written = False

    total_rows = 0

    for part_file in tqdm(
        prediction_part_files,
        desc="Combining prediction files",
        unit="file"
    ):

        reader = pd.read_csv(
            part_file,
            dtype=str,
            keep_default_na=False,
            chunksize=COMBINE_CHUNK_SIZE
        )

        for chunk in reader:

            chunk.to_csv(
                tmp_csv,
                mode="a",
                index=False,
                header=not header_written
            )

            header_written = True

            total_rows += len(chunk)

    os.replace(
        tmp_csv,
        CROP_PREDICTIONS_CSV
    )

    return prediction_part_files, total_rows


# ============================================================
# TABLET-LEVEL PERIOD VOTING
# ============================================================

def run_tablet_level_period_voting():

    all_predictions_df = pd.read_csv(
        CROP_PREDICTIONS_CSV,
        dtype=str,
        keep_default_na=False
    )

    required_cols = {
        "fragmentNumber",
        "period",
        "pred_period",
        "pred_confidence"
    }

    missing = required_cols - set(
        all_predictions_df.columns
    )

    if missing:

        raise ValueError(
            f"Missing columns in predictions:\n{sorted(missing)}"
        )

    all_predictions_df["pred_confidence"] = pd.to_numeric(
        all_predictions_df["pred_confidence"],
        errors="coerce"
    )

    all_predictions_df = (
        all_predictions_df
        .dropna(
            subset=[
                "fragmentNumber",
                "pred_period",
                "pred_confidence"
            ]
        )
        .copy()
    )

    all_predictions_df["fragmentNumber"] = (
        all_predictions_df["fragmentNumber"]
        .astype(str)
        .str.strip()
    )

    all_predictions_df["pred_period"] = (
        all_predictions_df["pred_period"]
        .astype(str)
        .str.strip()
    )

    all_predictions_df = (
        all_predictions_df[
            (all_predictions_df["fragmentNumber"] != "")
            &
            (all_predictions_df["pred_period"] != "")
        ]
        .copy()
    )

    fragment_counts_before_threshold = (
        all_predictions_df
        .groupby("fragmentNumber")
        .size()
        .to_dict()
    )

    total_fragments_before_threshold = (
        all_predictions_df["fragmentNumber"]
        .nunique()
    )

    predictions_before_threshold = len(
        all_predictions_df
    )

    pred_df = (
        all_predictions_df[
            all_predictions_df["pred_confidence"]
            >= SIGN_CONFIDENCE_THRESHOLD
        ]
        .copy()
    )

    predictions_after_threshold = len(
        pred_df
    )

    evaluated_fragment_count = (
        pred_df["fragmentNumber"]
        .nunique()
    )

    fragments_without_confident_crops = (
        total_fragments_before_threshold
        - evaluated_fragment_count
    )

    print()
    print("=" * 90)
    print("TABLET-LEVEL VOTING INPUT")
    print("=" * 90)
    print("Predictions before filtering:", f"{predictions_before_threshold:,}")
    print("Predictions after filtering :", f"{predictions_after_threshold:,}")
    print("Fragments before filtering  :", f"{total_fragments_before_threshold:,}")
    print("Fragments evaluated         :", f"{evaluated_fragment_count:,}")
    print("Fragments without confident :", f"{fragments_without_confident_crops:,}")

    if len(pred_df) == 0:

        raise RuntimeError(
            "No predictions remain after threshold filtering."
        )

    tablet_rows = []
    detail_rows = []

    grouped = pred_df.groupby(
        "fragmentNumber",
        sort=False
    )

    for fragment_number, group in tqdm(
        grouped,
        total=evaluated_fragment_count,
        desc="Tablet-level period voting",
        unit="fragment"
    ):

        period_values = (
            group["period"]
            .astype(str)
            .str.strip()
        )

        period_values = period_values[
            period_values != ""
        ]

        if len(period_values) > 0:

            true_period = (
                period_values
                .mode()
                .iloc[0]
            )

        else:

            true_period = ""

        period_summary = (
            group
            .groupby("pred_period", dropna=False)
            .agg(
                num_votes=("pred_period", "count"),
                mean_confidence=("pred_confidence", "mean"),
                weighted_score=("pred_confidence", "sum")
            )
            .reset_index()
        )

        period_summary = (
            period_summary
            .sort_values(
                by=[
                    "weighted_score",
                    "num_votes",
                    "mean_confidence",
                    "pred_period"
                ],
                ascending=[
                    False,
                    False,
                    False,
                    True
                ]
            )
            .reset_index(drop=True)
        )

        total_weight = (
            period_summary["weighted_score"]
            .sum()
        )

        if total_weight > 0:

            period_summary["vote_probability"] = (
                period_summary["weighted_score"]
                / total_weight
            )

        else:

            period_summary["vote_probability"] = 0.0

        best_row = period_summary.iloc[0]

        predicted_period = str(
            best_row["pred_period"]
        )

        correct = (
            true_period
            == predicted_period
        )

        if len(period_summary) > 1:

            second_vote_probability = float(
                period_summary.iloc[1]["vote_probability"]
            )

        else:

            second_vote_probability = 0.0

        winning_vote_probability = float(
            best_row["vote_probability"]
        )

        vote_margin = (
            winning_vote_probability
            - second_vote_probability
        )

        tablet_rows.append({
            "fragmentNumber": fragment_number,
            "true_period": true_period,
            "predicted_period": predicted_period,
            "num_crops_before_threshold": int(
                fragment_counts_before_threshold.get(
                    fragment_number,
                    len(group)
                )
            ),
            "num_crops_used": int(len(group)),
            "winning_votes": int(best_row["num_votes"]),
            "winning_mean_confidence": float(best_row["mean_confidence"]),
            "winning_weighted_score": float(best_row["weighted_score"]),
            "winning_vote_probability": winning_vote_probability,
            "second_vote_probability": second_vote_probability,
            "vote_margin": float(vote_margin),
            "correct": bool(correct)
        })

        for rank, row in period_summary.iterrows():

            detail_rows.append({
                "fragmentNumber": fragment_number,
                "true_period": true_period,
                "rank": int(rank + 1),
                "candidate_period": str(row["pred_period"]),
                "num_votes": int(row["num_votes"]),
                "mean_confidence": float(row["mean_confidence"]),
                "weighted_score": float(row["weighted_score"]),
                "vote_probability": float(row["vote_probability"])
            })

    tablet_votes_df = pd.DataFrame(
        tablet_rows
    )

    period_details_df = pd.DataFrame(
        detail_rows
    )

    tablet_votes_df.to_csv(
        TABLET_PERIOD_VOTES_CSV,
        index=False
    )

    period_details_df.to_csv(
        PERIOD_DETAILS_CSV,
        index=False
    )

    if len(tablet_votes_df) > 0:

        tablet_votes_df["correct"] = (
            tablet_votes_df["correct"]
            .astype(bool)
        )

        accuracy = (
            tablet_votes_df["correct"]
            .mean()
        )

        number_correct = int(
            tablet_votes_df["correct"]
            .sum()
        )

        number_evaluated = len(
            tablet_votes_df
        )

        number_incorrect = (
            number_evaluated
            - number_correct
        )

    else:

        accuracy = 0.0
        number_correct = 0
        number_evaluated = 0
        number_incorrect = 0

    return {
        "predictions_before_threshold": int(predictions_before_threshold),
        "predictions_after_threshold": int(predictions_after_threshold),
        "predictions_removed": int(predictions_before_threshold - predictions_after_threshold),
        "fragments_before_threshold": int(total_fragments_before_threshold),
        "fragments_evaluated": int(number_evaluated),
        "fragments_without_confident_crops": int(fragments_without_confident_crops),
        "correct_period_predictions": int(number_correct),
        "incorrect_period_predictions": int(number_incorrect),
        "tablet_level_accuracy": float(accuracy),
        "tablet_votes_df": tablet_votes_df
    }


# ============================================================
# RUN CONVNEXT BASE
# ============================================================

print()
print("#" * 100)
print("STARTING CONVNEXT BASE RUN")
print("#" * 100)

checkpoint_path = find_convnext_checkpoint()

checkpoint_state_dict = load_checkpoint_state_dict(
    checkpoint_path
)

checkpoint_num_classes, classifier_key = get_convnext_num_classes(
    checkpoint_state_dict
)

label_csv = choose_label_csv_for_checkpoint(
    checkpoint_num_classes
)

label_bundle = load_label_mapping(
    label_csv
)

idx_to_label = label_bundle["idx_to_label"]

label_info = label_bundle["label_info"]

num_classes = label_bundle["num_classes"]

print()
print("=" * 90)
print("MODEL FILES")
print("=" * 90)
print("Model                  :", DISPLAY_NAME)
print("Checkpoint             :", checkpoint_path)
print("Classifier key         :", classifier_key)
print("Checkpoint classes     :", checkpoint_num_classes)
print("Label CSV              :", label_csv)
print("Number of label classes:", num_classes)
print("Output dir             :", MODEL_OUTPUT_ROOT)
print("=" * 90)

model, classifier_key, checkpoint_num_classes = load_convnext_model(
    checkpoint_path=checkpoint_path,
    num_classes=num_classes
)

warmup_model(
    model
)

for part_idx, input_csv in enumerate(
    csv_part_files,
    start=1
):

    print()
    print("=" * 90)
    print(f"PROCESSING PART {part_idx}/{len(csv_part_files)}")
    print("Model:", DISPLAY_NAME)
    print("Input:", input_csv)
    print("=" * 90)

    try:

        stats = process_prediction_part(
            input_csv=input_csv,
            num_workers=NUM_WORKERS,
            model=model,
            idx_to_label=idx_to_label,
            label_info=label_info
        )

    except Exception as error:

        if NUM_WORKERS > 0 and FALLBACK_TO_SINGLE_WORKER:

            print()
            print("Multi-worker failed. Retrying with num_workers=0.")
            print("Original error:", repr(error))

            stats = process_prediction_part(
                input_csv=input_csv,
                num_workers=0,
                model=model,
                idx_to_label=idx_to_label,
                label_info=label_info
            )

        else:

            raise

    print("Part status:", stats["status"])

    if stats["status"] == "completed":

        print("Predictions:", f'{stats["predictions"]:,}')
        print("Failures   :", f'{stats["failures"]:,}')
        print("Elapsed min:", f'{stats["elapsed_seconds"] / 60:.2f}')


print()
print("=" * 90)
print("COMBINING PREDICTION PARTS")
print("=" * 90)

prediction_part_files, combined_rows = combine_prediction_parts()

print("Combined rows:", f"{combined_rows:,}")

voting_stats = run_tablet_level_period_voting()

print()
print("=" * 90)
print("FINAL RESULTS:", DISPLAY_NAME)
print("=" * 90)
print("Total crop predictions           :", f"{combined_rows:,}")
print("Crop predictions used            :", f'{voting_stats["predictions_after_threshold"]:,}')
print("Fragments before threshold       :", f'{voting_stats["fragments_before_threshold"]:,}')
print("Fragments evaluated              :", f'{voting_stats["fragments_evaluated"]:,}')
print("Fragments without confident crops:", f'{voting_stats["fragments_without_confident_crops"]:,}')
print("Correct period predictions       :", f'{voting_stats["correct_period_predictions"]:,}')
print("Incorrect period predictions     :", f'{voting_stats["incorrect_period_predictions"]:,}')
print("Tablet-level accuracy            :", f'{voting_stats["tablet_level_accuracy"]:.4f}')
print("Tablet-level accuracy (%)        :", f'{voting_stats["tablet_level_accuracy"] * 100:.2f}%')

summary_df = pd.DataFrame([
    {
        "run_name": RUN_NAME,
        "display_name": DISPLAY_NAME,
        "architecture": "torchvision_convnext_base",
        "label_csv": str(label_csv),
        "checkpoint": str(checkpoint_path),
        "classifier_key": classifier_key,
        "checkpoint_num_classes": int(checkpoint_num_classes),
        "label_num_classes": int(num_classes),
        "output_dir": str(MODEL_OUTPUT_ROOT),
        "detr_csv_parts": int(len(csv_part_files)),
        "prediction_csv_parts": int(len(prediction_part_files)),
        "combined_prediction_rows": int(combined_rows),
        "predictions_before_threshold": voting_stats["predictions_before_threshold"],
        "predictions_after_threshold": voting_stats["predictions_after_threshold"],
        "predictions_removed": voting_stats["predictions_removed"],
        "fragments_before_threshold": voting_stats["fragments_before_threshold"],
        "fragments_evaluated": voting_stats["fragments_evaluated"],
        "fragments_without_confident_crops": voting_stats["fragments_without_confident_crops"],
        "correct_period_predictions": voting_stats["correct_period_predictions"],
        "incorrect_period_predictions": voting_stats["incorrect_period_predictions"],
        "tablet_level_accuracy": voting_stats["tablet_level_accuracy"],
        "tablet_level_accuracy_percent": voting_stats["tablet_level_accuracy"] * 100.0,
        "confidence_threshold": float(SIGN_CONFIDENCE_THRESHOLD),
    }
])

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

print()
print("Saved crop predictions:")
print(CROP_PREDICTIONS_CSV)

print()
print("Saved tablet-period votes:")
print(TABLET_PERIOD_VOTES_CSV)

print()
print("Saved period details:")
print(PERIOD_DETAILS_CSV)

print()
print("Saved summary:")
print(SUMMARY_CSV)

try:

    display(summary_df)

except NameError:

    print(
        summary_df.to_string(
            index=False
        )
    )


del model

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# PERIOD ACCURACY BY NUMBER OF CONFIDENT CROPS
# FOR CONVNEXT BASE
#
# GROUPS:
#   1 confident crop
#   2-4 confident crops
#   5-9 confident crops
#   10 or more confident crops
#
# INPUT:
#   mongo_unannotated_tablet_outputs/
#       convnext_base_predictions/
#           mongo_unannotated_tablet_period_votes.csv
#
# OUTPUT:
#   mongo_unannotated_tablet_outputs/
#       convnext_base_predictions/
#           period_accuracy_by_crop_count.csv
# ============================================================

from pathlib import Path

import pandas as pd


# ============================================================
# PATHS
# ============================================================

OUTPUT_DIR = Path(
    r"cuneiform-ocr-main"
    r"\mongo_unannotated_tablet_outputs"
)

MODEL_OUTPUT_ROOT = (
    OUTPUT_DIR
    / "convnext_base_predictions"
)

TABLET_PERIOD_VOTES_CSV = (
    MODEL_OUTPUT_ROOT
    / "mongo_unannotated_tablet_period_votes.csv"
)

ACCURACY_BY_CROP_COUNT_CSV = (
    MODEL_OUTPUT_ROOT
    / "period_accuracy_by_crop_count.csv"
)


# ============================================================
# MODEL INFORMATION
# ============================================================

RUN_NAME = "convnext_base"

DISPLAY_NAME = "ConvNeXt Base"


# ============================================================
# CHECK INPUT FILE
# ============================================================

if not TABLET_PERIOD_VOTES_CSV.exists():

    raise FileNotFoundError(
        "Tablet-level period-voting file was not found.\n\n"
        f"Expected file:\n{TABLET_PERIOD_VOTES_CSV}\n\n"
        "Run the ConvNeXt Base prediction/voting script first."
    )


# ============================================================
# CROP-COUNT GROUP FUNCTION
# ============================================================

def assign_crop_count_group(num_crops):
    """
    Assign each fragment to one of four evidence groups.
    """

    if num_crops == 1:
        return "1 confident crop"

    if 2 <= num_crops <= 4:
        return "2â€“4 confident crops"

    if 5 <= num_crops <= 9:
        return "5â€“9 confident crops"

    if num_crops >= 10:
        return "10 or more confident crops"

    return "No confident crops"


GROUP_ORDER = [
    "1 confident crop",
    "2â€“4 confident crops",
    "5â€“9 confident crops",
    "10 or more confident crops"
]


# ============================================================
# BOOLEAN CLEANING
# ============================================================

def clean_correct_column(series):
    """
    Convert correct column safely to boolean.

    Handles:
        True / False
        true / false
        1 / 0
        yes / no
    """

    if series.dtype == bool:
        return series

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False
        })
    )


# ============================================================
# LOAD TABLET-LEVEL RESULTS
# ============================================================

tablet_votes_df = pd.read_csv(
    TABLET_PERIOD_VOTES_CSV,
    dtype={
        "fragmentNumber": str,
        "true_period": str,
        "predicted_period": str
    },
    keep_default_na=False
)

print("=" * 100)
print("CONVNEXT BASE PERIOD ACCURACY BY NUMBER OF CONFIDENT CROPS")
print("=" * 100)

print("Model:", DISPLAY_NAME)
print("Input:", TABLET_PERIOD_VOTES_CSV)
print(
    "Loaded tablet-level predictions:",
    f"{len(tablet_votes_df):,}"
)


# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

required_columns = {
    "fragmentNumber",
    "true_period",
    "predicted_period",
    "num_crops_used",
    "correct"
}

missing_columns = (
    required_columns
    - set(tablet_votes_df.columns)
)

if missing_columns:

    raise ValueError(
        f"Missing columns in:\n{TABLET_PERIOD_VOTES_CSV}\n\n"
        f"Missing columns:\n{sorted(missing_columns)}\n\n"
        f"Available columns:\n{list(tablet_votes_df.columns)}"
    )


# ============================================================
# CLEAN REQUIRED COLUMNS
# ============================================================

tablet_votes_df["num_crops_used"] = pd.to_numeric(
    tablet_votes_df["num_crops_used"],
    errors="coerce"
)

tablet_votes_df["correct"] = clean_correct_column(
    tablet_votes_df["correct"]
)

tablet_votes_df = tablet_votes_df.dropna(
    subset=[
        "num_crops_used",
        "correct"
    ]
).copy()

tablet_votes_df["num_crops_used"] = (
    tablet_votes_df["num_crops_used"]
    .astype(int)
)

tablet_votes_df["correct"] = (
    tablet_votes_df["correct"]
    .astype(bool)
)

if len(tablet_votes_df) == 0:

    raise RuntimeError(
        f"No valid rows remain after cleaning:\n{TABLET_PERIOD_VOTES_CSV}"
    )


# ============================================================
# ASSIGN CROP-COUNT GROUP
# ============================================================

tablet_votes_df["crop_count_group"] = (
    tablet_votes_df["num_crops_used"]
    .apply(assign_crop_count_group)
)

tablet_votes_df["crop_count_group"] = pd.Categorical(
    tablet_votes_df["crop_count_group"],
    categories=GROUP_ORDER,
    ordered=True
)

# Keep only the four main groups.
tablet_votes_df = (
    tablet_votes_df[
        tablet_votes_df["crop_count_group"].notna()
    ]
    .copy()
)


# ============================================================
# CALCULATE ACCURACY FOR EACH GROUP
# ============================================================

accuracy_by_crop_count = (
    tablet_votes_df
    .groupby(
        "crop_count_group",
        observed=False
    )
    .agg(
        num_fragments=(
            "fragmentNumber",
            "count"
        ),
        correct_predictions=(
            "correct",
            "sum"
        ),
        mean_crops_used=(
            "num_crops_used",
            "mean"
        ),
        median_crops_used=(
            "num_crops_used",
            "median"
        )
    )
    .reset_index()
)

accuracy_by_crop_count["incorrect_predictions"] = (
    accuracy_by_crop_count["num_fragments"]
    - accuracy_by_crop_count["correct_predictions"]
)

accuracy_by_crop_count["accuracy"] = (
    accuracy_by_crop_count["correct_predictions"]
    / accuracy_by_crop_count["num_fragments"]
)

accuracy_by_crop_count["accuracy_percent"] = (
    accuracy_by_crop_count["accuracy"]
    * 100
)

accuracy_by_crop_count["percentage_of_evaluated_fragments"] = (
    accuracy_by_crop_count["num_fragments"]
    / len(tablet_votes_df)
    * 100
)


# ============================================================
# ADD MODEL INFORMATION
# ============================================================

accuracy_by_crop_count.insert(
    0,
    "run_name",
    RUN_NAME
)

accuracy_by_crop_count.insert(
    1,
    "model",
    DISPLAY_NAME
)

accuracy_by_crop_count.insert(
    2,
    "total_evaluated_fragments",
    len(tablet_votes_df)
)


# ============================================================
# FORMAT NUMERIC COLUMNS
# ============================================================

accuracy_by_crop_count["num_fragments"] = (
    accuracy_by_crop_count["num_fragments"]
    .fillna(0)
    .astype(int)
)

accuracy_by_crop_count["correct_predictions"] = (
    accuracy_by_crop_count["correct_predictions"]
    .fillna(0)
    .astype(int)
)

accuracy_by_crop_count["incorrect_predictions"] = (
    accuracy_by_crop_count["incorrect_predictions"]
    .fillna(0)
    .astype(int)
)

accuracy_by_crop_count["mean_crops_used"] = (
    accuracy_by_crop_count["mean_crops_used"]
    .round(2)
)

accuracy_by_crop_count["median_crops_used"] = (
    accuracy_by_crop_count["median_crops_used"]
    .round(2)
)

accuracy_by_crop_count["accuracy"] = (
    accuracy_by_crop_count["accuracy"]
    .round(4)
)

accuracy_by_crop_count["accuracy_percent"] = (
    accuracy_by_crop_count["accuracy_percent"]
    .round(2)
)

accuracy_by_crop_count[
    "percentage_of_evaluated_fragments"
] = (
    accuracy_by_crop_count[
        "percentage_of_evaluated_fragments"
    ]
    .round(2)
)


# ============================================================
# SELECT OUTPUT COLUMN ORDER
# ============================================================

accuracy_by_crop_count = accuracy_by_crop_count[
    [
        "run_name",
        "model",
        "total_evaluated_fragments",
        "crop_count_group",
        "num_fragments",
        "correct_predictions",
        "incorrect_predictions",
        "accuracy",
        "accuracy_percent",
        "percentage_of_evaluated_fragments",
        "mean_crops_used",
        "median_crops_used"
    ]
]


# ============================================================
# SAVE RESULTS
# ============================================================

accuracy_by_crop_count.to_csv(
    ACCURACY_BY_CROP_COUNT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("=" * 100)
print(f"PERIOD ACCURACY BY NUMBER OF CONFIDENT CROPS â€” {DISPLAY_NAME}")
print("=" * 100)

try:
    display(accuracy_by_crop_count)

except NameError:
    print(
        accuracy_by_crop_count.to_string(
            index=False
        )
    )


# ============================================================
# PRINT SIMPLE SUMMARY
# ============================================================

print()

for _, row in accuracy_by_crop_count.iterrows():

    print(
        f"{row['model']} | "
        f"{row['crop_count_group']}: "
        f"{int(row['num_fragments']):,} fragments | "
        f"{int(row['correct_predictions']):,} correct | "
        f"{int(row['incorrect_predictions']):,} incorrect | "
        f"accuracy = {row['accuracy_percent']:.2f}%"
    )

print()
print("Saved:", ACCURACY_BY_CROP_COUNT_CSV)

In [ ]:
# ============================================================
# VIT BASE + SWIN BASE SIGN CLASSIFICATION
# + TABLET-LEVEL PERIOD VOTING
#
# Supports:
#   1. ViT Base
#   2. Swin Base
#
# INPUT:
#   cuneiform-ocr-main
#       \mongo_unannotated_tablet_outputs
#           \csv_parts
#               detr_crops_part_*.csv
#           \detr_crops
#
# OUTPUT:
#   mongo_unannotated_tablet_outputs/
#       vit_swin_predictions/
#           vit_base/
#               prediction_parts/
#               failed_prediction_crops/
#               mongo_unannotated_crop_predictions.csv
#               mongo_unannotated_tablet_period_votes.csv
#               mongo_unannotated_period_vote_details.csv
#           swin_base/
#               prediction_parts/
#               failed_prediction_crops/
#               mongo_unannotated_crop_predictions.csv
#               mongo_unannotated_tablet_period_votes.csv
#               mongo_unannotated_period_vote_details.csv
#           all_vit_swin_period_vote_summary.csv
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import gc
import time
import textwrap
import importlib
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torchvision import models


# ============================================================
# OPTIONAL TIMM IMPORT
# ============================================================

try:
    import timm
    TIMM_AVAILABLE = True
except ImportError:
    TIMM_AVAILABLE = False


# ============================================================
# PATH SETTINGS
# ============================================================

OCR_ROOT = Path(
    r"cuneiform-ocr-main"
)

OUTPUT_DIR = (
    OCR_ROOT
    / "mongo_unannotated_tablet_outputs"
)

CSV_PARTS_DIR = (
    OUTPUT_DIR
    / "csv_parts"
)

BASE_ROOT = Path(
    r"ebl_tablets_and_sign_crops"
)

EXTENSION_ROOT = (
    BASE_ROOT
    / "sign_classification_extension"
)

RESNET_ROOT = (
    EXTENSION_ROOT
    / "ResNet"
)

TABLET_HOLDOUT_TRAIN_CSV = (
    RESNET_ROOT
    / "tablet_holdout_train.csv"
)

TABLET_HOLDOUT_VAL_CSV = (
    RESNET_ROOT
    / "tablet_holdout_val.csv"
)

TABLET_HOLDOUT_TEST_CSV = (
    RESNET_ROOT
    / "tablet_holdout_test.csv"
)

IMAGE_SPLIT_LABEL_MAPPING_CSV = (
    RESNET_ROOT
    / "combined_train_val_test_label_mapping_482.csv"
)

MODEL_OUTPUT_ROOT = (
    OUTPUT_DIR
    / "vit_swin_predictions"
)

MODEL_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

ALL_MODELS_SUMMARY_CSV = (
    MODEL_OUTPUT_ROOT
    / "all_vit_swin_period_vote_summary.csv"
)


# ============================================================
# ROOT RESOLVER
# ============================================================

def first_existing_dir(candidates, fallback):

    for candidate in candidates:

        candidate = Path(candidate)

        if candidate.exists():

            return candidate

    return fallback


VIT_ROOT = first_existing_dir(
    [
        EXTENSION_ROOT / "ViT",
        EXTENSION_ROOT / "VIT",
        EXTENSION_ROOT / "vit",
        EXTENSION_ROOT / "ViT_Base",
        EXTENSION_ROOT / "VIT_Base",
        EXTENSION_ROOT / "vit_base",
    ],
    EXTENSION_ROOT
)

SWIN_ROOT = first_existing_dir(
    [
        EXTENSION_ROOT / "Swin",
        EXTENSION_ROOT / "SWIN",
        EXTENSION_ROOT / "swin",
        EXTENSION_ROOT / "Swin_Base",
        EXTENSION_ROOT / "SWIN_Base",
        EXTENSION_ROOT / "swin_base",
        EXTENSION_ROOT / "SwinTransformer",
    ],
    EXTENSION_ROOT
)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_CONFIGS = [

    {
        "run_name": "vit_base",
        "display_name": "ViT Base",
        "model_family": "vit",
        "checkpoint_candidates": [
            VIT_ROOT / "best_vit_base_tablet_holdout_sign_classifier.pth",
            VIT_ROOT / "best_vit_base_tablet_holdout_classifier.pth",
            VIT_ROOT / "best_vit_base_holdout_sign_classifier.pth",
            VIT_ROOT / "best_vit_base_holdout.pth",
            VIT_ROOT / "best_vit_b_16_tablet_holdout_sign_classifier.pth",
            VIT_ROOT / "best_vit_b_16_tablet_holdout_classifier.pth",
            VIT_ROOT / "best_vit_base_sign_classifier.pth",
            VIT_ROOT / "best_vit_base_classifier.pth",
            VIT_ROOT / "best_vit_base.pth",
            VIT_ROOT / "vit_base_best.pth",
            VIT_ROOT / "vit_base_sign_classifier.pth",
            EXTENSION_ROOT / "best_vit_base_tablet_holdout_sign_classifier.pth",
            EXTENSION_ROOT / "best_vit_base_sign_classifier.pth",
            EXTENSION_ROOT / "best_vit_base.pth",
        ],
    },

    {
        "run_name": "swin_base",
        "display_name": "Swin Base",
        "model_family": "swin",
        "checkpoint_candidates": [
            SWIN_ROOT / "best_swin_base_tablet_holdout_sign_classifier.pth",
            SWIN_ROOT / "best_swin_base_tablet_holdout_classifier.pth",
            SWIN_ROOT / "best_swin_base_holdout_sign_classifier.pth",
            SWIN_ROOT / "best_swin_base_holdout.pth",
            SWIN_ROOT / "best_swin_b_tablet_holdout_sign_classifier.pth",
            SWIN_ROOT / "best_swin_b_tablet_holdout_classifier.pth",
            SWIN_ROOT / "best_swin_base_sign_classifier.pth",
            SWIN_ROOT / "best_swin_base_classifier.pth",
            SWIN_ROOT / "best_swin_base.pth",
            SWIN_ROOT / "swin_base_best.pth",
            SWIN_ROOT / "swin_base_sign_classifier.pth",
            EXTENSION_ROOT / "best_swin_base_tablet_holdout_sign_classifier.pth",
            EXTENSION_ROOT / "best_swin_base_sign_classifier.pth",
            EXTENSION_ROOT / "best_swin_base.pth",
        ],
    },
]


# ============================================================
# SETTINGS
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 32

NUM_WORKERS = 8

PREFETCH_FACTOR = 4

WRITE_BUFFER_SIZE = 10_000

COMBINE_CHUNK_SIZE = 50_000

SIGN_CONFIDENCE_THRESHOLD = 0.50

OVERWRITE_EXISTING_PARTS = False

FALLBACK_TO_SINGLE_WORKER = True

# Use None to run both models.
# Example:
# RUN_ONLY = {"vit_base"}
# RUN_ONLY = {"swin_base"}
RUN_ONLY = None


# ============================================================
# DEVICE INFO
# ============================================================

print("=" * 90)
print("DEVICE AND SETTINGS")
print("=" * 90)
print("Device              :", DEVICE)
print("Batch size          :", BATCH_SIZE)
print("Workers             :", NUM_WORKERS)
print("Confidence threshold:", SIGN_CONFIDENCE_THRESHOLD)
print("Output dir          :", MODEL_OUTPUT_ROOT)
print("Extension root      :", EXTENSION_ROOT)
print("ViT root            :", VIT_ROOT)
print("Swin root           :", SWIN_ROOT)
print("ResNet root for CSV :", RESNET_ROOT)
print("TIMM available      :", TIMM_AVAILABLE)

if DEVICE.type == "cuda":
    print("GPU                 :", torch.cuda.get_device_name(0))
    print("CUDA                :", torch.version.cuda)

print("=" * 90)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


# ============================================================
# PATH CHECKS
# ============================================================

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"Missing output dir:\n{OUTPUT_DIR}"
    )

if not CSV_PARTS_DIR.exists():
    raise FileNotFoundError(
        f"Missing csv_parts dir:\n{CSV_PARTS_DIR}"
    )

if not EXTENSION_ROOT.exists():
    raise FileNotFoundError(
        f"Missing extension root:\n{EXTENSION_ROOT}"
    )

if not RESNET_ROOT.exists():
    raise FileNotFoundError(
        f"Missing ResNet folder containing tablet_holdout_train.csv:\n{RESNET_ROOT}"
    )

if not TABLET_HOLDOUT_TRAIN_CSV.exists():
    raise FileNotFoundError(
        f"Missing label CSV:\n{TABLET_HOLDOUT_TRAIN_CSV}"
    )


# ============================================================
# FIND DETR CSV PARTS
# ============================================================

csv_part_files = sorted(
    CSV_PARTS_DIR.glob(
        "detr_crops_part_*.csv"
    )
)

if not csv_part_files:

    csv_part_files = sorted(
        OUTPUT_DIR.glob(
            "detr_crops_part_*.csv"
        )
    )

if not csv_part_files:

    raise FileNotFoundError(
        "No detr_crops_part_*.csv files found."
    )

print()
print("=" * 90)
print("DETR CROP CSV FILES")
print("=" * 90)
print("Number of parts:", len(csv_part_files))

for file in csv_part_files:
    print(" -", file.name)


# ============================================================
# CREATE WORKER MODULE
# ============================================================

WORKER_MODULE_NAME = "_cuneiform_crop_loader_vit_swin"

WORKER_MODULE_PATH = (
    OUTPUT_DIR
    / f"{WORKER_MODULE_NAME}.py"
)

worker_module_code = r'''
from pathlib import Path

import torch

from PIL import Image, ImageFile
from torchvision import transforms
from torch.utils.data import Dataset


Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True


class CropDataset(Dataset):

    def __init__(self, crop_paths, output_dir):

        self.crop_paths = [str(path) for path in crop_paths]

        self.output_dir = Path(output_dir)

        self.detr_crops_dir = self.output_dir / "detr_crops"

        self.transform = transforms.Compose([
            transforms.Resize(232),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])


    def __len__(self):

        return len(self.crop_paths)


    def resolve_crop_path(self, crop_path_value):

        crop_path_text = str(crop_path_value).strip()

        if not crop_path_text:
            return None

        crop_path = Path(crop_path_text)

        candidates = [
            crop_path,
            self.output_dir / crop_path,
            self.detr_crops_dir / crop_path,
            self.detr_crops_dir / crop_path.name
        ]

        for candidate in candidates:

            try:

                if candidate.exists():
                    return candidate

            except OSError:
                continue

        return None


    def __getitem__(self, index):

        original_crop_path = self.crop_paths[index]

        resolved_crop_path = self.resolve_crop_path(original_crop_path)

        if resolved_crop_path is None:

            return {
                "index": index,
                "image": None,
                "error": "Crop image file not found",
                "resolved_path": ""
            }

        try:

            with Image.open(resolved_crop_path) as image:

                image = image.convert("RGB")

                image_tensor = self.transform(image)

            return {
                "index": index,
                "image": image_tensor,
                "error": "",
                "resolved_path": str(resolved_crop_path)
            }

        except Exception as error:

            return {
                "index": index,
                "image": None,
                "error": str(error),
                "resolved_path": str(resolved_crop_path)
            }


def crop_collate_fn(batch):

    valid_images = []
    valid_indices = []
    valid_paths = []
    failed_items = []

    for item in batch:

        if item["image"] is None:

            failed_items.append({
                "index": int(item["index"]),
                "error": item["error"],
                "resolved_path": item["resolved_path"]
            })

        else:

            valid_images.append(item["image"])
            valid_indices.append(int(item["index"]))
            valid_paths.append(item["resolved_path"])

    if valid_images:
        image_batch = torch.stack(valid_images, dim=0)
    else:
        image_batch = None

    return {
        "images": image_batch,
        "indices": valid_indices,
        "resolved_paths": valid_paths,
        "failed": failed_items,
        "batch_size": len(batch)
    }


def dataloader_worker_init(worker_id):

    torch.set_num_threads(1)
'''

WORKER_MODULE_PATH.write_text(
    textwrap.dedent(worker_module_code),
    encoding="utf-8"
)

if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))

importlib.invalidate_caches()

if WORKER_MODULE_NAME in sys.modules:
    del sys.modules[WORKER_MODULE_NAME]

worker_module = importlib.import_module(
    WORKER_MODULE_NAME
)

CropDataset = worker_module.CropDataset
crop_collate_fn = worker_module.crop_collate_fn
dataloader_worker_init = worker_module.dataloader_worker_init

print()
print("Worker module created:")
print(WORKER_MODULE_PATH)


# ============================================================
# GENERAL HELPERS
# ============================================================

def first_existing_path(candidates):

    for path in candidates:

        path = Path(path)

        if path.exists():
            return path

    return None


def auto_find_checkpoint(model_family):

    all_pths = sorted(
        EXTENSION_ROOT.rglob("*.pth")
    )

    candidates = []

    for path in all_pths:

        name = path.name.lower()

        if model_family == "vit":

            if "vit" not in name:
                continue

            if (
                "tiny" in name
                or "small" in name
                or "large" in name
                or "huge" in name
            ):
                continue

        elif model_family == "swin":

            if "swin" not in name:
                continue

            if (
                "tiny" in name
                or "small" in name
                or "large" in name
                or "xlarge" in name
            ):
                continue

        else:

            continue

        candidates.append(path)

    if not candidates:
        return None

    def checkpoint_score(path):

        name = path.name.lower()

        score = 0

        if "base" in name:
            score += 30

        if "b_16" in name:
            score += 25

        if "best" in name:
            score += 20

        if "tablet" in name:
            score += 10

        if "holdout" in name:
            score += 10

        if "sign" in name:
            score += 5

        if "classifier" in name:
            score += 5

        return -score, name

    candidates = sorted(
        candidates,
        key=checkpoint_score
    )

    return candidates[0]


def find_checkpoint(model_cfg):

    checkpoint_path = first_existing_path(
        model_cfg["checkpoint_candidates"]
    )

    if checkpoint_path is not None:
        return checkpoint_path

    checkpoint_path = auto_find_checkpoint(
        model_cfg["model_family"]
    )

    if checkpoint_path is None:

        all_pths = sorted(
            EXTENSION_ROOT.rglob("*.pth")
        )

        pth_text = "\n".join(
            str(path)
            for path in all_pths
        )

        raise FileNotFoundError(
            f"Could not find checkpoint for {model_cfg['display_name']}.\n\n"
            f"Searched under:\n{EXTENSION_ROOT}\n\n"
            f"Available .pth files:\n{pth_text}"
        )

    return checkpoint_path


def clean_checkpoint_state_dict(checkpoint):

    if isinstance(checkpoint, dict):

        if "model_state_dict" in checkpoint:
            checkpoint = checkpoint["model_state_dict"]

        elif "state_dict" in checkpoint:
            checkpoint = checkpoint["state_dict"]

        elif "model" in checkpoint:
            checkpoint = checkpoint["model"]

    if not isinstance(checkpoint, dict):
        raise ValueError("Checkpoint is not a valid state_dict.")

    prefixes_to_strip = [
        "module.",
        "_orig_mod.",
        "model."
    ]

    cleaned = {}

    for key, value in checkpoint.items():

        new_key = key

        changed = True

        while changed:

            changed = False

            for prefix in prefixes_to_strip:

                if new_key.startswith(prefix):

                    new_key = new_key[len(prefix):]

                    changed = True

        cleaned[new_key] = value

    return cleaned


def load_checkpoint_state_dict(checkpoint_path):

    try:

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

    except TypeError:

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu"
        )

    return clean_checkpoint_state_dict(
        checkpoint
    )


# ============================================================
# LABEL MAPPING HELPERS
# ============================================================

def create_combined_label_mapping_csv():

    required_columns = {
        "label",
        "signName",
        "period"
    }

    csv_files = [
        TABLET_HOLDOUT_TRAIN_CSV,
        TABLET_HOLDOUT_VAL_CSV,
        TABLET_HOLDOUT_TEST_CSV,
    ]

    dfs = []

    for csv_file in csv_files:

        if not csv_file.exists():
            continue

        df = pd.read_csv(
            csv_file,
            dtype=str,
            keep_default_na=False
        )

        missing = required_columns - set(df.columns)

        if missing:

            raise ValueError(
                f"Missing columns in:\n{csv_file}\n\n"
                f"Missing:\n{sorted(missing)}\n\n"
                f"Available columns:\n{list(df.columns)}"
            )

        dfs.append(df)

    if not dfs:

        raise RuntimeError(
            "Could not create combined label mapping because no CSV files were found."
        )

    combined_df = pd.concat(
        dfs,
        ignore_index=True
    )

    combined_df["label"] = (
        combined_df["label"]
        .astype(str)
        .str.strip()
    )

    combined_df = (
        combined_df[
            combined_df["label"] != ""
        ]
        .copy()
    )

    combined_label_df = (
        combined_df[
            [
                "label",
                "signName",
                "period"
            ]
        ]
        .drop_duplicates(
            subset=["label"]
        )
        .sort_values(
            by="label"
        )
        .reset_index(drop=True)
    )

    combined_label_df.to_csv(
        IMAGE_SPLIT_LABEL_MAPPING_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("=" * 90)
    print("COMBINED LABEL MAPPING CREATED")
    print("=" * 90)
    print("Saved:", IMAGE_SPLIT_LABEL_MAPPING_CSV)
    print("Number of labels:", combined_label_df["label"].nunique())
    print("=" * 90)

    return IMAGE_SPLIT_LABEL_MAPPING_CSV


def load_label_mapping(label_csv):

    df = pd.read_csv(
        label_csv,
        dtype=str,
        keep_default_na=False
    )

    required_columns = {
        "label",
        "signName",
        "period"
    }

    missing = required_columns - set(df.columns)

    if missing:

        raise ValueError(
            f"Missing columns in label CSV:\n{sorted(missing)}\n\n"
            f"File:\n{label_csv}\n\n"
            f"Available columns:\n{list(df.columns)}"
        )

    df["label"] = (
        df["label"]
        .astype(str)
        .str.strip()
    )

    df = (
        df[
            df["label"] != ""
        ]
        .copy()
    )

    labels = sorted(
        df["label"].unique()
    )

    label_to_idx = {
        label: idx
        for idx, label in enumerate(labels)
    }

    idx_to_label = {
        idx: label
        for label, idx in label_to_idx.items()
    }

    label_info = (
        df[
            [
                "label",
                "signName",
                "period"
            ]
        ]
        .drop_duplicates(subset=["label"])
        .set_index("label")
        .to_dict("index")
    )

    return {
        "df": df,
        "labels": labels,
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "label_info": label_info,
        "num_classes": len(labels)
    }


def choose_label_csv_for_checkpoint(checkpoint_num_classes):

    if checkpoint_num_classes == 471:

        return TABLET_HOLDOUT_TRAIN_CSV

    if checkpoint_num_classes == 482:

        label_csv = create_combined_label_mapping_csv()

        label_bundle = load_label_mapping(
            label_csv
        )

        if label_bundle["num_classes"] != 482:

            raise RuntimeError(
                "Checkpoint has 482 classes, but the available CSV files "
                "do not provide a 482-class label mapping.\n\n"
                f"Combined label CSV:\n{label_csv}\n"
                f"Classes found: {label_bundle['num_classes']}\n\n"
                "You need the original 482-class image-split training CSV used "
                "when training this checkpoint."
            )

        return label_csv

    raise RuntimeError(
        f"Unsupported checkpoint class count: {checkpoint_num_classes}\n\n"
        "Expected either 471 for tablet-holdout or 482 for image-split."
    )


# ============================================================
# ARCHITECTURE DETECTION
# ============================================================

def detect_model_implementation(model_family, state_dict):
    """
    Detect whether checkpoint is torchvision or timm style.
    """

    keys = list(state_dict.keys())

    key_set = set(keys)

    if model_family == "vit":

        if (
            "heads.head.weight" in key_set
            or "conv_proj.weight" in key_set
            or any(key.startswith("encoder.layers.") for key in keys)
        ):
            return "torchvision_vit_b_16"

        if (
            "head.weight" in key_set
            or "patch_embed.proj.weight" in key_set
            or any(key.startswith("blocks.") for key in keys)
        ):
            return "timm_vit_base_patch16_224"

    if model_family == "swin":

        if (
            "head.weight" in key_set
            and any(key.startswith("features.") for key in keys)
        ):
            return "torchvision_swin_b"

        if (
            "head.weight" in key_set
            and (
                "patch_embed.proj.weight" in key_set
                or any(key.startswith("layers.") for key in keys)
            )
        ):
            return "timm_swin_base_patch4_window7_224"

    raise RuntimeError(
        f"Could not detect model implementation for {model_family}.\n\n"
        "First 30 checkpoint keys:\n"
        + "\n".join(keys[:30])
    )


def get_num_classes_from_checkpoint(model_impl, state_dict):

    possible_keys = []

    if model_impl == "torchvision_vit_b_16":
        possible_keys = [
            "heads.head.weight",
            "head.weight",
            "classifier.weight",
            "fc.weight"
        ]

    elif model_impl == "timm_vit_base_patch16_224":
        possible_keys = [
            "head.weight",
            "heads.head.weight",
            "classifier.weight",
            "fc.weight"
        ]

    elif model_impl == "torchvision_swin_b":
        possible_keys = [
            "head.weight",
            "classifier.weight",
            "fc.weight"
        ]

    elif model_impl == "timm_swin_base_patch4_window7_224":
        possible_keys = [
            "head.weight",
            "classifier.weight",
            "fc.weight"
        ]

    for key in possible_keys:

        if key in state_dict:

            weight = state_dict[key]

            if hasattr(weight, "shape") and len(weight.shape) == 2:

                return int(weight.shape[0]), key

    raise RuntimeError(
        f"Could not infer number of classes for {model_impl}.\n\n"
        "First 30 checkpoint keys:\n"
        + "\n".join(list(state_dict.keys())[:30])
    )


# ============================================================
# MODEL BUILDERS
# ============================================================

def build_model(model_impl, num_classes):

    if model_impl == "torchvision_vit_b_16":

        model = models.vit_b_16(
            weights=None
        )

        in_features = model.heads.head.in_features

        model.heads.head = nn.Linear(
            in_features,
            num_classes
        )

        return model

    if model_impl == "torchvision_swin_b":

        model = models.swin_b(
            weights=None
        )

        in_features = model.head.in_features

        model.head = nn.Linear(
            in_features,
            num_classes
        )

        return model

    if model_impl == "timm_vit_base_patch16_224":

        if not TIMM_AVAILABLE:

            raise ImportError(
                "This ViT checkpoint appears to be timm-based, but timm is not installed.\n"
                "Install it using:\n"
                "pip install timm"
            )

        model = timm.create_model(
            "vit_base_patch16_224",
            pretrained=False,
            num_classes=num_classes
        )

        return model

    if model_impl == "timm_swin_base_patch4_window7_224":

        if not TIMM_AVAILABLE:

            raise ImportError(
                "This Swin checkpoint appears to be timm-based, but timm is not installed.\n"
                "Install it using:\n"
                "pip install timm"
            )

        model = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=False,
            num_classes=num_classes
        )

        return model

    raise ValueError(
        f"Unsupported model implementation: {model_impl}"
    )


def load_model_for_run(model_cfg, checkpoint_path, num_classes):

    state_dict = load_checkpoint_state_dict(
        checkpoint_path
    )

    model_impl = detect_model_implementation(
        model_cfg["model_family"],
        state_dict
    )

    checkpoint_num_classes, classifier_key = get_num_classes_from_checkpoint(
        model_impl,
        state_dict
    )

    print("Detected implementation:", model_impl)
    print("Classes from label CSV :", num_classes)
    print("Classes from checkpoint:", checkpoint_num_classes)
    print("Classifier key         :", classifier_key)

    if checkpoint_num_classes != num_classes:

        raise RuntimeError(
            "Class-count mismatch.\n\n"
            f"Model: {model_cfg['display_name']}\n"
            f"Checkpoint: {checkpoint_path}\n"
            f"Label CSV classes: {num_classes}\n"
            f"Checkpoint classes: {checkpoint_num_classes}\n\n"
            "The selected label mapping does not match this checkpoint."
        )

    model = build_model(
        model_impl=model_impl,
        num_classes=num_classes
    )

    try:

        model.load_state_dict(
            state_dict,
            strict=True
        )

    except RuntimeError as error:

        raise RuntimeError(
            f"Failed to load checkpoint strictly for {model_cfg['display_name']}.\n\n"
            f"Detected implementation: {model_impl}\n"
            f"Checkpoint:\n{checkpoint_path}\n\n"
            "First 30 checkpoint keys:\n"
            + "\n".join(list(state_dict.keys())[:30])
            + "\n\nOriginal error:\n"
            + str(error)
        )

    model = model.to(
        DEVICE
    )

    if DEVICE.type == "cuda":

        model = model.to(
            memory_format=torch.channels_last
        )

    model.eval()

    return model, model_impl, classifier_key, checkpoint_num_classes


def warmup_model(model):

    if DEVICE.type != "cuda":
        return

    print()
    print("Warming up GPU...")

    warmup_batch = torch.zeros(
        (BATCH_SIZE, 3, 224, 224),
        device=DEVICE
    )

    warmup_batch = warmup_batch.contiguous(
        memory_format=torch.channels_last
    )

    with torch.inference_mode():

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            _ = model(
                warmup_batch
            )

    torch.cuda.synchronize()

    del warmup_batch

    torch.cuda.empty_cache()

    print("GPU warm-up completed.")


# ============================================================
# DATALOADER + CSV HELPERS
# ============================================================

def create_dataloader(dataset, num_workers):

    loader_args = {
        "dataset": dataset,
        "batch_size": BATCH_SIZE,
        "shuffle": False,
        "num_workers": num_workers,
        "pin_memory": DEVICE.type == "cuda",
        "drop_last": False,
        "collate_fn": crop_collate_fn
    }

    if num_workers > 0:

        loader_args.update({
            "prefetch_factor": PREFETCH_FACTOR,
            "persistent_workers": False,
            "worker_init_fn": dataloader_worker_init
        })

    return DataLoader(
        **loader_args
    )


def append_rows_to_csv(rows, output_path, header_written):

    if not rows:
        return header_written

    pd.DataFrame(rows).to_csv(
        output_path,
        mode="a",
        index=False,
        header=not header_written
    )

    return True


# ============================================================
# CLASSIFICATION
# ============================================================

def classify_tensor_batch(
    image_batch,
    model,
    idx_to_label,
    label_info
):

    if image_batch is None:
        return []

    image_batch = image_batch.to(
        DEVICE,
        non_blocking=True
    )

    if DEVICE.type == "cuda":

        image_batch = image_batch.contiguous(
            memory_format=torch.channels_last
        )

    with torch.inference_mode():

        if DEVICE.type == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(
                    image_batch
                )

        else:

            logits = model(
                image_batch
            )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        top_probabilities, top_indices = probabilities.max(
            dim=1
        )

    top_probabilities = (
        top_probabilities
        .detach()
        .float()
        .cpu()
        .tolist()
    )

    top_indices = (
        top_indices
        .detach()
        .cpu()
        .tolist()
    )

    results = []

    for pred_idx, pred_conf in zip(
        top_indices,
        top_probabilities
    ):

        pred_idx = int(
            pred_idx
        )

        pred_label = idx_to_label[
            pred_idx
        ]

        metadata = label_info.get(
            pred_label,
            {}
        )

        results.append({
            "pred_label": pred_label,
            "pred_signName": metadata.get("signName", ""),
            "pred_period": metadata.get("period", ""),
            "pred_confidence": float(pred_conf)
        })

    return results


# ============================================================
# PROCESS ONE DETR CSV PART
# ============================================================

def process_prediction_part(
    input_csv,
    num_workers,
    model,
    idx_to_label,
    label_info,
    prediction_parts_dir,
    failed_crops_dir
):

    part_suffix = input_csv.stem.replace(
        "detr_crops_part_",
        ""
    )

    output_part_csv = (
        prediction_parts_dir
        / f"crop_predictions_part_{part_suffix}.csv"
    )

    temporary_output_csv = Path(
        str(output_part_csv)
        + ".tmp"
    )

    failed_part_csv = (
        failed_crops_dir
        / f"failed_crops_part_{part_suffix}.csv"
    )

    if output_part_csv.exists() and not OVERWRITE_EXISTING_PARTS:

        print("Prediction part already exists. Skipping:")
        print(output_part_csv)

        return {
            "status": "skipped",
            "output_path": output_part_csv,
            "input_rows": 0,
            "predictions": 0,
            "failures": 0,
            "elapsed_seconds": 0.0,
            "gpu_batches": 0
        }

    if output_part_csv.exists():
        output_part_csv.unlink()

    if temporary_output_csv.exists():
        temporary_output_csv.unlink()

    if failed_part_csv.exists():
        failed_part_csv.unlink()

    print()
    print("Reading:")
    print(input_csv)

    part_df = pd.read_csv(
        input_csv,
        dtype=str,
        keep_default_na=False
    )

    if "cropPath" not in part_df.columns:

        raise ValueError(
            f"'cropPath' column missing in:\n{input_csv}"
        )

    number_of_rows = len(part_df)

    source_records = part_df.to_dict(
        orient="records"
    )

    crop_paths = (
        part_df["cropPath"]
        .astype(str)
        .tolist()
    )

    del part_df

    gc.collect()

    dataset = CropDataset(
        crop_paths=crop_paths,
        output_dir=str(OUTPUT_DIR)
    )

    loader = create_dataloader(
        dataset,
        num_workers
    )

    prediction_buffer = []
    failure_buffer = []

    prediction_header_written = False
    failure_header_written = False

    total_predictions = 0
    total_failures = 0
    total_gpu_batches = 0

    start_time = time.time()

    progress_bar = tqdm(
        total=number_of_rows,
        desc=input_csv.name,
        unit="crop"
    )

    try:

        for loader_batch in loader:

            current_batch_size = int(
                loader_batch["batch_size"]
            )

            for failed_item in loader_batch["failed"]:

                failed_index = int(
                    failed_item["index"]
                )

                failed_row = dict(
                    source_records[failed_index]
                )

                failed_row.update({
                    "failure_reason": failed_item["error"],
                    "resolved_crop_path": failed_item["resolved_path"]
                })

                failure_buffer.append(
                    failed_row
                )

                total_failures += 1

            image_batch = loader_batch["images"]

            valid_indices = loader_batch["indices"]

            resolved_paths = loader_batch["resolved_paths"]

            if image_batch is not None:

                batch_predictions = classify_tensor_batch(
                    image_batch=image_batch,
                    model=model,
                    idx_to_label=idx_to_label,
                    label_info=label_info
                )

                total_gpu_batches += 1

                for source_index, resolved_path, prediction in zip(
                    valid_indices,
                    resolved_paths,
                    batch_predictions
                ):

                    prediction_row = dict(
                        source_records[int(source_index)]
                    )

                    prediction_row.update(
                        prediction
                    )

                    prediction_row["resolved_crop_path"] = resolved_path

                    prediction_buffer.append(
                        prediction_row
                    )

                    total_predictions += 1

            if len(prediction_buffer) >= WRITE_BUFFER_SIZE:

                prediction_header_written = append_rows_to_csv(
                    prediction_buffer,
                    temporary_output_csv,
                    prediction_header_written
                )

                prediction_buffer = []

            if len(failure_buffer) >= WRITE_BUFFER_SIZE:

                failure_header_written = append_rows_to_csv(
                    failure_buffer,
                    failed_part_csv,
                    failure_header_written
                )

                failure_buffer = []

            progress_bar.update(
                current_batch_size
            )

            elapsed = time.time() - start_time

            crops_per_second = (
                progress_bar.n / elapsed
                if elapsed > 0
                else 0.0
            )

            progress_bar.set_postfix({
                "GPU_batches": total_gpu_batches,
                "predicted": total_predictions,
                "failed": total_failures,
                "crops/s": f"{crops_per_second:.1f}"
            })

    finally:

        progress_bar.close()

    prediction_header_written = append_rows_to_csv(
        prediction_buffer,
        temporary_output_csv,
        prediction_header_written
    )

    failure_header_written = append_rows_to_csv(
        failure_buffer,
        failed_part_csv,
        failure_header_written
    )

    if not temporary_output_csv.exists():

        raise RuntimeError(
            f"No predictions produced for:\n{input_csv}"
        )

    os.replace(
        temporary_output_csv,
        output_part_csv
    )

    elapsed = time.time() - start_time

    del loader
    del dataset
    del source_records
    del crop_paths

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "status": "completed",
        "output_path": output_part_csv,
        "input_rows": number_of_rows,
        "predictions": total_predictions,
        "failures": total_failures,
        "elapsed_seconds": elapsed,
        "gpu_batches": total_gpu_batches
    }


# ============================================================
# COMBINE PREDICTION PARTS
# ============================================================

def combine_prediction_parts(prediction_parts_dir, crop_predictions_csv):

    prediction_part_files = sorted(
        prediction_parts_dir.glob(
            "crop_predictions_part_*.csv"
        )
    )

    if not prediction_part_files:

        raise RuntimeError(
            f"No prediction parts found in:\n{prediction_parts_dir}"
        )

    tmp_csv = Path(
        str(crop_predictions_csv)
        + ".tmp"
    )

    if tmp_csv.exists():
        tmp_csv.unlink()

    header_written = False

    total_rows = 0

    for part_file in tqdm(
        prediction_part_files,
        desc="Combining prediction files",
        unit="file"
    ):

        reader = pd.read_csv(
            part_file,
            dtype=str,
            keep_default_na=False,
            chunksize=COMBINE_CHUNK_SIZE
        )

        for chunk in reader:

            chunk.to_csv(
                tmp_csv,
                mode="a",
                index=False,
                header=not header_written
            )

            header_written = True

            total_rows += len(chunk)

    os.replace(
        tmp_csv,
        crop_predictions_csv
    )

    return prediction_part_files, total_rows


# ============================================================
# TABLET-LEVEL PERIOD VOTING
# ============================================================

def run_tablet_level_period_voting(
    crop_predictions_csv,
    tablet_period_votes_csv,
    period_details_csv
):

    all_predictions_df = pd.read_csv(
        crop_predictions_csv,
        dtype=str,
        keep_default_na=False
    )

    required_cols = {
        "fragmentNumber",
        "period",
        "pred_period",
        "pred_confidence"
    }

    missing = required_cols - set(
        all_predictions_df.columns
    )

    if missing:

        raise ValueError(
            f"Missing columns in predictions:\n{sorted(missing)}"
        )

    all_predictions_df["pred_confidence"] = pd.to_numeric(
        all_predictions_df["pred_confidence"],
        errors="coerce"
    )

    all_predictions_df = (
        all_predictions_df
        .dropna(
            subset=[
                "fragmentNumber",
                "pred_period",
                "pred_confidence"
            ]
        )
        .copy()
    )

    all_predictions_df["fragmentNumber"] = (
        all_predictions_df["fragmentNumber"]
        .astype(str)
        .str.strip()
    )

    all_predictions_df["pred_period"] = (
        all_predictions_df["pred_period"]
        .astype(str)
        .str.strip()
    )

    all_predictions_df = (
        all_predictions_df[
            (all_predictions_df["fragmentNumber"] != "")
            &
            (all_predictions_df["pred_period"] != "")
        ]
        .copy()
    )

    fragment_counts_before_threshold = (
        all_predictions_df
        .groupby("fragmentNumber")
        .size()
        .to_dict()
    )

    total_fragments_before_threshold = (
        all_predictions_df["fragmentNumber"]
        .nunique()
    )

    predictions_before_threshold = len(
        all_predictions_df
    )

    pred_df = (
        all_predictions_df[
            all_predictions_df["pred_confidence"]
            >= SIGN_CONFIDENCE_THRESHOLD
        ]
        .copy()
    )

    predictions_after_threshold = len(
        pred_df
    )

    evaluated_fragment_count = (
        pred_df["fragmentNumber"]
        .nunique()
    )

    fragments_without_confident_crops = (
        total_fragments_before_threshold
        - evaluated_fragment_count
    )

    print()
    print("=" * 90)
    print("TABLET-LEVEL VOTING INPUT")
    print("=" * 90)
    print("Predictions before filtering:", f"{predictions_before_threshold:,}")
    print("Predictions after filtering :", f"{predictions_after_threshold:,}")
    print("Fragments before filtering  :", f"{total_fragments_before_threshold:,}")
    print("Fragments evaluated         :", f"{evaluated_fragment_count:,}")
    print("Fragments without confident :", f"{fragments_without_confident_crops:,}")

    if len(pred_df) == 0:

        raise RuntimeError(
            "No predictions remain after threshold filtering."
        )

    tablet_rows = []
    detail_rows = []

    grouped = pred_df.groupby(
        "fragmentNumber",
        sort=False
    )

    for fragment_number, group in tqdm(
        grouped,
        total=evaluated_fragment_count,
        desc="Tablet-level period voting",
        unit="fragment"
    ):

        period_values = (
            group["period"]
            .astype(str)
            .str.strip()
        )

        period_values = period_values[
            period_values != ""
        ]

        if len(period_values) > 0:

            true_period = (
                period_values
                .mode()
                .iloc[0]
            )

        else:

            true_period = ""

        period_summary = (
            group
            .groupby("pred_period", dropna=False)
            .agg(
                num_votes=("pred_period", "count"),
                mean_confidence=("pred_confidence", "mean"),
                weighted_score=("pred_confidence", "sum")
            )
            .reset_index()
        )

        period_summary = (
            period_summary
            .sort_values(
                by=[
                    "weighted_score",
                    "num_votes",
                    "mean_confidence",
                    "pred_period"
                ],
                ascending=[
                    False,
                    False,
                    False,
                    True
                ]
            )
            .reset_index(drop=True)
        )

        total_weight = (
            period_summary["weighted_score"]
            .sum()
        )

        if total_weight > 0:

            period_summary["vote_probability"] = (
                period_summary["weighted_score"]
                / total_weight
            )

        else:

            period_summary["vote_probability"] = 0.0

        best_row = period_summary.iloc[0]

        predicted_period = str(
            best_row["pred_period"]
        )

        correct = (
            true_period
            == predicted_period
        )

        if len(period_summary) > 1:

            second_vote_probability = float(
                period_summary.iloc[1]["vote_probability"]
            )

        else:

            second_vote_probability = 0.0

        winning_vote_probability = float(
            best_row["vote_probability"]
        )

        vote_margin = (
            winning_vote_probability
            - second_vote_probability
        )

        tablet_rows.append({
            "fragmentNumber": fragment_number,
            "true_period": true_period,
            "predicted_period": predicted_period,
            "num_crops_before_threshold": int(
                fragment_counts_before_threshold.get(
                    fragment_number,
                    len(group)
                )
            ),
            "num_crops_used": int(len(group)),
            "winning_votes": int(best_row["num_votes"]),
            "winning_mean_confidence": float(best_row["mean_confidence"]),
            "winning_weighted_score": float(best_row["weighted_score"]),
            "winning_vote_probability": winning_vote_probability,
            "second_vote_probability": second_vote_probability,
            "vote_margin": float(vote_margin),
            "correct": bool(correct)
        })

        for rank, row in period_summary.iterrows():

            detail_rows.append({
                "fragmentNumber": fragment_number,
                "true_period": true_period,
                "rank": int(rank + 1),
                "candidate_period": str(row["pred_period"]),
                "num_votes": int(row["num_votes"]),
                "mean_confidence": float(row["mean_confidence"]),
                "weighted_score": float(row["weighted_score"]),
                "vote_probability": float(row["vote_probability"])
            })

    tablet_votes_df = pd.DataFrame(
        tablet_rows
    )

    period_details_df = pd.DataFrame(
        detail_rows
    )

    tablet_votes_df.to_csv(
        tablet_period_votes_csv,
        index=False
    )

    period_details_df.to_csv(
        period_details_csv,
        index=False
    )

    if len(tablet_votes_df) > 0:

        tablet_votes_df["correct"] = (
            tablet_votes_df["correct"]
            .astype(bool)
        )

        accuracy = (
            tablet_votes_df["correct"]
            .mean()
        )

        number_correct = int(
            tablet_votes_df["correct"]
            .sum()
        )

        number_evaluated = len(
            tablet_votes_df
        )

        number_incorrect = (
            number_evaluated
            - number_correct
        )

    else:

        accuracy = 0.0
        number_correct = 0
        number_evaluated = 0
        number_incorrect = 0

    return {
        "predictions_before_threshold": int(predictions_before_threshold),
        "predictions_after_threshold": int(predictions_after_threshold),
        "predictions_removed": int(predictions_before_threshold - predictions_after_threshold),
        "fragments_before_threshold": int(total_fragments_before_threshold),
        "fragments_evaluated": int(number_evaluated),
        "fragments_without_confident_crops": int(fragments_without_confident_crops),
        "correct_period_predictions": int(number_correct),
        "incorrect_period_predictions": int(number_incorrect),
        "tablet_level_accuracy": float(accuracy),
        "tablet_votes_df": tablet_votes_df
    }


# ============================================================
# RUN ONE MODEL
# ============================================================

def run_one_model(model_cfg):

    run_name = model_cfg["run_name"]
    display_name = model_cfg["display_name"]

    print()
    print("#" * 100)
    print("STARTING MODEL:", display_name)
    print("#" * 100)

    model_output_dir = (
        MODEL_OUTPUT_ROOT
        / run_name
    )

    prediction_parts_dir = (
        model_output_dir
        / "prediction_parts"
    )

    failed_crops_dir = (
        model_output_dir
        / "failed_prediction_crops"
    )

    prediction_parts_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    failed_crops_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    crop_predictions_csv = (
        model_output_dir
        / "mongo_unannotated_crop_predictions.csv"
    )

    tablet_period_votes_csv = (
        model_output_dir
        / "mongo_unannotated_tablet_period_votes.csv"
    )

    period_details_csv = (
        model_output_dir
        / "mongo_unannotated_period_vote_details.csv"
    )

    checkpoint_path = find_checkpoint(
        model_cfg
    )

    checkpoint_state_dict = load_checkpoint_state_dict(
        checkpoint_path
    )

    model_impl = detect_model_implementation(
        model_cfg["model_family"],
        checkpoint_state_dict
    )

    checkpoint_num_classes, classifier_key = get_num_classes_from_checkpoint(
        model_impl,
        checkpoint_state_dict
    )

    label_csv = choose_label_csv_for_checkpoint(
        checkpoint_num_classes
    )

    label_bundle = load_label_mapping(
        label_csv
    )

    idx_to_label = label_bundle["idx_to_label"]
    label_info = label_bundle["label_info"]
    num_classes = label_bundle["num_classes"]

    print()
    print("=" * 90)
    print("MODEL FILES")
    print("=" * 90)
    print("Model                  :", display_name)
    print("Run name               :", run_name)
    print("Detected implementation:", model_impl)
    print("Checkpoint             :", checkpoint_path)
    print("Classifier key         :", classifier_key)
    print("Checkpoint classes     :", checkpoint_num_classes)
    print("Label CSV              :", label_csv)
    print("Number of label classes:", num_classes)
    print("Output dir             :", model_output_dir)
    print("=" * 90)

    model, model_impl, classifier_key, checkpoint_num_classes = load_model_for_run(
        model_cfg=model_cfg,
        checkpoint_path=checkpoint_path,
        num_classes=num_classes
    )

    warmup_model(
        model
    )

    for part_idx, input_csv in enumerate(
        csv_part_files,
        start=1
    ):

        print()
        print("=" * 90)
        print(f"PROCESSING PART {part_idx}/{len(csv_part_files)}")
        print("Model:", display_name)
        print("Input:", input_csv)
        print("=" * 90)

        try:

            stats = process_prediction_part(
                input_csv=input_csv,
                num_workers=NUM_WORKERS,
                model=model,
                idx_to_label=idx_to_label,
                label_info=label_info,
                prediction_parts_dir=prediction_parts_dir,
                failed_crops_dir=failed_crops_dir
            )

        except Exception as error:

            if NUM_WORKERS > 0 and FALLBACK_TO_SINGLE_WORKER:

                print()
                print("Multi-worker failed. Retrying with num_workers=0.")
                print("Original error:", repr(error))

                stats = process_prediction_part(
                    input_csv=input_csv,
                    num_workers=0,
                    model=model,
                    idx_to_label=idx_to_label,
                    label_info=label_info,
                    prediction_parts_dir=prediction_parts_dir,
                    failed_crops_dir=failed_crops_dir
                )

            else:

                raise

        print("Part status:", stats["status"])

        if stats["status"] == "completed":

            print("Predictions:", f'{stats["predictions"]:,}')
            print("Failures   :", f'{stats["failures"]:,}')
            print("Elapsed min:", f'{stats["elapsed_seconds"] / 60:.2f}')

    print()
    print("=" * 90)
    print("COMBINING PREDICTION PARTS")
    print("=" * 90)

    prediction_part_files, combined_rows = combine_prediction_parts(
        prediction_parts_dir=prediction_parts_dir,
        crop_predictions_csv=crop_predictions_csv
    )

    print("Combined rows:", f"{combined_rows:,}")

    voting_stats = run_tablet_level_period_voting(
        crop_predictions_csv=crop_predictions_csv,
        tablet_period_votes_csv=tablet_period_votes_csv,
        period_details_csv=period_details_csv
    )

    print()
    print("=" * 90)
    print("FINAL RESULTS:", display_name)
    print("=" * 90)
    print("Total crop predictions           :", f"{combined_rows:,}")
    print("Crop predictions used            :", f'{voting_stats["predictions_after_threshold"]:,}')
    print("Fragments before threshold       :", f'{voting_stats["fragments_before_threshold"]:,}')
    print("Fragments evaluated              :", f'{voting_stats["fragments_evaluated"]:,}')
    print("Fragments without confident crops:", f'{voting_stats["fragments_without_confident_crops"]:,}')
    print("Correct period predictions       :", f'{voting_stats["correct_period_predictions"]:,}')
    print("Incorrect period predictions     :", f'{voting_stats["incorrect_period_predictions"]:,}')
    print("Tablet-level accuracy            :", f'{voting_stats["tablet_level_accuracy"]:.4f}')
    print("Tablet-level accuracy (%)        :", f'{voting_stats["tablet_level_accuracy"] * 100:.2f}%')

    summary_row = {
        "run_name": run_name,
        "display_name": display_name,
        "model_family": model_cfg["model_family"],
        "detected_implementation": model_impl,
        "label_csv": str(label_csv),
        "checkpoint": str(checkpoint_path),
        "classifier_key": classifier_key,
        "checkpoint_num_classes": int(checkpoint_num_classes),
        "label_num_classes": int(num_classes),
        "output_dir": str(model_output_dir),
        "detr_csv_parts": int(len(csv_part_files)),
        "prediction_csv_parts": int(len(prediction_part_files)),
        "combined_prediction_rows": int(combined_rows),
        "predictions_before_threshold": voting_stats["predictions_before_threshold"],
        "predictions_after_threshold": voting_stats["predictions_after_threshold"],
        "predictions_removed": voting_stats["predictions_removed"],
        "fragments_before_threshold": voting_stats["fragments_before_threshold"],
        "fragments_evaluated": voting_stats["fragments_evaluated"],
        "fragments_without_confident_crops": voting_stats["fragments_without_confident_crops"],
        "correct_period_predictions": voting_stats["correct_period_predictions"],
        "incorrect_period_predictions": voting_stats["incorrect_period_predictions"],
        "tablet_level_accuracy": voting_stats["tablet_level_accuracy"],
        "tablet_level_accuracy_percent": voting_stats["tablet_level_accuracy"] * 100.0,
        "confidence_threshold": float(SIGN_CONFIDENCE_THRESHOLD),
    }

    print()
    print("Saved crop predictions:")
    print(crop_predictions_csv)

    print()
    print("Saved tablet-period votes:")
    print(tablet_period_votes_csv)

    print()
    print("Saved period details:")
    print(period_details_csv)

    del model

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return summary_row


# ============================================================
# RUN ALL MODELS
# ============================================================

print()
print("#" * 100)
print("STARTING VIT + SWIN RUNS")
print("#" * 100)

summary_rows = []

for model_cfg in MODEL_CONFIGS:

    if RUN_ONLY is not None:

        if model_cfg["run_name"] not in RUN_ONLY:

            print()
            print("Skipping:", model_cfg["run_name"])
            continue

    summary_row = run_one_model(
        model_cfg
    )

    summary_rows.append(
        summary_row
    )

    summary_df = pd.DataFrame(
        summary_rows
    )

    summary_df.to_csv(
        ALL_MODELS_SUMMARY_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("Intermediate summary saved:")
    print(ALL_MODELS_SUMMARY_CSV)


summary_df = pd.DataFrame(
    summary_rows
)

summary_df.to_csv(
    ALL_MODELS_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

print()
print("#" * 100)
print("ALL VIT + SWIN RUNS COMPLETED")
print("#" * 100)

print()
print("Summary saved:")
print(ALL_MODELS_SUMMARY_CSV)

try:

    display(summary_df)

except NameError:

    print(
        summary_df.to_string(
            index=False
        )
    )

In [ ]:
# ============================================================
# PERIOD ACCURACY BY NUMBER OF CONFIDENT CROPS
# FOR VIT BASE AND SWIN BASE
#
# GROUPS:
#   1 confident crop
#   2-4 confident crops
#   5-9 confident crops
#   10 or more confident crops
#
# INPUT:
#   mongo_unannotated_tablet_outputs/
#       vit_swin_predictions/
#           vit_base/
#               mongo_unannotated_tablet_period_votes.csv
#           swin_base/
#               mongo_unannotated_tablet_period_votes.csv
#
# OUTPUT:
#   Each model folder:
#       period_accuracy_by_crop_count.csv
#
#   Combined:
#       all_vit_swin_accuracy_by_crop_count.csv
# ============================================================

from pathlib import Path

import pandas as pd


# ============================================================
# PATHS
# ============================================================

OUTPUT_DIR = Path(
    r"cuneiform-ocr-main"
    r"\mongo_unannotated_tablet_outputs"
)

MODEL_OUTPUT_ROOT = (
    OUTPUT_DIR
    / "vit_swin_predictions"
)

COMBINED_ACCURACY_CSV = (
    MODEL_OUTPUT_ROOT
    / "all_vit_swin_accuracy_by_crop_count.csv"
)


# ============================================================
# MODELS TO PROCESS
# ============================================================

MODEL_CONFIGS = [

    {
        "run_name": "vit_base",
        "display_name": "ViT Base",
    },

    {
        "run_name": "swin_base",
        "display_name": "Swin Base",
    },
]


# ============================================================
# CROP-COUNT GROUP FUNCTION
# ============================================================

def assign_crop_count_group(num_crops):
    """
    Assign each fragment to one of four evidence groups.
    """

    if num_crops == 1:
        return "1 confident crop"

    if 2 <= num_crops <= 4:
        return "2â€“4 confident crops"

    if 5 <= num_crops <= 9:
        return "5â€“9 confident crops"

    if num_crops >= 10:
        return "10 or more confident crops"

    return "No confident crops"


GROUP_ORDER = [
    "1 confident crop",
    "2â€“4 confident crops",
    "5â€“9 confident crops",
    "10 or more confident crops"
]


# ============================================================
# BOOLEAN CLEANING
# ============================================================

def clean_correct_column(series):
    """
    Convert correct column safely to boolean.

    Handles:
        True / False
        true / false
        1 / 0
        yes / no
    """

    if series.dtype == bool:
        return series

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False
        })
    )


# ============================================================
# PROCESS ONE MODEL
# ============================================================

def process_one_model(model_cfg):
    """
    Calculate period accuracy by number of confident crops
    for one model.
    """

    run_name = model_cfg["run_name"]
    display_name = model_cfg["display_name"]

    model_dir = (
        MODEL_OUTPUT_ROOT
        / run_name
    )

    tablet_period_votes_csv = (
        model_dir
        / "mongo_unannotated_tablet_period_votes.csv"
    )

    accuracy_by_crop_count_csv = (
        model_dir
        / "period_accuracy_by_crop_count.csv"
    )

    if not tablet_period_votes_csv.exists():

        print()
        print("=" * 100)
        print("SKIPPING MODEL â€” FILE NOT FOUND")
        print("=" * 100)
        print("Model:", display_name)
        print("Missing file:")
        print(tablet_period_votes_csv)
        print()
        print("This usually means the prediction/voting script has not been run for this model yet.")

        return None

    print()
    print("=" * 100)
    print("PROCESSING MODEL")
    print("=" * 100)
    print("Model:", display_name)
    print("Input:", tablet_period_votes_csv)

    # --------------------------------------------------------
    # LOAD TABLET-LEVEL RESULTS
    # --------------------------------------------------------

    tablet_votes_df = pd.read_csv(
        tablet_period_votes_csv,
        dtype={
            "fragmentNumber": str,
            "true_period": str,
            "predicted_period": str
        },
        keep_default_na=False
    )

    print(
        "Loaded tablet-level predictions:",
        f"{len(tablet_votes_df):,}"
    )

    # --------------------------------------------------------
    # CHECK REQUIRED COLUMNS
    # --------------------------------------------------------

    required_columns = {
        "fragmentNumber",
        "true_period",
        "predicted_period",
        "num_crops_used",
        "correct"
    }

    missing_columns = (
        required_columns
        - set(tablet_votes_df.columns)
    )

    if missing_columns:

        raise ValueError(
            f"Missing columns in:\n{tablet_period_votes_csv}\n\n"
            f"Missing columns:\n{sorted(missing_columns)}\n\n"
            f"Available columns:\n{list(tablet_votes_df.columns)}"
        )

    # --------------------------------------------------------
    # CLEAN REQUIRED COLUMNS
    # --------------------------------------------------------

    tablet_votes_df["num_crops_used"] = pd.to_numeric(
        tablet_votes_df["num_crops_used"],
        errors="coerce"
    )

    tablet_votes_df["correct"] = clean_correct_column(
        tablet_votes_df["correct"]
    )

    tablet_votes_df = tablet_votes_df.dropna(
        subset=[
            "num_crops_used",
            "correct"
        ]
    ).copy()

    tablet_votes_df["num_crops_used"] = (
        tablet_votes_df["num_crops_used"]
        .astype(int)
    )

    tablet_votes_df["correct"] = (
        tablet_votes_df["correct"]
        .astype(bool)
    )

    if len(tablet_votes_df) == 0:

        raise RuntimeError(
            f"No valid rows remain after cleaning:\n{tablet_period_votes_csv}"
        )

    # --------------------------------------------------------
    # ASSIGN CROP-COUNT GROUP
    # --------------------------------------------------------

    tablet_votes_df["crop_count_group"] = (
        tablet_votes_df["num_crops_used"]
        .apply(assign_crop_count_group)
    )

    tablet_votes_df["crop_count_group"] = pd.Categorical(
        tablet_votes_df["crop_count_group"],
        categories=GROUP_ORDER,
        ordered=True
    )

    # Keep only the four main groups.
    tablet_votes_df = (
        tablet_votes_df[
            tablet_votes_df["crop_count_group"].notna()
        ]
        .copy()
    )

    # --------------------------------------------------------
    # CALCULATE ACCURACY FOR EACH GROUP
    # --------------------------------------------------------

    accuracy_by_crop_count = (
        tablet_votes_df
        .groupby(
            "crop_count_group",
            observed=False
        )
        .agg(
            num_fragments=(
                "fragmentNumber",
                "count"
            ),
            correct_predictions=(
                "correct",
                "sum"
            ),
            mean_crops_used=(
                "num_crops_used",
                "mean"
            ),
            median_crops_used=(
                "num_crops_used",
                "median"
            )
        )
        .reset_index()
    )

    accuracy_by_crop_count["incorrect_predictions"] = (
        accuracy_by_crop_count["num_fragments"]
        - accuracy_by_crop_count["correct_predictions"]
    )

    accuracy_by_crop_count["accuracy"] = (
        accuracy_by_crop_count["correct_predictions"]
        / accuracy_by_crop_count["num_fragments"]
    )

    accuracy_by_crop_count["accuracy_percent"] = (
        accuracy_by_crop_count["accuracy"]
        * 100
    )

    accuracy_by_crop_count["percentage_of_evaluated_fragments"] = (
        accuracy_by_crop_count["num_fragments"]
        / len(tablet_votes_df)
        * 100
    )

    # --------------------------------------------------------
    # ADD MODEL INFORMATION
    # --------------------------------------------------------

    accuracy_by_crop_count.insert(
        0,
        "run_name",
        run_name
    )

    accuracy_by_crop_count.insert(
        1,
        "model",
        display_name
    )

    accuracy_by_crop_count.insert(
        2,
        "total_evaluated_fragments",
        len(tablet_votes_df)
    )

    # --------------------------------------------------------
    # FORMAT NUMERIC COLUMNS
    # --------------------------------------------------------

    accuracy_by_crop_count["num_fragments"] = (
        accuracy_by_crop_count["num_fragments"]
        .fillna(0)
        .astype(int)
    )

    accuracy_by_crop_count["correct_predictions"] = (
        accuracy_by_crop_count["correct_predictions"]
        .fillna(0)
        .astype(int)
    )

    accuracy_by_crop_count["incorrect_predictions"] = (
        accuracy_by_crop_count["incorrect_predictions"]
        .fillna(0)
        .astype(int)
    )

    accuracy_by_crop_count["mean_crops_used"] = (
        accuracy_by_crop_count["mean_crops_used"]
        .round(2)
    )

    accuracy_by_crop_count["median_crops_used"] = (
        accuracy_by_crop_count["median_crops_used"]
        .round(2)
    )

    accuracy_by_crop_count["accuracy"] = (
        accuracy_by_crop_count["accuracy"]
        .round(4)
    )

    accuracy_by_crop_count["accuracy_percent"] = (
        accuracy_by_crop_count["accuracy_percent"]
        .round(2)
    )

    accuracy_by_crop_count[
        "percentage_of_evaluated_fragments"
    ] = (
        accuracy_by_crop_count[
            "percentage_of_evaluated_fragments"
        ]
        .round(2)
    )

    # --------------------------------------------------------
    # SELECT OUTPUT COLUMN ORDER
    # --------------------------------------------------------

    accuracy_by_crop_count = accuracy_by_crop_count[
        [
            "run_name",
            "model",
            "total_evaluated_fragments",
            "crop_count_group",
            "num_fragments",
            "correct_predictions",
            "incorrect_predictions",
            "accuracy",
            "accuracy_percent",
            "percentage_of_evaluated_fragments",
            "mean_crops_used",
            "median_crops_used"
        ]
    ]

    # --------------------------------------------------------
    # SAVE MODEL-SPECIFIC RESULT
    # --------------------------------------------------------

    accuracy_by_crop_count.to_csv(
        accuracy_by_crop_count_csv,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("Saved model result:")
    print(accuracy_by_crop_count_csv)

    # --------------------------------------------------------
    # DISPLAY MODEL RESULT
    # --------------------------------------------------------

    print()
    print("=" * 100)
    print(f"PERIOD ACCURACY BY NUMBER OF CONFIDENT CROPS â€” {display_name}")
    print("=" * 100)

    try:
        display(accuracy_by_crop_count)

    except NameError:
        print(
            accuracy_by_crop_count.to_string(
                index=False
            )
        )

    # --------------------------------------------------------
    # PRINT SIMPLE SUMMARY
    # --------------------------------------------------------

    print()

    for _, row in accuracy_by_crop_count.iterrows():

        print(
            f"{row['model']} | "
            f"{row['crop_count_group']}: "
            f"{int(row['num_fragments']):,} fragments | "
            f"{int(row['correct_predictions']):,} correct | "
            f"{int(row['incorrect_predictions']):,} incorrect | "
            f"accuracy = {row['accuracy_percent']:.2f}%"
        )

    return accuracy_by_crop_count


# ============================================================
# RUN ALL MODELS
# ============================================================

all_accuracy_tables = []

for model_cfg in MODEL_CONFIGS:

    result_df = process_one_model(
        model_cfg
    )

    if result_df is not None:

        all_accuracy_tables.append(
            result_df
        )


# ============================================================
# COMBINE ALL MODEL RESULTS
# ============================================================

if all_accuracy_tables:

    combined_accuracy_df = pd.concat(
        all_accuracy_tables,
        ignore_index=True
    )

    combined_accuracy_df.to_csv(
        COMBINED_ACCURACY_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("=" * 100)
    print("COMBINED PERIOD ACCURACY BY NUMBER OF CONFIDENT CROPS â€” VIT + SWIN")
    print("=" * 100)

    try:
        display(combined_accuracy_df)

    except NameError:
        print(
            combined_accuracy_df.to_string(
                index=False
            )
        )

    print()
    print("Saved combined result:")
    print(COMBINED_ACCURACY_CSV)

else:

    print()
    print("No model results were created.")